In [ ]:
# =============================================
# STEP 5A: Mount Google Drive
# =============================================
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
%cd /content/drive/MyDrive/

/content/drive/MyDrive


In [ ]:
# ============================================================
# ONE-CELL END-TO-END FAST RAGNET PIPELINE FOR ADULT DATASET
# FULL CORRECTED VERSION
#
# FIXES INCLUDED
# - same categorical class of same attribute -> same input embedding
# - shared event-based memory bank storing only training row indices
# - selective balanced memory construction instead of saturated first rows
# - missing attributes: original raw value + original PLM embedding are masked
# - masked attributes are initialized from memory via theory-based retrieval
# - project_batch no longer zeroes masked attributes
# - masked categorical F1 weighted by evaluated masked count
# - masked numerical MAE/MSE weighted by evaluated masked count
# - text retrieval works with event-based memory
# ============================================================

# -----------------------------
# 0) INSTALLS
# -----------------------------
!pip -q install transformers==4.38.2 nltk scikit-learn pandas numpy tqdm

# -----------------------------
# 1) IMPORTS
# -----------------------------
import os, json, math, urllib.request, random
from collections import defaultdict, Counter

import numpy as np
import pandas as pd
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score

import nltk
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
nltk.download("punkt", quiet=True)
try:
    nltk.download("punkt_tab", quiet=True)
except:
    pass

from transformers import AutoTokenizer, AutoModel
from google.colab import drive

# -----------------------------
# 2) CONFIG
# -----------------------------
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

PROJECT_ROOT = "/content/drive/MyDrive/RAGNet_Adult_OneCell_FullCorrected"
DATA_DIR = os.path.join(PROJECT_ROOT, "data")
OUT_DIR = os.path.join(PROJECT_ROOT, "outputs")
MODEL_DIR = os.path.join(PROJECT_ROOT, "models")
EMB_DIR = os.path.join(PROJECT_ROOT, "embeddings")

for p in [PROJECT_ROOT, DATA_DIR, OUT_DIR, MODEL_DIR, EMB_DIR]:
    os.makedirs(p, exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("DEVICE:", DEVICE)

TEXTUAL_COLUMNS = ["summary_text_1", "summary_text_2"]
CATEGORICAL_COLUMNS = ["workclass", "education", "occupation", "sex"]
NUMERICAL_COLUMNS = ["age", "education_num", "hours_per_week"]
TARGET_COLUMN = "income"

EMBED_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"

BATCH_SIZE_EMB = 128
TRAIN_BATCH_SIZE = 256
MAX_LEN_SUMMARY = 16
MAX_LEN_ATTR = 12

TRAIN_SUBSET = 50000
TEST_SUBSET = 10000

# set smaller first for debugging, then raise
EPOCHS = 20
LR = 1e-4

TOP_K_EDGE = 2
MEMORY_LIMIT = 300

# training masking
USE_RANDOM_MASK_TRAINING = True
MASK_ONE_ATTR_PER_SAMPLE = True
MASK_PROB = 0.15

# IMPORTANT: loss on visible attrs only
LOSS_ON_MASKED_ONLY = False

# loss weights
USE_MARIP = True
MARIP_BLEND = 0.1
REG_WEIGHT = 1e-4

# same categorical class -> same embedding
USE_CLASS_LOOKUP_EMB = True

# same-class hidden-state consistency
USE_CAT_CLASS_CONSISTENCY_LOSS = True
CAT_CLASS_CONSISTENCY_WEIGHT = 1e-2
CONSISTENCY_ON_VISIBLE_ONLY = True

# memory-based missing initialization
USE_MISSING_INIT = True
MISSING_INIT_TOPK = 5
MISSING_INIT_R = 2

# balanced memory selection
MEMORY_NUM_BINS = 5

# -----------------------------
# 3) MOUNT DRIVE
# -----------------------------
drive.mount('/content/drive')

# -----------------------------
# 4) DOWNLOAD ADULT DATA IF NEEDED
# -----------------------------
adult_train_path = os.path.join(DATA_DIR, "adult.data")
adult_test_path = os.path.join(DATA_DIR, "adult.test")

if not os.path.exists(adult_train_path):
    urllib.request.urlretrieve(
        "https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.data",
        adult_train_path
    )
if not os.path.exists(adult_test_path):
    urllib.request.urlretrieve(
        "https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.test",
        adult_test_path
    )

adult_columns = [
    "age","workclass","fnlwgt","education","education_num","marital_status",
    "occupation","relationship","race","sex","capital_gain","capital_loss",
    "hours_per_week","native_country","income"
]

df_train = pd.read_csv(
    adult_train_path, header=None, names=adult_columns,
    na_values=["?", " ?"], skipinitialspace=True
)
df_test = pd.read_csv(
    adult_test_path, header=None, names=adult_columns,
    na_values=["?", " ?"], skipinitialspace=True, skiprows=1
)
df = pd.concat([df_train, df_test], ignore_index=True)
df["income"] = df["income"].astype(str).str.replace(".", "", regex=False).str.strip()

# -----------------------------
# 5) BASIC CLEANING
# -----------------------------
for col in CATEGORICAL_COLUMNS + [TARGET_COLUMN]:
    df[col] = df[col].fillna("Missing").astype(str).str.strip()
    df.loc[df[col] == "", col] = "Missing"

for col in NUMERICAL_COLUMNS:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# -----------------------------
# 6) SHORT SUMMARY TEXTS
# -----------------------------
def summary_text_1(row):
    age = row["age"]
    education = str(row["education"]).lower() if pd.notna(row["education"]) else "unknown"
    if pd.isna(age):
        age_phrase = "adult"
    elif float(age) < 30:
        age_phrase = "young adult"
    elif float(age) < 50:
        age_phrase = "middle-aged adult"
    else:
        age_phrase = "older adult"
    return f"{age_phrase} with {education} education."

def summary_text_2(row):
    workclass = str(row["workclass"]).lower() if pd.notna(row["workclass"]) else "unknown sector"
    occupation = str(row["occupation"]).lower() if pd.notna(row["occupation"]) else "unknown job"
    return f"{workclass} worker in {occupation} role."

df["summary_text_1"] = df.apply(summary_text_1, axis=1)
df["summary_text_2"] = df.apply(summary_text_2, axis=1)

# -----------------------------
# 7) ATTRIBUTE TEXTS
# -----------------------------
def make_attr_text(name, value):
    if pd.isna(value):
        return f"{name}: missing"
    return f"{name}: {value}"

for col in CATEGORICAL_COLUMNS:
    df[f"{col}_attr_text"] = df[col].apply(lambda x: make_attr_text(col, x))
for col in NUMERICAL_COLUMNS:
    df[f"{col}_attr_text"] = df[col].apply(lambda x: make_attr_text(col, x))

# -----------------------------
# 8) SAVE PREPARED CSV
# -----------------------------
prepared_csv = os.path.join(DATA_DIR, "adult_prepared_small.csv")
df[
    TEXTUAL_COLUMNS +
    CATEGORICAL_COLUMNS +
    NUMERICAL_COLUMNS +
    [f"{c}_attr_text" for c in CATEGORICAL_COLUMNS] +
    [f"{c}_attr_text" for c in NUMERICAL_COLUMNS] +
    [TARGET_COLUMN]
].to_csv(prepared_csv, index=False)

# -----------------------------
# 9) ENCODE CATEGORICAL + TARGET
# -----------------------------
cat_maps = {}
cat_encoded_df = pd.DataFrame()
num_classes_per_cat_attr = []

for col in CATEGORICAL_COLUMNS:
    le = LabelEncoder()
    enc = le.fit_transform(df[col].astype(str))
    cat_encoded_df[col] = enc
    cat_maps[col] = [str(x) for x in le.classes_]
    num_classes_per_cat_attr.append(len(le.classes_))

target_le = LabelEncoder()
target_labels = target_le.fit_transform(df[TARGET_COLUMN].astype(str))
target_map = {int(i): str(v) for i, v in enumerate(target_le.classes_)}

# -----------------------------
# 10) NUMERICAL VALUES
# -----------------------------
num_values_df = df[NUMERICAL_COLUMNS].copy()
num_values_raw = num_values_df.fillna(-1).to_numpy(dtype=np.float32)

scaled_num = np.zeros_like(num_values_raw, dtype=np.float32)
num_scale_stats = {}
for j in range(num_values_raw.shape[1]):
    col = num_values_raw[:, j]
    mask = col != -1
    if mask.sum() > 0:
        mean_j = col[mask].mean()
        std_j = col[mask].std()
        if std_j < 1e-8:
            std_j = 1.0
        scaled_num[mask, j] = (col[mask] - mean_j) / std_j
        scaled_num[~mask, j] = -1.0
        num_scale_stats[NUMERICAL_COLUMNS[j]] = {"mean": float(mean_j), "std": float(std_j)}
    else:
        scaled_num[:, j] = -1.0
        num_scale_stats[NUMERICAL_COLUMNS[j]] = {"mean": 0.0, "std": 1.0}

# -----------------------------
# 11) LOAD EMBEDDING MODEL
# -----------------------------
emb_tokenizer = AutoTokenizer.from_pretrained(EMBED_MODEL_NAME)
emb_model = AutoModel.from_pretrained(EMBED_MODEL_NAME).to(DEVICE)
if DEVICE.type == "cuda":
    emb_model = emb_model.half()
emb_model.eval()

def mean_pooling(model_output, attention_mask):
    token_embeddings = model_output.last_hidden_state
    mask = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
    return torch.sum(token_embeddings * mask, dim=1) / torch.clamp(mask.sum(dim=1), min=1e-9)

def embed_text_list(text_list, batch_size=128, max_length=16):
    all_embs = []
    for i in tqdm(range(0, len(text_list), batch_size), leave=False):
        batch = text_list[i:i+batch_size]
        encoded = emb_tokenizer(
            batch, padding=True, truncation=True, max_length=max_length, return_tensors="pt"
        ).to(DEVICE)
        with torch.no_grad():
            with torch.cuda.amp.autocast(enabled=(DEVICE.type == "cuda")):
                out = emb_model(**encoded)
                emb = mean_pooling(out, encoded["attention_mask"]).float()
        all_embs.append(emb.cpu().numpy())
        del encoded, out, emb
        if DEVICE.type == "cuda":
            torch.cuda.empty_cache()
    return np.vstack(all_embs)

# -----------------------------
# 12) COMPUTE EMBEDDINGS
# -----------------------------
textual_embeddings = np.stack([
    embed_text_list(df[col].fillna("").astype(str).tolist(), BATCH_SIZE_EMB, MAX_LEN_SUMMARY)
    for col in TEXTUAL_COLUMNS
], axis=1).astype(np.float32)

def build_same_class_categorical_embeddings(df, categorical_columns, batch_size=128, max_length=12):
    """
    same attribute + same class -> exactly same embedding
    """
    per_attr_embeddings = []
    categorical_class_embedding_maps = {}

    for col in categorical_columns:
        values = df[col].fillna("Missing").astype(str).str.strip().tolist()
        unique_vals = sorted(list(set(values)))
        unique_texts = [make_attr_text(col, v) for v in unique_vals]
        unique_embs = embed_text_list(unique_texts, batch_size=batch_size, max_length=max_length).astype(np.float32)
        val_to_emb = {v: unique_embs[i] for i, v in enumerate(unique_vals)}
        categorical_class_embedding_maps[col] = val_to_emb
        row_embs = np.stack([val_to_emb[v] for v in values], axis=0).astype(np.float32)
        per_attr_embeddings.append(row_embs)

    return np.stack(per_attr_embeddings, axis=1).astype(np.float32), categorical_class_embedding_maps

if USE_CLASS_LOOKUP_EMB:
    categorical_embeddings, categorical_class_embedding_maps = build_same_class_categorical_embeddings(
        df=df,
        categorical_columns=CATEGORICAL_COLUMNS,
        batch_size=BATCH_SIZE_EMB,
        max_length=MAX_LEN_ATTR
    )
else:
    categorical_embeddings = np.stack([
        embed_text_list(df[f"{col}_attr_text"].fillna("").astype(str).tolist(), BATCH_SIZE_EMB, MAX_LEN_ATTR)
        for col in CATEGORICAL_COLUMNS
    ], axis=1).astype(np.float32)
    categorical_class_embedding_maps = None

numerical_embeddings = np.stack([
    embed_text_list(df[f"{col}_attr_text"].fillna("").astype(str).tolist(), BATCH_SIZE_EMB, MAX_LEN_ATTR)
    for col in NUMERICAL_COLUMNS
], axis=1).astype(np.float32)

all_embeddings_np = np.concatenate([textual_embeddings, categorical_embeddings, numerical_embeddings], axis=1)
all_embeddings = torch.tensor(all_embeddings_np, dtype=torch.float32)

normalized_embeddings = all_embeddings.clone()
N, M, D = normalized_embeddings.shape
for m in range(M):
    attr = all_embeddings[:, m, :]
    nz = (attr.abs().sum(dim=1) != 0)
    if nz.sum() > 0:
        mean_m = attr[nz].mean(dim=0)
        std_m = attr[nz].std(dim=0)
        normalized_embeddings[nz, m, :] = (attr[nz] - mean_m) / (std_m + 1e-8)

# -----------------------------
# 13) BUILD GROUND TRUTH
# -----------------------------
ground_truth_text = df[TEXTUAL_COLUMNS].fillna("").astype(str).to_numpy()
ground_truth_cat = cat_encoded_df.to_numpy(dtype=np.int64)
ground_truth_num = scaled_num.astype(np.float32)

MODALITIES = ["txt"] * len(TEXTUAL_COLUMNS) + ["cat"] * len(CATEGORICAL_COLUMNS) + ["num"] * len(NUMERICAL_COLUMNS)
ALL_ATTRIBUTE_NAMES = TEXTUAL_COLUMNS + CATEGORICAL_COLUMNS + NUMERICAL_COLUMNS

categorical_indices = cat_encoded_df.to_numpy(dtype=np.int64)
numerical_values = num_values_raw.astype(np.float32)

categorical_indices_tensor = torch.tensor(categorical_indices, dtype=torch.long)
numerical_values_tensor = torch.tensor(numerical_values, dtype=torch.float32)

# -----------------------------
# 14) TRAIN/TEST SPLIT
# -----------------------------
indices = np.arange(len(df))
train_idx, test_idx = train_test_split(
    indices, test_size=0.2, random_state=SEED, stratify=target_labels
)
train_idx = train_idx[:TRAIN_SUBSET]
test_idx = test_idx[:TEST_SUBSET]

test_df = df.iloc[test_idx].reset_index(drop=True)

# -----------------------------
# 15) HELPERS
# -----------------------------
def safe_word_tokenize(x):
    try:
        return nltk.word_tokenize(str(x))
    except:
        return str(x).split()

def compute_text_bleu(text_refs, text_preds):
    smoothie = SmoothingFunction().method1
    scores = []
    for ref, pred in zip(text_refs, text_preds):
        ref_tokens = safe_word_tokenize(ref)
        pred_tokens = safe_word_tokenize(pred)
        if len(ref_tokens) == 0 or len(pred_tokens) == 0:
            scores.append(0.0)
            continue
        scores.append(sentence_bleu([ref_tokens], pred_tokens, smoothing_function=smoothie))
    return float(np.mean(scores)) if len(scores) else 0.0

def compute_weighted_cat_f1_per_attribute(cat_true, cat_pred, categorical_columns):
    results = {}
    scores = []
    for j, col in enumerate(categorical_columns):
        f1 = f1_score(cat_true[:, j], cat_pred[:, j], average="weighted", zero_division=0)
        results[col] = float(f1)
        scores.append(f1)
    results["weighted_attribute_f1"] = float(np.mean(scores)) if len(scores) else 0.0
    return results

def compute_weighted_num_mae_per_attribute(num_true, num_pred, numerical_columns):
    results = {}
    maes = []
    for j, col in enumerate(numerical_columns):
        valid = num_true[:, j] != -1
        if valid.any():
            mae = np.mean(np.abs(num_true[valid, j] - num_pred[valid, j]))
            results[col] = float(mae)
            maes.append(mae)
        else:
            results[col] = None
    results["weighted_attribute_mae"] = float(np.mean(maes)) if len(maes) else 0.0
    return results

def build_training_attr_mask(batch_size, num_attrs, device, mask_one_attr_per_sample=True, mask_prob=0.15):
    attr_mask = torch.ones(batch_size, num_attrs, device=device)
    if mask_one_attr_per_sample:
        masked_attr = torch.randint(0, num_attrs, (batch_size,), device=device)
        attr_mask[torch.arange(batch_size, device=device), masked_attr] = 0.0
    else:
        rand_mask = (torch.rand(batch_size, num_attrs, device=device) < mask_prob).float()
        empty_rows = rand_mask.sum(dim=1) == 0
        if empty_rows.any():
            row_ids = torch.where(empty_rows)[0]
            forced = torch.randint(0, num_attrs, (row_ids.numel(),), device=device)
            rand_mask[row_ids] = 0.0
            rand_mask[row_ids, forced] = 1.0
        attr_mask = 1.0 - rand_mask
    return attr_mask

def apply_input_mask(llm, rawn, cati, attr_mask):
    """
    remove original value + original embedding for masked attrs
    """
    llm = llm.clone()
    rawn = rawn.clone()
    cati = cati.clone()

    for attr_id, mod in enumerate(MODALITIES):
        masked_rows = attr_mask[:, attr_id] == 0
        if not masked_rows.any():
            continue

        llm[masked_rows, attr_id, :] = 0.0

        if mod == "cat":
            local_cat = attr_id - len(TEXTUAL_COLUMNS)
            cati[masked_rows, local_cat] = 0

        elif mod == "num":
            local_num = attr_id - len(TEXTUAL_COLUMNS) - len(CATEGORICAL_COLUMNS)
            rawn[masked_rows, local_num] = 0.0

    return llm, rawn, cati

def hidden_regularization(H, attr_mask=None):
    reg = (H ** 2).mean()
    if attr_mask is not None:
        masked_h = H * (1.0 - attr_mask.unsqueeze(-1))
        reg = reg + 0.1 * (masked_h ** 2).mean()
    return reg

def categorical_class_consistency_loss(H, cat_targets, attr_mask=None, visible_only=True):
    total_loss = 0.0
    total_terms = 0

    for j in range(len(CATEGORICAL_COLUMNS)):
        attr_id = len(TEXTUAL_COLUMNS) + j
        h = H[:, attr_id, :]
        y = cat_targets[:, j]

        if attr_mask is not None and visible_only:
            valid_rows = (attr_mask[:, attr_id] == 1)
        else:
            valid_rows = torch.ones(h.size(0), dtype=torch.bool, device=h.device)

        if valid_rows.sum() < 2:
            continue

        y_valid = y[valid_rows]
        h_valid = h[valid_rows]

        unique_classes = torch.unique(y_valid)
        for cls in unique_classes:
            cls_rows = (y_valid == cls)
            if cls_rows.sum() < 2:
                continue

            h_cls = h_valid[cls_rows]
            proto = h_cls.mean(dim=0, keepdim=True)
            total_loss = total_loss + ((h_cls - proto) ** 2).mean()
            total_terms += 1

    if total_terms == 0:
        return torch.tensor(0.0, device=H.device)

    return total_loss / total_terms

def weighted_average_by_count(items, value_key, count_key):
    valid_items = [
        x for x in items
        if x.get(value_key) is not None and x.get(count_key) is not None
    ]
    total_weight = sum(x[count_key] for x in valid_items)
    if total_weight <= 0:
        return None, 0
    score = sum(x[value_key] * x[count_key] for x in valid_items) / total_weight
    return float(score), int(total_weight)

def build_balanced_memory_row_ids(
    train_idx,
    df,
    categorical_columns,
    numerical_columns,
    target_column,
    memory_limit=300,
    bins_per_num=5,
    seed=42,
):
    """
    selective memory rows with broad coverage
    """
    rng = np.random.RandomState(seed)
    train_idx = np.array(train_idx, dtype=int)
    train_df = df.iloc[train_idx].copy()

    selected = []
    selected_set = set()

    def add_row(rid):
        rid = int(rid)
        if rid not in selected_set and len(selected) < memory_limit:
            selected.append(rid)
            selected_set.add(rid)

    # 1) cover target classes
    if target_column in train_df.columns:
        for _, sub in train_df.groupby(target_column):
            candidate_rows = sub.index.to_numpy()
            rng.shuffle(candidate_rows)
            if len(candidate_rows) > 0:
                add_row(candidate_rows[0])

    # 2) cover every categorical class
    for col in categorical_columns:
        for _, sub in train_df.groupby(col):
            candidate_rows = sub.index.to_numpy()
            rng.shuffle(candidate_rows)
            if len(candidate_rows) > 0:
                add_row(candidate_rows[0])

    # 3) cover numerical bins
    for col in numerical_columns:
        valid_mask = train_df[col].notna()
        valid_df = train_df.loc[valid_mask, [col]]
        if len(valid_df) == 0:
            continue

        try:
            n_bins = min(bins_per_num, max(1, valid_df[col].nunique()))
            bin_ids = pd.qcut(valid_df[col], q=n_bins, duplicates="drop")
        except:
            continue

        tmp = valid_df.copy()
        tmp["_bin"] = bin_ids

        for _, sub in tmp.groupby("_bin"):
            candidate_rows = sub.index.to_numpy()
            rng.shuffle(candidate_rows)
            if len(candidate_rows) > 0:
                add_row(candidate_rows[0])

    # 4) strengthen rare categorical classes
    if len(selected) < memory_limit:
        for col in categorical_columns:
            value_counts = train_df[col].value_counts()
            rare_first = value_counts.sort_values().index.tolist()

            for val in rare_first:
                sub = train_df[train_df[col] == val]
                candidate_rows = sub.index.to_numpy()
                rng.shuffle(candidate_rows)
                for rid in candidate_rows[:2]:
                    add_row(rid)
                    if len(selected) >= memory_limit:
                        break
                if len(selected) >= memory_limit:
                    break
            if len(selected) >= memory_limit:
                break

    # 5) fill remaining randomly
    if len(selected) < memory_limit:
        remaining = [int(r) for r in train_idx if int(r) not in selected_set]
        rng.shuffle(remaining)
        for rid in remaining:
            add_row(rid)
            if len(selected) >= memory_limit:
                break

    return selected

# -----------------------------
# 16) EVENT-BASED MEMORY BANK
# -----------------------------
class EventMemoryBank:
    """
    Memory indexed by shared historical event ids.
    Stores only row ids; embeddings/values are retrieved lazily.
    """
    def __init__(self, normalized_embeddings, df, categorical_indices, numerical_values,
                 textual_columns, categorical_columns, numerical_columns, modalities):
        self.row_ids = []
        self.normalized_embeddings = normalized_embeddings
        self.df = df
        self.categorical_indices = categorical_indices
        self.numerical_values = numerical_values
        self.textual_columns = textual_columns
        self.categorical_columns = categorical_columns
        self.numerical_columns = numerical_columns
        self.modalities = modalities
        self.emb_cache = {}
        self.values_cache = {}
        self.valid_mask_cache = {}

    def set_row_ids(self, row_ids):
        self.row_ids = [int(x) for x in row_ids]
        self.emb_cache = {}
        self.values_cache = {}
        self.valid_mask_cache = {}

    def size(self):
        return len(self.row_ids)

    def get_row_ids(self):
        return self.row_ids

    def get_embeddings_for_attr(self, attr_id):
        if attr_id in self.emb_cache:
            return self.emb_cache[attr_id]
        if len(self.row_ids) == 0:
            return None
        emb = self.normalized_embeddings[self.row_ids, attr_id, :].detach().cpu()
        self.emb_cache[attr_id] = emb
        return emb

    def get_values_for_attr(self, attr_id):
        if attr_id in self.values_cache:
            return self.values_cache[attr_id]

        vals = []
        if self.modalities[attr_id] == "txt":
            local_txt = attr_id
            col = self.textual_columns[local_txt]
            for rid in self.row_ids:
                vals.append(str(self.df.iloc[rid][col]).strip())

        elif self.modalities[attr_id] == "cat":
            local_cat = attr_id - len(self.textual_columns)
            for rid in self.row_ids:
                vals.append(int(self.categorical_indices[rid, local_cat]))

        else:
            local_num = attr_id - len(self.textual_columns) - len(self.categorical_columns)
            for rid in self.row_ids:
                vals.append(float(self.numerical_values[rid, local_num]))

        self.values_cache[attr_id] = vals
        return vals

    def get_valid_mask_for_attr(self, attr_id):
        if attr_id in self.valid_mask_cache:
            return self.valid_mask_cache[attr_id]

        if len(self.row_ids) == 0:
            out = torch.zeros(0, dtype=torch.bool)
            self.valid_mask_cache[attr_id] = out
            return out

        mod = self.modalities[attr_id]

        if mod == "txt":
            local_txt = attr_id
            col = self.textual_columns[local_txt]
            mask = []
            for rid in self.row_ids:
                v = str(self.df.iloc[rid][col]).strip()
                mask.append(v != "")
            out = torch.tensor(mask, dtype=torch.bool)

        elif mod == "cat":
            out = torch.ones(len(self.row_ids), dtype=torch.bool)

        else:
            local_num = attr_id - len(self.textual_columns) - len(self.categorical_columns)
            mask = []
            for rid in self.row_ids:
                v = float(self.numerical_values[rid, local_num])
                mask.append(v != -1)
            out = torch.tensor(mask, dtype=torch.bool)

        self.valid_mask_cache[attr_id] = out
        return out

memory_bank = EventMemoryBank(
    normalized_embeddings=normalized_embeddings,
    df=df,
    categorical_indices=categorical_indices,
    numerical_values=numerical_values,
    textual_columns=TEXTUAL_COLUMNS,
    categorical_columns=CATEGORICAL_COLUMNS,
    numerical_columns=NUMERICAL_COLUMNS,
    modalities=MODALITIES
)

memory_row_ids = build_balanced_memory_row_ids(
    train_idx=train_idx,
    df=df,
    categorical_columns=CATEGORICAL_COLUMNS,
    numerical_columns=NUMERICAL_COLUMNS,
    target_column=TARGET_COLUMN,
    memory_limit=MEMORY_LIMIT,
    bins_per_num=MEMORY_NUM_BINS,
    seed=SEED,
)
memory_bank.set_row_ids(memory_row_ids)
print("Event-based memory bank built. size =", memory_bank.size())
print("First 10 memory row ids:", memory_row_ids[:10])

# -----------------------------
# 17) EDGE INDEX
# -----------------------------
def build_modality_based_edge_index(norm_embs, modality_types, top_k=2):
    N0, M0, D0 = norm_embs.shape
    attr_means = []
    for m in range(M0):
        e = norm_embs[:, m, :].detach().cpu()
        nz = (e.abs().sum(dim=1) != 0)
        if nz.any():
            mean_e = e[nz].mean(dim=0)
        else:
            mean_e = torch.zeros(D0)
        attr_means.append(mean_e)

    attr_means = torch.stack(attr_means, dim=0)
    attr_means = F.normalize(attr_means, dim=1)

    groups = defaultdict(list)
    for i, mod in enumerate(modality_types):
        groups[mod].append(i)

    edges = []
    for _, idxs in groups.items():
        if len(idxs) < 2:
            continue
        em = attr_means[idxs]
        sim = torch.matmul(em, em.T)
        for i, src in enumerate(idxs):
            top = torch.topk(sim[i], k=min(top_k + 1, len(idxs))).indices.tolist()
            for t in top:
                tgt = idxs[t]
                if src != tgt:
                    edges.append((src, tgt))
    edges += [(j, i) for (i, j) in edges]
    edges = list(set(edges))
    if len(edges) == 0:
        return torch.empty((2, 0), dtype=torch.long)
    return torch.tensor(edges, dtype=torch.long).T

edge_index = build_modality_based_edge_index(normalized_embeddings, MODALITIES, top_k=TOP_K_EDGE).to(DEVICE)
print("edge_index:", edge_index.shape)

# -----------------------------
# 18) THEORY-BASED MISSING INIT
# -----------------------------
def initialize_missing_embeddings_theory(
    missing_attr_idx,
    known_attr,
    known_embedding,
    memory_bank,
    modality_types,
    k=5,
    r=2,
    d_h=384,
):
    """
    Implements theory:
    keep memory event positions that appear in Top-K lists of at least r observed attrs
    then intersect with valid target-attribute memory
    initialize as mean of retrieved missing-attribute embeddings
    """
    device = known_embedding.device
    missing_modality = modality_types[missing_attr_idx]

    if known_embedding.dim() != 2:
        raise ValueError("known_embedding must be [num_attributes, d_h] for one sample")

    M_mem = memory_bank.size()
    if M_mem == 0:
        return torch.zeros(d_h, device=device), None, [], []

    # Step 1: Top-K retrieval for each observed attr
    topk_lists = []
    for i_obs in known_attr:
        mem_obs = memory_bank.get_embeddings_for_attr(i_obs)
        if mem_obs is None or mem_obs.size(0) == 0:
            continue

        mem_obs = mem_obs.to(device)
        non_zero_mask = mem_obs.abs().sum(dim=1) > 1e-6
        if not non_zero_mask.any():
            continue

        query = F.normalize(known_embedding[i_obs], dim=0)
        mem_norm = F.normalize(mem_obs[non_zero_mask], dim=1)
        sim = torch.matmul(mem_norm, query)

        topk_local = torch.topk(sim, k=min(k, sim.numel())).indices.tolist()
        valid_positions = torch.where(non_zero_mask)[0].tolist()
        topk_positions = [valid_positions[j] for j in topk_local]
        topk_lists.append(topk_positions)

    # Step 2: candidate positions appearing in at least r observed attrs
    candidate_positions = []
    if len(topk_lists) > 0:
        counter = Counter()
        for lst in topk_lists:
            counter.update(lst)

        threshold = min(r, len(topk_lists))
        candidate_positions = [p for p, cnt in counter.items() if cnt >= threshold]

        if len(candidate_positions) == 0 and len(counter) > 0:
            max_count = max(counter.values())
            candidate_positions = [p for p, cnt in counter.items() if cnt == max_count]

    # Step 3: intersect with valid target-attribute memory
    mem_missing = memory_bank.get_embeddings_for_attr(missing_attr_idx)
    if mem_missing is None or mem_missing.size(0) == 0:
        return torch.zeros(d_h, device=device), None, [], []

    mem_missing = mem_missing.to(device)
    valid_mask_missing = memory_bank.get_valid_mask_for_attr(missing_attr_idx).to(device)
    non_zero_missing = mem_missing.abs().sum(dim=1) > 1e-6
    valid_mask_missing = valid_mask_missing & non_zero_missing

    candidate_positions = [p for p in candidate_positions if p < mem_missing.size(0)]
    reliable_positions = [p for p in candidate_positions if valid_mask_missing[p].item()]

    # fallback: all valid target memory
    if len(reliable_positions) == 0:
        reliable_positions = torch.where(valid_mask_missing)[0].cpu().tolist()

    if len(reliable_positions) == 0:
        return torch.zeros(d_h, device=device), None, [], []

    # Step 4: embedding init
    selected_embeddings = mem_missing[reliable_positions]
    init_embedding = selected_embeddings.mean(dim=0)

    # Step 5: raw value init for num/cat
    init_value = None
    values_missing = memory_bank.get_values_for_attr(missing_attr_idx)
    selected_values = [values_missing[p] for p in reliable_positions]

    if missing_modality == "num":
        valid_values = []
        for v in selected_values:
            try:
                fv = float(v)
                if fv != -1:
                    valid_values.append(fv)
            except:
                pass
        if len(valid_values) > 0:
            init_value = sum(valid_values) / len(valid_values)

    elif missing_modality == "cat":
        valid_values = [int(v) for v in selected_values if v is not None]
        if len(valid_values) > 0:
            init_value = Counter(valid_values).most_common(1)[0][0]

    elif missing_modality == "txt":
        init_value = None

    retrieved_event_row_ids = [memory_bank.get_row_ids()[p] for p in reliable_positions]
    return init_embedding, init_value, reliable_positions, retrieved_event_row_ids

def apply_memory_initialization(
    llm,
    rawn,
    cati,
    attr_mask,
    memory_bank,
    modality_types,
    k=5,
    r=2,
):
    """
    after masking original info, fill masked attrs from memory
    """
    llm = llm.clone()
    rawn = rawn.clone()
    cati = cati.clone()

    B, M0, D0 = llm.shape
    device = llm.device

    for b in range(B):
        known_attr = torch.where(attr_mask[b] == 1)[0].tolist()
        missing_attr = torch.where(attr_mask[b] == 0)[0].tolist()

        for miss_id in missing_attr:
            init_emb, init_val, _, _ = initialize_missing_embeddings_theory(
                missing_attr_idx=miss_id,
                known_attr=known_attr,
                known_embedding=llm[b],
                memory_bank=memory_bank,
                modality_types=modality_types,
                k=k,
                r=r,
                d_h=D0,
            )

            llm[b, miss_id, :] = init_emb.to(device)

            if modality_types[miss_id] == "cat" and init_val is not None:
                local_cat = miss_id - len(TEXTUAL_COLUMNS)
                cati[b, local_cat] = int(init_val)

            elif modality_types[miss_id] == "num" and init_val is not None:
                local_num = miss_id - len(TEXTUAL_COLUMNS) - len(CATEGORICAL_COLUMNS)
                rawn[b, local_num] = float(init_val)

    return llm, rawn, cati

# -----------------------------
# 19) MARIP LOSS
# -----------------------------
class MARIPLoss(nn.Module):
    def __init__(self, alpha=0.1, epsilon=1e-6):
        super().__init__()
        self.alpha = alpha
        self.epsilon = epsilon
        self.class_counts = {}
        self.observed_freq = {}
        self.num_scale = 0.5
        self.cat_scale = 0.5

    def register_class_counts(self, attr_idx, class_labels):
        class_labels = np.asarray(class_labels)
        unique, counts = np.unique(class_labels, return_counts=True)
        self.class_counts[attr_idx] = dict(zip(unique.tolist(), counts.tolist()))

    def register_observed_frequency(self, attr_idx, observed_count, total):
        self.observed_freq[attr_idx] = observed_count / (total + self.epsilon)

    def compute_lambda(self, idx, modality, y_true_value=None):
        obs = self.observed_freq.get(idx, 0.1)
        lam_miss = (1.0 / (obs + self.epsilon)) ** 0.5
        lam_miss = float(np.clip(lam_miss, 0.75, 2.5))

        lam_cls = 1.0
        lam_num = 1.0

        if modality == "cat":
            if idx in self.class_counts and y_true_value is not None:
                counts_dict = self.class_counts[idx]
                count = counts_dict.get(int(y_true_value), 1)
                total_count = sum(counts_dict.values())
                num_classes = max(len(counts_dict), 1)
                avg_count = total_count / num_classes
                lam_cls = 1.0 + (avg_count / (count + self.epsilon)) ** 0.5
                lam_cls = float(np.clip(lam_cls, 1.0, 3.0))
            lam = lam_miss * lam_cls
        elif modality == "num":
            lam = lam_miss * lam_num
        else:
            lam = 1.0

        lam = float(np.clip(lam, 1.0, 3.0))
        return lam

marip_loss_fn = MARIPLoss(alpha=0.1)

for j, col in enumerate(CATEGORICAL_COLUMNS):
    global_attr_idx = len(TEXTUAL_COLUMNS) + j
    marip_loss_fn.register_class_counts(global_attr_idx, ground_truth_cat[:, j])

total_n = len(df)
for j, col in enumerate(CATEGORICAL_COLUMNS):
    global_attr_idx = len(TEXTUAL_COLUMNS) + j
    observed_count = int(df[col].notna().sum())
    marip_loss_fn.register_observed_frequency(global_attr_idx, observed_count, total_n)

for j, col in enumerate(NUMERICAL_COLUMNS):
    global_attr_idx = len(TEXTUAL_COLUMNS) + len(CATEGORICAL_COLUMNS) + j
    observed_count = int(df[col].notna().sum())
    marip_loss_fn.register_observed_frequency(global_attr_idx, observed_count, total_n)

def compute_masked_marip_loss(cat_outs, num_outs, cat_targets, num_targets, attr_mask, marip_loss_fn, loss_on_masked_only=False):
    total_loss = 0.0
    valid_count = 0

    # categorical
    for j, logits in enumerate(cat_outs):
        global_attr_idx = len(TEXTUAL_COLUMNS) + j
        y_true = cat_targets[:, j].long().to(logits.device)

        masked_rows = (attr_mask[:, global_attr_idx] == 0)
        visible_rows = (attr_mask[:, global_attr_idx] == 1)
        use_rows = masked_rows if loss_on_masked_only else visible_rows

        if not use_rows.any():
            continue

        logits_use = logits[use_rows]
        y_true_use = y_true[use_rows]
        rec_loss = F.cross_entropy(logits_use, y_true_use, reduction="none")

        lam_list = []
        for b in range(y_true_use.size(0)):
            lam = marip_loss_fn.compute_lambda(global_attr_idx, "cat", y_true_use[b].item())
            lam_list.append(lam)

        lam_tensor = torch.tensor(lam_list, dtype=rec_loss.dtype, device=rec_loss.device)
        total_loss = total_loss + marip_loss_fn.cat_scale * (lam_tensor * rec_loss).mean()
        valid_count += 1

    # numerical
    for j, pred in enumerate(num_outs):
        global_attr_idx = len(TEXTUAL_COLUMNS) + len(CATEGORICAL_COLUMNS) + j
        y_true = num_targets[:, j].to(pred.device)
        valid_num = y_true != -1

        masked_rows = (attr_mask[:, global_attr_idx] == 0)
        visible_rows = (attr_mask[:, global_attr_idx] == 1)
        use_rows = masked_rows if loss_on_masked_only else visible_rows
        use_valid = use_rows & valid_num

        if not use_valid.any():
            continue

        pred_use = pred[use_valid]
        y_true_use = y_true[use_valid]
        rec_loss = F.mse_loss(pred_use, y_true_use, reduction="none")

        lam_list = []
        for b in range(y_true_use.size(0)):
            lam = marip_loss_fn.compute_lambda(global_attr_idx, "num", float(y_true_use[b].item()))
            lam_list.append(lam)

        lam_tensor = torch.tensor(lam_list, dtype=rec_loss.dtype, device=rec_loss.device)
        total_loss = total_loss + marip_loss_fn.num_scale * (lam_tensor * rec_loss).mean()
        valid_count += 1

    if valid_count == 0:
        return torch.tensor(0.0, device=attr_mask.device, requires_grad=True)

    return total_loss

# -----------------------------
# 20) MODEL
# -----------------------------
class VectorizedCrossModalGNN(nn.Module):
    def __init__(self, d):
        super().__init__()
        self.Wg = nn.Linear(d, d, bias=False)
        self.Wq = nn.Linear(d, d, bias=False)
        self.Wk = nn.Linear(d, d, bias=False)
        self.Wc = nn.Linear(d, d, bias=False)
        self.a = nn.Parameter(torch.randn(2 * d))

    def forward(self, Z, edge_index):
        B, M0, D0 = Z.shape
        if edge_index.numel() == 0:
            Q = self.Wq(Z)
            K = self.Wk(Z)
            beta = F.softmax(torch.matmul(Q, K.transpose(1, 2)) / math.sqrt(D0), dim=-1)
            global_msg = torch.matmul(beta, self.Wc(Z))
            return F.leaky_relu(global_msg + Z)

        WgZ = self.Wg(Z)

        src = edge_index[0]
        tgt = edge_index[1]

        src_feat = WgZ[:, src, :]
        tgt_feat = WgZ[:, tgt, :]
        pair = torch.cat([src_feat, tgt_feat], dim=-1)
        e = F.leaky_relu(pair @ self.a)

        alpha = torch.zeros(B, M0, M0, device=Z.device)
        alpha[:, src, tgt] = e

        for i in range(M0):
            nbrs = tgt[src == i]
            if len(nbrs) > 0:
                alpha[:, i, nbrs] = F.softmax(alpha[:, i, nbrs], dim=-1)

        local_msg = torch.matmul(alpha, WgZ)

        Q = self.Wq(Z)
        K = self.Wk(Z)
        beta = F.softmax(torch.matmul(Q, K.transpose(1, 2)) / math.sqrt(D0), dim=-1)
        global_msg = torch.matmul(beta, self.Wc(Z))

        return F.leaky_relu(local_msg + global_msg + Z)

class FastAdultRAGNet(nn.Module):
    def __init__(self, embed_dim=384, d_shared=384, d_num=32, d_cat=16,
                 num_numerical_attrs=3, num_categorical_attrs=4,
                 num_classes_per_cat_attr=None, memory_bank=None):
        super().__init__()
        if num_classes_per_cat_attr is None:
            num_classes_per_cat_attr = [10] * num_categorical_attrs

        self.text_proj = nn.Linear(embed_dim, d_shared)

        self.num_proj = nn.ModuleList([
            nn.Sequential(nn.Linear(1, d_num), nn.ReLU(), nn.LayerNorm(d_num))
            for _ in range(num_numerical_attrs)
        ])
        self.cat_emb = nn.ModuleList([
            nn.Embedding(n_cls, d_cat) for n_cls in num_classes_per_cat_attr
        ])

        self.num_shared = nn.ModuleList([
            nn.Linear(embed_dim + d_num, d_shared) for _ in range(num_numerical_attrs)
        ])
        self.cat_shared = nn.ModuleList([
            nn.Linear(embed_dim + d_cat, d_shared) for _ in range(num_categorical_attrs)
        ])

        self.gnn = VectorizedCrossModalGNN(d_shared)
        self.memory_bank = memory_bank

        self.gate_h = nn.Linear(d_shared, d_shared)
        self.gate_m = nn.Linear(d_shared, d_shared)

        self.num_heads = nn.ModuleList([nn.Linear(d_shared, 1) for _ in range(num_numerical_attrs)])
        self.cat_heads = nn.ModuleList([nn.Linear(d_shared, n_cls) for n_cls in num_classes_per_cat_attr])

    def project_batch(self, llm_embeddings, raw_numericals, cat_indices):
        """
        pure projection only
        do NOT zero masked attrs here
        """
        Z = []
        num_idx = 0
        cat_idx = 0

        for i, modality in enumerate(MODALITIES):
            x = llm_embeddings[:, i, :]

            if modality == "txt":
                z = self.text_proj(x)

            elif modality == "cat":
                c = self.cat_emb[cat_idx](cat_indices[:, cat_idx])
                z = self.cat_shared[cat_idx](torch.cat([x, c], dim=-1))
                cat_idx += 1

            else:
                n = raw_numericals[:, num_idx].unsqueeze(-1)
                nvec = self.num_proj[num_idx](n)
                z = self.num_shared[num_idx](torch.cat([x, nvec], dim=-1))
                num_idx += 1

            Z.append(z)

        return torch.stack(Z, dim=1)

    def memory_fusion(self, H, attr_mask=None):
        if self.memory_bank is None:
            return H

        out = H.clone()
        B, M0, D0 = H.shape

        for i in range(M0):
            mem = self.memory_bank.get_embeddings_for_attr(i)
            if mem is None:
                continue

            mem = mem.to(H.device)
            valid_mask = self.memory_bank.get_valid_mask_for_attr(i).to(H.device)
            non_zero_mask = mem.abs().sum(dim=1) > 1e-8
            use_mask = valid_mask & non_zero_mask

            if use_mask.sum() == 0:
                continue

            mem = mem[use_mask]
            mem = mem / (mem.norm(dim=1, keepdim=True) + 1e-8)

            h = H[:, i, :]
            h_norm = h / (h.norm(dim=1, keepdim=True) + 1e-8)

            sim = torch.matmul(h_norm, mem.T)
            attn = F.softmax(sim, dim=-1)
            r = torch.matmul(attn, mem)

            g = torch.sigmoid(self.gate_h(h) + self.gate_m(r))
            out[:, i, :] = g * r + (1 - g) * h

        return out

    def forward(self, llm_embeddings, raw_numericals, cat_indices, edge_index, attr_mask=None):
        Z = self.project_batch(llm_embeddings, raw_numericals, cat_indices)
        H = self.gnn(Z, edge_index)
        H = self.memory_fusion(H, attr_mask=attr_mask)

        cat_outs = []
        num_outs = []

        cat_idx = 0
        num_idx = 0
        for i, modality in enumerate(MODALITIES):
            if modality == "cat":
                cat_outs.append(self.cat_heads[cat_idx](H[:, i, :]))
                cat_idx += 1
            elif modality == "num":
                num_outs.append(self.num_heads[num_idx](H[:, i, :]).squeeze(-1))
                num_idx += 1

        return cat_outs, num_outs, H

model = FastAdultRAGNet(
    embed_dim=384,
    d_shared=384,
    d_num=32,
    d_cat=16,
    num_numerical_attrs=len(NUMERICAL_COLUMNS),
    num_categorical_attrs=len(CATEGORICAL_COLUMNS),
    num_classes_per_cat_attr=num_classes_per_cat_attr,
    memory_bank=memory_bank
).to(DEVICE)

# -----------------------------
# 21) TRAIN TENSORS
# -----------------------------
X_all = normalized_embeddings
X_train = X_all[train_idx]
X_test = X_all[test_idx]

cat_targets_t = torch.tensor(ground_truth_cat, dtype=torch.long)
num_targets_t = torch.tensor(ground_truth_num, dtype=torch.float32)

cat_train = cat_targets_t[train_idx]
cat_test = cat_targets_t[test_idx]

num_train = num_targets_t[train_idx]
num_test = num_targets_t[test_idx]

raw_num_train = numerical_values_tensor[train_idx]
raw_num_test = numerical_values_tensor[test_idx]

cat_val_train = categorical_indices_tensor[train_idx]
cat_val_test = categorical_indices_tensor[test_idx]

# -----------------------------
# 22) TRAIN LOSS
# -----------------------------
def compute_masked_training_loss(cat_outs, num_outs, H, caty, numy, attr_mask,
                                 marip_loss_fn=None,
                                 loss_on_masked_only=False,
                                 reg_weight=1e-4,
                                 marip_blend=0.5,
                                 use_cat_class_consistency_loss=True,
                                 cat_class_consistency_weight=1e-2):
    recon_loss = 0.0
    valid_terms = 0

    cat_ptr = 0
    num_ptr = 0

    for attr_id, mod in enumerate(MODALITIES):
        masked_rows = (attr_mask[:, attr_id] == 0)
        visible_rows = (attr_mask[:, attr_id] == 1)
        use_rows = masked_rows if loss_on_masked_only else visible_rows

        if mod == "cat":
            if use_rows.any():
                logits = cat_outs[cat_ptr][use_rows]
                target = caty[use_rows, cat_ptr]
                recon_loss = recon_loss + F.cross_entropy(logits, target)
                valid_terms += 1
            cat_ptr += 1

        elif mod == "num":
            valid_num = numy[:, num_ptr] != -1
            use_valid = use_rows & valid_num
            if use_valid.any():
                pred = num_outs[num_ptr][use_valid]
                target = numy[use_valid, num_ptr]
                recon_loss = recon_loss + F.mse_loss(pred, target)
                valid_terms += 1
            num_ptr += 1

    if valid_terms == 0:
        recon_loss = torch.tensor(0.0, device=H.device, requires_grad=True)

    marip_loss = torch.tensor(0.0, device=H.device)
    if marip_loss_fn is not None:
        marip_loss = compute_masked_marip_loss(
            cat_outs=cat_outs,
            num_outs=num_outs,
            cat_targets=caty,
            num_targets=numy,
            attr_mask=attr_mask,
            marip_loss_fn=marip_loss_fn,
            loss_on_masked_only=loss_on_masked_only
        )

    reg_loss = hidden_regularization(H, attr_mask=attr_mask)

    cat_consistency_loss = torch.tensor(0.0, device=H.device)
    if use_cat_class_consistency_loss:
        cat_consistency_loss = categorical_class_consistency_loss(
            H=H,
            cat_targets=caty,
            attr_mask=attr_mask,
            visible_only=CONSISTENCY_ON_VISIBLE_ONLY
        )

    if marip_loss_fn is not None:
        total_loss = (
            (1.0 - marip_blend) * recon_loss
            + marip_blend * marip_loss
            + reg_weight * reg_loss
            + cat_class_consistency_weight * cat_consistency_loss
        )
    else:
        total_loss = recon_loss + reg_weight * reg_loss + cat_class_consistency_weight * cat_consistency_loss

    return (
        total_loss,
        recon_loss.detach(),
        marip_loss.detach(),
        reg_loss.detach(),
        cat_consistency_loss.detach()
    )

# -----------------------------
# 23) EVALUATION HELPERS
# -----------------------------
def retrieve_text_from_memory(query_h, mem_emb, mem_vals):
    query_h = query_h / (query_h.norm(dim=1, keepdim=True) + 1e-8)
    mem_emb = mem_emb / (mem_emb.norm(dim=1, keepdim=True) + 1e-8)
    sim = torch.matmul(query_h, mem_emb.T)
    idx = sim.argmax(dim=1).cpu().tolist()
    return [mem_vals[i] for i in idx]

def evaluate_standard(model, X_eval, raw_num_eval, cat_val_eval, cat_true, num_true):
    model.eval()

    cat_preds = []
    num_preds = []

    with torch.no_grad():
        for s in range(0, len(X_eval), 256):
            llm = X_eval[s:s+256].to(DEVICE)
            rawn = raw_num_eval[s:s+256].to(DEVICE)
            cati = cat_val_eval[s:s+256].to(DEVICE)

            cat_outs, num_outs, _ = model(llm, rawn, cati, edge_index, attr_mask=None)

            cur_cat = [x.argmax(dim=-1).cpu().numpy() for x in cat_outs]
            cur_num = [x.cpu().numpy() for x in num_outs]

            if len(cur_cat) > 0:
                cat_preds.append(np.stack(cur_cat, axis=1))
            if len(cur_num) > 0:
                num_preds.append(np.stack(cur_num, axis=1))

    cat_preds = np.concatenate(cat_preds, axis=0) if len(cat_preds) else np.zeros((len(X_eval), 0), dtype=np.int64)
    num_preds = np.concatenate(num_preds, axis=0) if len(num_preds) else np.zeros((len(X_eval), 0), dtype=np.float32)

    cat_true_np = cat_true.cpu().numpy()
    num_true_np = num_true.cpu().numpy()

    overall_cat_f1 = None
    overall_cat_acc = None
    if cat_true_np.size > 0:
        overall_cat_f1 = float(f1_score(cat_true_np.reshape(-1), cat_preds.reshape(-1), average="weighted", zero_division=0))
        overall_cat_acc = float((cat_true_np.reshape(-1) == cat_preds.reshape(-1)).mean())

    overall_num_mae = None
    overall_num_mse = None
    if num_true_np.size > 0:
        valid = num_true_np != -1
        if valid.any():
            overall_num_mae = float(np.mean(np.abs(num_true_np[valid] - num_preds[valid])))
            overall_num_mse = float(np.mean((num_true_np[valid] - num_preds[valid]) ** 2))

    cat_attr_metrics = compute_weighted_cat_f1_per_attribute(cat_true_np, cat_preds, CATEGORICAL_COLUMNS)
    num_attr_metrics = compute_weighted_num_mae_per_attribute(num_true_np, num_preds, NUMERICAL_COLUMNS)

    return {
        "cat_f1_weighted_global": overall_cat_f1,
        "cat_acc_weighted_global": overall_cat_acc,
        "num_mae_weighted_global": overall_num_mae,
        "num_mse_weighted_global": overall_num_mse,
        "cat_attribute_f1": cat_attr_metrics,
        "num_attribute_mae": num_attr_metrics,
        "text_metrics": {
            "text_bleu_mean": None,
            "note": "Standard evaluation has no text generation head; BLEU is reported in masked text retrieval evaluation."
        }
    }

def evaluate_single_masked_attribute(model, X_eval, raw_num_eval, cat_val_eval, cat_true, num_true, eval_df, attr_id, batch_size=256):
    model.eval()

    all_cat_preds = []
    all_num_preds = []
    text_preds = []

    with torch.no_grad():
        for s in range(0, len(X_eval), batch_size):
            llm = X_eval[s:s+batch_size].to(DEVICE).clone()
            rawn = raw_num_eval[s:s+batch_size].to(DEVICE).clone()
            cati = cat_val_eval[s:s+batch_size].to(DEVICE).clone()

            B = llm.size(0)
            attr_mask = torch.ones(B, len(MODALITIES), device=DEVICE)
            attr_mask[:, attr_id] = 0.0

            llm, rawn, cati = apply_input_mask(llm, rawn, cati, attr_mask)

            if USE_MISSING_INIT:
                llm, rawn, cati = apply_memory_initialization(
                    llm=llm,
                    rawn=rawn,
                    cati=cati,
                    attr_mask=attr_mask,
                    memory_bank=model.memory_bank,
                    modality_types=MODALITIES,
                    k=MISSING_INIT_TOPK,
                    r=MISSING_INIT_R,
                )

            cat_outs, num_outs, H = model(llm, rawn, cati, edge_index, attr_mask=attr_mask)

            cur_cat = [x.argmax(dim=-1).cpu().numpy() for x in cat_outs]
            cur_num = [x.cpu().numpy() for x in num_outs]

            if len(cur_cat) > 0:
                all_cat_preds.append(np.stack(cur_cat, axis=1))
            if len(cur_num) > 0:
                all_num_preds.append(np.stack(cur_num, axis=1))

            if MODALITIES[attr_id] == "txt":
                mem_emb = model.memory_bank.get_embeddings_for_attr(attr_id)
                mem_vals = model.memory_bank.get_values_for_attr(attr_id)
                valid_mask = model.memory_bank.get_valid_mask_for_attr(attr_id)

                if mem_emb is not None and mem_vals is not None and len(mem_vals) > 0:
                    mem_emb = mem_emb[valid_mask]
                    mem_vals = [v for j, v in enumerate(mem_vals) if valid_mask[j].item()]
                    if len(mem_vals) > 0:
                        preds = retrieve_text_from_memory(H[:, attr_id, :], mem_emb.to(DEVICE), mem_vals)
                    else:
                        preds = [""] * B
                else:
                    preds = [""] * B

                text_preds.extend(preds)

    cat_preds = np.concatenate(all_cat_preds, axis=0) if len(all_cat_preds) else None
    num_preds = np.concatenate(all_num_preds, axis=0) if len(all_num_preds) else None

    if MODALITIES[attr_id] == "cat":
        local_cat = attr_id - len(TEXTUAL_COLUMNS)
        y_true = cat_true.cpu().numpy()[:, local_cat]
        y_pred = cat_preds[:, local_cat]

        evaluated_count = int(len(y_true))
        f1_val = float(f1_score(y_true, y_pred, average="weighted", zero_division=0))

        return {
            "attr_id": int(attr_id),
            "attr_name": CATEGORICAL_COLUMNS[local_cat],
            "attr_type": "cat",
            "weighted_f1": f1_val,
            "accuracy": float((y_true == y_pred).mean()),
            "evaluated_masked_count": evaluated_count
        }

    elif MODALITIES[attr_id] == "num":
        local_num = attr_id - len(TEXTUAL_COLUMNS) - len(CATEGORICAL_COLUMNS)
        y_true = num_true.cpu().numpy()[:, local_num]
        y_pred = num_preds[:, local_num]

        valid = y_true != -1
        valid_count = int(valid.sum())

        mae = np.mean(np.abs(y_true[valid] - y_pred[valid])) if valid.any() else None
        mse = np.mean((y_true[valid] - y_pred[valid]) ** 2) if valid.any() else None

        return {
            "attr_id": int(attr_id),
            "attr_name": NUMERICAL_COLUMNS[local_num],
            "attr_type": "num",
            "weighted_mae": float(mae) if mae is not None else None,
            "weighted_mse": float(mse) if mse is not None else None,
            "evaluated_masked_count": valid_count
        }

    else:
        local_txt = attr_id
        refs = eval_df[TEXTUAL_COLUMNS[local_txt]].fillna("").astype(str).tolist()
        bleu = compute_text_bleu(refs, text_preds)
        return {
            "attr_id": int(attr_id),
            "attr_name": TEXTUAL_COLUMNS[local_txt],
            "attr_type": "txt",
            "text_bleu_mean": float(bleu),
            "evaluated_masked_count": int(len(refs))
        }

def evaluate_full(model, eval_idx):
    X_eval = X_all[eval_idx]
    raw_num_eval = numerical_values_tensor[eval_idx]
    cat_val_eval = categorical_indices_tensor[eval_idx]
    cat_true = cat_targets_t[eval_idx]
    num_true = num_targets_t[eval_idx]
    eval_df = df.iloc[eval_idx].reset_index(drop=True)

    standard_results = evaluate_standard(
        model=model,
        X_eval=X_eval,
        raw_num_eval=raw_num_eval,
        cat_val_eval=cat_val_eval,
        cat_true=cat_true,
        num_true=num_true
    )

    masked_results = []
    for attr_id in range(len(MODALITIES)):
        res = evaluate_single_masked_attribute(
            model=model,
            X_eval=X_eval,
            raw_num_eval=raw_num_eval,
            cat_val_eval=cat_val_eval,
            cat_true=cat_true,
            num_true=num_true,
            eval_df=eval_df,
            attr_id=attr_id,
            batch_size=256
        )
        masked_results.append(res)

    cat_items = [
        x for x in masked_results
        if x["attr_type"] == "cat"
        and x.get("weighted_f1") is not None
        and x.get("evaluated_masked_count") is not None
    ]
    num_items = [
        x for x in masked_results
        if x["attr_type"] == "num"
        and x.get("weighted_mae") is not None
        and x.get("evaluated_masked_count") is not None
    ]
    txt_items = [
        x for x in masked_results
        if x["attr_type"] == "txt"
        and x.get("text_bleu_mean") is not None
        and x.get("evaluated_masked_count") is not None
    ]

    masked_weighted_attribute_f1, total_cat_weight = weighted_average_by_count(
        cat_items, "weighted_f1", "evaluated_masked_count"
    )
    masked_weighted_attribute_mae, total_num_weight = weighted_average_by_count(
        num_items, "weighted_mae", "evaluated_masked_count"
    )
    masked_weighted_attribute_mse, _ = weighted_average_by_count(
        num_items, "weighted_mse", "evaluated_masked_count"
    )
    masked_text_bleu_mean, total_txt_weight = weighted_average_by_count(
        txt_items, "text_bleu_mean", "evaluated_masked_count"
    )

    out = dict(standard_results)
    out["masked_attribute_results"] = masked_results
    out["masked_summary"] = {
        "masked_weighted_attribute_f1": masked_weighted_attribute_f1,
        "masked_weighted_attribute_f1_total_weight": total_cat_weight,
        "masked_weighted_attribute_mae": masked_weighted_attribute_mae,
        "masked_weighted_attribute_mae_total_weight": total_num_weight,
        "masked_weighted_attribute_mse": masked_weighted_attribute_mse,
        "masked_text_bleu_mean": masked_text_bleu_mean,
        "masked_text_bleu_total_weight": total_txt_weight
    }
    out["eval_loss"] = None
    return out

# -----------------------------
# 24) TRAINING
# -----------------------------
def train_ragnet(model, train_idx, test_idx, epochs=3, lr=1e-4, eval_every=1):
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    history = []
    best_score = -1e18
    best_path = os.path.join(MODEL_DIR, "adult_ragnet_best.pt")

    for epoch in range(epochs):
        model.train()
        total_train_loss = 0.0
        total_recon_loss = 0.0
        total_marip_loss = 0.0
        total_reg_loss = 0.0
        total_cat_consistency_loss = 0.0
        valid_train = 0

        shuffled = np.random.permutation(train_idx)

        for s in tqdm(range(0, len(shuffled), TRAIN_BATCH_SIZE), desc=f"Epoch {epoch+1}/{epochs}", leave=False):
            batch_ids = shuffled[s:s+TRAIN_BATCH_SIZE]

            llm = X_all[batch_ids].to(DEVICE).clone()
            rawn = numerical_values_tensor[batch_ids].to(DEVICE).clone()
            cati = categorical_indices_tensor[batch_ids].to(DEVICE).clone()
            caty = cat_targets_t[batch_ids].to(DEVICE)
            numy = num_targets_t[batch_ids].to(DEVICE)

            if USE_RANDOM_MASK_TRAINING:
                attr_mask = build_training_attr_mask(
                    batch_size=llm.size(0),
                    num_attrs=len(MODALITIES),
                    device=DEVICE,
                    mask_one_attr_per_sample=MASK_ONE_ATTR_PER_SAMPLE,
                    mask_prob=MASK_PROB
                )

                # remove original masked value/embedding
                llm, rawn, cati = apply_input_mask(llm, rawn, cati, attr_mask)

                # initialize masked attrs from memory
                if USE_MISSING_INIT:
                    llm, rawn, cati = apply_memory_initialization(
                        llm=llm,
                        rawn=rawn,
                        cati=cati,
                        attr_mask=attr_mask,
                        memory_bank=memory_bank,
                        modality_types=MODALITIES,
                        k=MISSING_INIT_TOPK,
                        r=MISSING_INIT_R,
                    )
            else:
                attr_mask = torch.ones(llm.size(0), len(MODALITIES), device=DEVICE)

            cat_outs, num_outs, H = model(llm, rawn, cati, edge_index, attr_mask=attr_mask)

            loss, recon_loss, marip_loss, reg_loss, cat_consistency_loss = compute_masked_training_loss(
                cat_outs=cat_outs,
                num_outs=num_outs,
                H=H,
                caty=caty,
                numy=numy,
                attr_mask=attr_mask,
                marip_loss_fn=marip_loss_fn if USE_MARIP else None,
                loss_on_masked_only=LOSS_ON_MASKED_ONLY,
                reg_weight=REG_WEIGHT,
                marip_blend=MARIP_BLEND,
                use_cat_class_consistency_loss=USE_CAT_CLASS_CONSISTENCY_LOSS,
                cat_class_consistency_weight=CAT_CLASS_CONSISTENCY_WEIGHT
            )

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            total_train_loss += float(loss.item())
            total_recon_loss += float(recon_loss.item())
            total_marip_loss += float(marip_loss.item())
            total_reg_loss += float(reg_loss.item())
            total_cat_consistency_loss += float(cat_consistency_loss.item())
            valid_train += 1

        avg_train_loss = total_train_loss / max(valid_train, 1)
        avg_recon_loss = total_recon_loss / max(valid_train, 1)
        avg_marip_loss = total_marip_loss / max(valid_train, 1)
        avg_reg_loss = total_reg_loss / max(valid_train, 1)
        avg_cat_consistency_loss = total_cat_consistency_loss / max(valid_train, 1)

        if (epoch + 1) % eval_every == 0:
            eval_results = evaluate_full(model, test_idx)
        else:
            eval_results = {"eval_loss": None, "masked_summary": {}}

        row = {
            "epoch": epoch + 1,
            "train_loss": avg_train_loss,
            "train_recon_loss": avg_recon_loss,
            "train_marip_loss": avg_marip_loss,
            "train_reg_loss": avg_reg_loss,
            "train_cat_consistency_loss": avg_cat_consistency_loss,
            "cat_f1_weighted_global": eval_results.get("cat_f1_weighted_global"),
            "num_mae_weighted_global": eval_results.get("num_mae_weighted_global"),
            "masked_weighted_attribute_f1": eval_results.get("masked_summary", {}).get("masked_weighted_attribute_f1"),
            "masked_weighted_attribute_mae": eval_results.get("masked_summary", {}).get("masked_weighted_attribute_mae"),
            "masked_text_bleu_mean": eval_results.get("masked_summary", {}).get("masked_text_bleu_mean"),
        }
        history.append(row)

        print(f"\n[Epoch {epoch+1}/{epochs}]")
        print(f"Train Loss                     : {avg_train_loss:.4f}")
        print(f"Train Recon Loss               : {avg_recon_loss:.4f}")
        print(f"Train MARIP Loss               : {avg_marip_loss:.4f}")
        print(f"Train Reg Loss                 : {avg_reg_loss:.6f}")
        print(f"Train Cat Consistency Loss     : {avg_cat_consistency_loss:.6f}")
        print(f"Cat F1 Weighted Global         : {eval_results.get('cat_f1_weighted_global', 0.0) or 0.0:.4f}")
        print(f"Num MAE Weighted Global        : {eval_results.get('num_mae_weighted_global', 0.0) or 0.0:.4f}")
        print(f"Masked Weighted Attribute F1   : {eval_results.get('masked_summary', {}).get('masked_weighted_attribute_f1', 0.0) or 0.0:.4f}")
        print(f"Masked Weighted Attribute MAE  : {eval_results.get('masked_summary', {}).get('masked_weighted_attribute_mae', 0.0) or 0.0:.4f}")
        print(f"Masked Text BLEU Mean          : {eval_results.get('masked_summary', {}).get('masked_text_bleu_mean', 0.0) or 0.0:.4f}")

        cur_score = 0.0
        if eval_results.get("masked_summary", {}).get("masked_weighted_attribute_f1") is not None:
            cur_score += float(eval_results["masked_summary"]["masked_weighted_attribute_f1"])
        if eval_results.get("masked_summary", {}).get("masked_text_bleu_mean") is not None:
            cur_score += float(eval_results["masked_summary"]["masked_text_bleu_mean"])
        if eval_results.get("masked_summary", {}).get("masked_weighted_attribute_mae") is not None:
            cur_score -= float(eval_results["masked_summary"]["masked_weighted_attribute_mae"])

        if cur_score > best_score:
            best_score = cur_score
            torch.save({"model_state_dict": model.state_dict(), "history": history}, best_path)

    return history

history = train_ragnet(model, train_idx, test_idx, epochs=EPOCHS, lr=LR, eval_every=1)

# -----------------------------
# 25) FINAL EVALUATION
# -----------------------------
final_results = evaluate_full(model, test_idx)

print("\n===== FINAL TEST RESULTS =====")
for k, v in final_results.items():
    if isinstance(v, dict):
        print(f"{k}: {v}")
    elif isinstance(v, list):
        print(f"{k}: {v[:3]} ... total={len(v)}")
    else:
        print(f"{k}: {v:.4f}" if isinstance(v, (int, float, np.floating)) and v is not None else f"{k}: {v}")

# -----------------------------
# 26) SAVE OUTPUTS
# -----------------------------
np.save(os.path.join(EMB_DIR, "textual_embeddings.npy"), textual_embeddings)
np.save(os.path.join(EMB_DIR, "categorical_embeddings.npy"), categorical_embeddings)
np.save(os.path.join(EMB_DIR, "numerical_embeddings.npy"), numerical_embeddings)
np.save(os.path.join(EMB_DIR, "categorical_indices.npy"), categorical_indices)
np.save(os.path.join(EMB_DIR, "numerical_values.npy"), numerical_values)

np.save(os.path.join(OUT_DIR, "all_embeddings.npy"), all_embeddings_np)
np.save(os.path.join(OUT_DIR, "normalized_embeddings.npy"), normalized_embeddings.detach().cpu().numpy())
np.save(os.path.join(OUT_DIR, "ground_truth_cat.npy"), ground_truth_cat)
np.save(os.path.join(OUT_DIR, "ground_truth_num.npy"), ground_truth_num)
np.save(os.path.join(OUT_DIR, "target_labels.npy"), target_labels)

torch.save(edge_index.detach().cpu(), os.path.join(OUT_DIR, "edge_index.pt"))
torch.save({"model_state_dict": model.state_dict()}, os.path.join(MODEL_DIR, "adult_ragnet_last.pt"))

with open(os.path.join(OUT_DIR, "history.json"), "w") as f:
    json.dump(history, f, indent=2)

with open(os.path.join(OUT_DIR, "final_results.json"), "w") as f:
    json.dump(final_results, f, indent=2, default=float)

metadata = {
    "textual_columns": TEXTUAL_COLUMNS,
    "categorical_columns": CATEGORICAL_COLUMNS,
    "numerical_columns": NUMERICAL_COLUMNS,
    "target_column": TARGET_COLUMN,
    "modalities": MODALITIES,
    "all_attribute_names": ALL_ATTRIBUTE_NAMES,
    "num_classes_per_cat_attr": num_classes_per_cat_attr,
    "target_map": target_map,
    "train_subset": len(train_idx),
    "test_subset": len(test_idx),
    "epochs": EPOCHS,
    "variant": "full-corrected-event-memory-balanced-same-class-embedding-missing-init-weighted-masked-metrics",
    "memory_size": memory_bank.size(),
    "memory_row_ids": memory_bank.get_row_ids(),
    "use_missing_init": USE_MISSING_INIT,
    "missing_init_topk": MISSING_INIT_TOPK,
    "missing_init_r": MISSING_INIT_R
}
with open(os.path.join(OUT_DIR, "metadata.json"), "w") as f:
    json.dump(metadata, f, indent=2)

print("\nSaved files to:", PROJECT_ROOT)
print("Done.")

DEVICE: cuda
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
  0%|          | 0/382 [00:00<?, ?it/s]/tmp/ipykernel_3132/325766195.py:285: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=(DEVICE.type == "cuda")):
  0%|          | 0/382 [00:00<?, ?it/s]/tmp/ipykernel_3132/325766195.py:285: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=(DEVICE.type == "cuda")):
  0%|          | 0/1 [00:00<?, ?it/s]/tmp/ipykernel_3132/325766195.py:285: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)

Event-based memory bank built. size = 300
First 10 memory row ids: [26662, 7294, 33203, 41883, 14541, 20337, 16504, 19919, 34051, 35081]
edge_index: torch.Size([2, 18])



[Epoch 1/20]
Train Loss                     : 1.7861
Train Recon Loss               : 1.8047
Train MARIP Loss               : 1.6081
Train Reg Loss                 : 0.554661
Train Cat Consistency Loss     : 0.100525
Cat F1 Weighted Global         : 0.9994
Num MAE Weighted Global        : 0.0348
Masked Weighted Attribute F1   : 0.5934
Masked Weighted Attribute MAE  : 0.6430
Masked Text BLEU Mean          : 0.2243



[Epoch 2/20]
Train Loss                     : 0.0231
Train Recon Loss               : 0.0215
Train MARIP Loss               : 0.0217
Train Reg Loss                 : 1.040728
Train Cat Consistency Loss     : 0.147256
Cat F1 Weighted Global         : 0.9999
Num MAE Weighted Global        : 0.0220
Masked Weighted Attribute F1   : 0.5974
Masked Weighted Attribute MAE  : 0.6468
Masked Text BLEU Mean          : 0.2210


...


[Epoch 16/20]
Train Loss                     : 0.0012
Train Recon Loss               : 0.0006
Train MARIP Loss               : 0.0005
Train Reg Loss                 : 1.181571
Train Cat Consistency Loss     : 0.046348
Cat F1 Weighted Global         : 1.0000
Num MAE Weighted Global        : 0.0078
Masked Weighted Attribute F1   : 0.5689
Masked Weighted Attribute MAE  : 0.6520
Masked Text BLEU Mean          : 0.2189



[Epoch 17/20]
Train Loss                     : 0.0012
Train Recon Loss               : 0.0006
Train MARIP Loss               : 0.0005
Train Reg Loss                 : 1.172485
Train Cat Consistency Loss     : 0.042124
Cat F1 Weighted Global         : 1.0000
Num MAE Weighted Global        : 0.0076
Masked Weighted Attribute F1   : 0.5685
Masked Weighted Attribute MAE  : 0.6522
Masked Text BLEU Mean          : 0.2185



[Epoch 18/20]
Train Loss                     : 0.0011
Train Recon Loss               : 0.0006
Train MARIP Loss               : 0.0004
Train Reg Loss                 : 1.160694
Train Cat Consistency Loss     : 0.038697
Cat F1 Weighted Global         : 1.0000
Num MAE Weighted Global        : 0.0077
Masked Weighted Attribute F1   : 0.5680
Masked Weighted Attribute MAE  : 0.6525
Masked Text BLEU Mean          : 0.2185



[Epoch 19/20]
Train Loss                     : 0.0010
Train Recon Loss               : 0.0005
Train MARIP Loss               : 0.0004
Train Reg Loss                 : 1.152472
Train Cat Consistency Loss     : 0.035070
Cat F1 Weighted Global         : 1.0000
Num MAE Weighted Global        : 0.0075
Masked Weighted Attribute F1   : 0.5675
Masked Weighted Attribute MAE  : 0.6520
Masked Text BLEU Mean          : 0.2184



[Epoch 20/20]
Train Loss                     : 0.0010
Train Recon Loss               : 0.0005
Train MARIP Loss               : 0.0004
Train Reg Loss                 : 1.142734
Train Cat Consistency Loss     : 0.032261
Cat F1 Weighted Global         : 1.0000
Num MAE Weighted Global        : 0.0072
Masked Weighted Attribute F1   : 0.5671
Masked Weighted Attribute MAE  : 0.6526
Masked Text BLEU Mean          : 0.2185

===== FINAL TEST RESULTS =====
cat_f1_weighted_global: 1.0000
cat_acc_weighted_global: 1.0000
num_mae_weighted_global: 0.0072
num_mse_weighted_global: 0.0001
cat_attribute_f1: {'workclass': 1.0, 'education': 1.0, 'occupation': 1.0, 'sex': 1.0, 'weighted_attribute_f1': 1.0}
num_attribute_mae: {'age': 0.007316933944821358, 'education_num': 0.005569198168814182, 'hours_per_week': 0.00866764783859253, 'weighted_attribute_mae': 0.0071845934726297855}
text_metrics: {'text_bleu_mean': None, 'note': 'Standard evaluation has no text generation head; BLEU is reported in masked text r

In [ ]:
# ============================================================
# ONE-CELL END-TO-END FAST RAGNET PIPELINE FOR AIRBNB DATASET
# FULL CORRECTED VERSION
#
# FEATURES
# - textual_columns = ["name"]
# - categorical_columns = ["neighbourhood", "room_type"]
# - numerical_columns = ["price", "latitude", "longitude", "availability_365"]
#
# FIXES INCLUDED
# - same categorical class of same attribute -> same input embedding
# - shared event-based memory bank storing only training row indices
# - selective balanced memory construction
# - original masked value + original PLM embedding are removed
# - masked attributes are initialized from memory via theory-based retrieval
# - project_batch does NOT zero masked attributes
# - masked categorical F1 weighted by evaluated masked count
# - masked numerical MAE/MSE weighted by evaluated masked count
# ============================================================

# -----------------------------
# 0) INSTALLS
# -----------------------------
!pip -q install transformers==4.38.2 nltk scikit-learn pandas numpy tqdm

# -----------------------------
# 1) IMPORTS
# -----------------------------
import os, re, json, math, random
from collections import defaultdict, Counter

import numpy as np
import pandas as pd
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score

import nltk
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
nltk.download("punkt", quiet=True)
try:
    nltk.download("punkt_tab", quiet=True)
except:
    pass

from transformers import AutoTokenizer, AutoModel
from google.colab import drive

# -----------------------------
# 2) CONFIG
# -----------------------------
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DATA_PATH = "/content/drive/MyDrive/airbnb/airbnb.csv"
SAVE_DIR = "/content/drive/MyDrive/airbnb/ragnet_ultra_fast"
os.makedirs(SAVE_DIR, exist_ok=True)

OUT_DIR = os.path.join(SAVE_DIR, "outputs")
MODEL_DIR = os.path.join(SAVE_DIR, "models")
EMB_DIR = os.path.join(SAVE_DIR, "embeddings")
for p in [OUT_DIR, MODEL_DIR, EMB_DIR]:
    os.makedirs(p, exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("DEVICE:", DEVICE)

TEXTUAL_COLUMNS = ["name"]
CATEGORICAL_COLUMNS = ["neighbourhood", "room_type"]
NUMERICAL_COLUMNS = ["price", "latitude", "longitude", "availability_365"]

# pseudo target used only for stratified split / balanced memory
TARGET_COLUMN = "room_type"

EMBED_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"

BATCH_SIZE_EMB = 128
TRAIN_BATCH_SIZE = 256
MAX_LEN_SUMMARY = 24
MAX_LEN_ATTR = 16

TRAIN_SUBSET = 50000
TEST_SUBSET = 10000

# for first debug run keep low, then increase
EPOCHS = 20
LR = 1e-4

TOP_K_EDGE = 2
MEMORY_LIMIT = 300

# training masking
USE_RANDOM_MASK_TRAINING = True
MASK_ONE_ATTR_PER_SAMPLE = True
MASK_PROB = 0.15

# IMPORTANT: loss on visible attrs only
LOSS_ON_MASKED_ONLY = False

# MARIP + regularization
USE_MARIP = True
MARIP_BLEND = 0.1
REG_WEIGHT = 1e-4

# same categorical class => same input embedding
USE_CLASS_LOOKUP_EMB = True

# same-class hidden consistency
USE_CAT_CLASS_CONSISTENCY_LOSS = True
CAT_CLASS_CONSISTENCY_WEIGHT = 1e-2
CONSISTENCY_ON_VISIBLE_ONLY = True

# memory-based missing init
USE_MISSING_INIT = True
MISSING_INIT_TOPK = 5
MISSING_INIT_R = 2

# balanced memory selection
MEMORY_NUM_BINS = 5

# -----------------------------
# 3) MOUNT DRIVE
# -----------------------------
drive.mount('/content/drive')

# -----------------------------
# 4) LOAD AIRBNB DATA
# -----------------------------
df = pd.read_csv(DATA_PATH)
print("Raw shape:", df.shape)

required_cols = TEXTUAL_COLUMNS + CATEGORICAL_COLUMNS + NUMERICAL_COLUMNS
missing_required = [c for c in required_cols if c not in df.columns]
if len(missing_required) > 0:
    raise ValueError(f"Missing required columns in CSV: {missing_required}")

df = df[required_cols].copy()

# -----------------------------
# 5) BASIC CLEANING
# -----------------------------
for col in TEXTUAL_COLUMNS + CATEGORICAL_COLUMNS:
    df[col] = df[col].fillna("Missing").astype(str).str.strip()
    df.loc[df[col] == "", col] = "Missing"

def parse_numeric_series(series):
    """
    Robust parser for numeric columns like '$1,234.00', '123', etc.
    """
    if pd.api.types.is_numeric_dtype(series):
        return pd.to_numeric(series, errors="coerce")

    cleaned = (
        series.astype(str)
        .str.replace(r"[^0-9\.\-]", "", regex=True)
        .replace({"": np.nan, "nan": np.nan, "None": np.nan})
    )
    return pd.to_numeric(cleaned, errors="coerce")

for col in NUMERICAL_COLUMNS:
    df[col] = parse_numeric_series(df[col])

# optional clipping for price outliers
if "price" in df.columns:
    valid_price = df["price"].dropna()
    if len(valid_price) > 0:
        low, high = valid_price.quantile([0.01, 0.99]).tolist()
        df["price"] = df["price"].clip(lower=low, upper=high)

# -----------------------------
# 6) ATTRIBUTE TEXTS
# -----------------------------
def make_attr_text(name, value):
    if pd.isna(value):
        return f"{name}: missing"
    return f"{name}: {value}"

for col in CATEGORICAL_COLUMNS:
    df[f"{col}_attr_text"] = df[col].apply(lambda x: make_attr_text(col, x))
for col in NUMERICAL_COLUMNS:
    df[f"{col}_attr_text"] = df[col].apply(lambda x: make_attr_text(col, x))

prepared_csv = os.path.join(SAVE_DIR, "airbnb_prepared.csv")
df[
    TEXTUAL_COLUMNS +
    CATEGORICAL_COLUMNS +
    NUMERICAL_COLUMNS +
    [f"{c}_attr_text" for c in CATEGORICAL_COLUMNS] +
    [f"{c}_attr_text" for c in NUMERICAL_COLUMNS]
].to_csv(prepared_csv, index=False)

# -----------------------------
# 7) ENCODE CATEGORICAL
# -----------------------------
cat_maps = {}
cat_encoded_df = pd.DataFrame()
num_classes_per_cat_attr = []

for col in CATEGORICAL_COLUMNS:
    le = LabelEncoder()
    enc = le.fit_transform(df[col].astype(str))
    cat_encoded_df[col] = enc
    cat_maps[col] = [str(x) for x in le.classes_]
    num_classes_per_cat_attr.append(len(le.classes_))

target_le = LabelEncoder()
target_labels = target_le.fit_transform(df[TARGET_COLUMN].astype(str))
target_map = {int(i): str(v) for i, v in enumerate(target_le.classes_)}

# -----------------------------
# 8) NUMERICAL VALUES
# -----------------------------
num_values_df = df[NUMERICAL_COLUMNS].copy()
num_values_raw = num_values_df.fillna(-1).to_numpy(dtype=np.float32)

scaled_num = np.zeros_like(num_values_raw, dtype=np.float32)
num_scale_stats = {}
for j in range(num_values_raw.shape[1]):
    col = num_values_raw[:, j]
    mask = col != -1
    if mask.sum() > 0:
        mean_j = col[mask].mean()
        std_j = col[mask].std()
        if std_j < 1e-8:
            std_j = 1.0
        scaled_num[mask, j] = (col[mask] - mean_j) / std_j
        scaled_num[~mask, j] = -1.0
        num_scale_stats[NUMERICAL_COLUMNS[j]] = {"mean": float(mean_j), "std": float(std_j)}
    else:
        scaled_num[:, j] = -1.0
        num_scale_stats[NUMERICAL_COLUMNS[j]] = {"mean": 0.0, "std": 1.0}

# -----------------------------
# 9) LOAD EMBEDDING MODEL
# -----------------------------
emb_tokenizer = AutoTokenizer.from_pretrained(EMBED_MODEL_NAME)
emb_model = AutoModel.from_pretrained(EMBED_MODEL_NAME).to(DEVICE)
if DEVICE.type == "cuda":
    emb_model = emb_model.half()
emb_model.eval()

def mean_pooling(model_output, attention_mask):
    token_embeddings = model_output.last_hidden_state
    mask = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
    return torch.sum(token_embeddings * mask, dim=1) / torch.clamp(mask.sum(dim=1), min=1e-9)

def embed_text_list(text_list, batch_size=128, max_length=16):
    all_embs = []
    for i in tqdm(range(0, len(text_list), batch_size), leave=False):
        batch = text_list[i:i+batch_size]
        encoded = emb_tokenizer(
            batch,
            padding=True,
            truncation=True,
            max_length=max_length,
            return_tensors="pt"
        ).to(DEVICE)
        with torch.no_grad():
            with torch.cuda.amp.autocast(enabled=(DEVICE.type == "cuda")):
                out = emb_model(**encoded)
                emb = mean_pooling(out, encoded["attention_mask"]).float()
        all_embs.append(emb.cpu().numpy())
        del encoded, out, emb
        if DEVICE.type == "cuda":
            torch.cuda.empty_cache()
    return np.vstack(all_embs)

# -----------------------------
# 10) COMPUTE EMBEDDINGS
# -----------------------------
textual_embeddings = np.stack([
    embed_text_list(df[col].fillna("").astype(str).tolist(), BATCH_SIZE_EMB, MAX_LEN_SUMMARY)
    for col in TEXTUAL_COLUMNS
], axis=1).astype(np.float32)

def build_same_class_categorical_embeddings(df, categorical_columns, batch_size=128, max_length=12):
    """
    same attribute + same class -> exactly same embedding
    """
    per_attr_embeddings = []
    categorical_class_embedding_maps = {}

    for col in categorical_columns:
        values = df[col].fillna("Missing").astype(str).str.strip().tolist()
        unique_vals = sorted(list(set(values)))
        unique_texts = [make_attr_text(col, v) for v in unique_vals]
        unique_embs = embed_text_list(unique_texts, batch_size=batch_size, max_length=max_length).astype(np.float32)
        val_to_emb = {v: unique_embs[i] for i, v in enumerate(unique_vals)}
        categorical_class_embedding_maps[col] = val_to_emb
        row_embs = np.stack([val_to_emb[v] for v in values], axis=0).astype(np.float32)
        per_attr_embeddings.append(row_embs)

    return np.stack(per_attr_embeddings, axis=1).astype(np.float32), categorical_class_embedding_maps

if USE_CLASS_LOOKUP_EMB:
    categorical_embeddings, categorical_class_embedding_maps = build_same_class_categorical_embeddings(
        df=df,
        categorical_columns=CATEGORICAL_COLUMNS,
        batch_size=BATCH_SIZE_EMB,
        max_length=MAX_LEN_ATTR
    )
else:
    categorical_embeddings = np.stack([
        embed_text_list(df[f"{col}_attr_text"].fillna("").astype(str).tolist(), BATCH_SIZE_EMB, MAX_LEN_ATTR)
        for col in CATEGORICAL_COLUMNS
    ], axis=1).astype(np.float32)
    categorical_class_embedding_maps = None

numerical_embeddings = np.stack([
    embed_text_list(df[f"{col}_attr_text"].fillna("").astype(str).tolist(), BATCH_SIZE_EMB, MAX_LEN_ATTR)
    for col in NUMERICAL_COLUMNS
], axis=1).astype(np.float32)

all_embeddings_np = np.concatenate([textual_embeddings, categorical_embeddings, numerical_embeddings], axis=1)
all_embeddings = torch.tensor(all_embeddings_np, dtype=torch.float32)

normalized_embeddings = all_embeddings.clone()
N, M, D = normalized_embeddings.shape
for m in range(M):
    attr = all_embeddings[:, m, :]
    nz = (attr.abs().sum(dim=1) != 0)
    if nz.sum() > 0:
        mean_m = attr[nz].mean(dim=0)
        std_m = attr[nz].std(dim=0)
        normalized_embeddings[nz, m, :] = (attr[nz] - mean_m) / (std_m + 1e-8)

# -----------------------------
# 11) BUILD GROUND TRUTH
# -----------------------------
ground_truth_text = df[TEXTUAL_COLUMNS].fillna("").astype(str).to_numpy()
ground_truth_cat = cat_encoded_df.to_numpy(dtype=np.int64)
ground_truth_num = scaled_num.astype(np.float32)

MODALITIES = ["txt"] * len(TEXTUAL_COLUMNS) + ["cat"] * len(CATEGORICAL_COLUMNS) + ["num"] * len(NUMERICAL_COLUMNS)
ALL_ATTRIBUTE_NAMES = TEXTUAL_COLUMNS + CATEGORICAL_COLUMNS + NUMERICAL_COLUMNS

categorical_indices = cat_encoded_df.to_numpy(dtype=np.int64)
numerical_values = num_values_raw.astype(np.float32)

categorical_indices_tensor = torch.tensor(categorical_indices, dtype=torch.long)
numerical_values_tensor = torch.tensor(numerical_values, dtype=torch.float32)

# -----------------------------
# 12) TRAIN/TEST SPLIT
# -----------------------------
indices = np.arange(len(df))
train_idx, test_idx = train_test_split(
    indices,
    test_size=0.2,
    random_state=SEED,
    stratify=target_labels
)

train_idx = train_idx[:min(TRAIN_SUBSET, len(train_idx))]
test_idx = test_idx[:min(TEST_SUBSET, len(test_idx))]
test_df = df.iloc[test_idx].reset_index(drop=True)

# -----------------------------
# 13) HELPERS
# -----------------------------
def safe_word_tokenize(x):
    try:
        return nltk.word_tokenize(str(x))
    except:
        return str(x).split()

def compute_text_bleu(text_refs, text_preds):
    smoothie = SmoothingFunction().method1
    scores = []
    for ref, pred in zip(text_refs, text_preds):
        ref_tokens = safe_word_tokenize(ref)
        pred_tokens = safe_word_tokenize(pred)
        if len(ref_tokens) == 0 or len(pred_tokens) == 0:
            scores.append(0.0)
            continue
        scores.append(sentence_bleu([ref_tokens], pred_tokens, smoothing_function=smoothie))
    return float(np.mean(scores)) if len(scores) else 0.0

def compute_weighted_cat_f1_per_attribute(cat_true, cat_pred, categorical_columns):
    results = {}
    scores = []
    for j, col in enumerate(categorical_columns):
        f1 = f1_score(cat_true[:, j], cat_pred[:, j], average="weighted", zero_division=0)
        results[col] = float(f1)
        scores.append(f1)
    results["weighted_attribute_f1"] = float(np.mean(scores)) if len(scores) else 0.0
    return results

def compute_weighted_num_mae_per_attribute(num_true, num_pred, numerical_columns):
    results = {}
    maes = []
    for j, col in enumerate(numerical_columns):
        valid = num_true[:, j] != -1
        if valid.any():
            mae = np.mean(np.abs(num_true[valid, j] - num_pred[valid, j]))
            results[col] = float(mae)
            maes.append(mae)
        else:
            results[col] = None
    results["weighted_attribute_mae"] = float(np.mean(maes)) if len(maes) else 0.0
    return results

def build_training_attr_mask(batch_size, num_attrs, device, mask_one_attr_per_sample=True, mask_prob=0.15):
    attr_mask = torch.ones(batch_size, num_attrs, device=device)
    if mask_one_attr_per_sample:
        masked_attr = torch.randint(0, num_attrs, (batch_size,), device=device)
        attr_mask[torch.arange(batch_size, device=device), masked_attr] = 0.0
    else:
        rand_mask = (torch.rand(batch_size, num_attrs, device=device) < mask_prob).float()
        empty_rows = rand_mask.sum(dim=1) == 0
        if empty_rows.any():
            row_ids = torch.where(empty_rows)[0]
            forced = torch.randint(0, num_attrs, (row_ids.numel(),), device=device)
            rand_mask[row_ids] = 0.0
            rand_mask[row_ids, forced] = 1.0
        attr_mask = 1.0 - rand_mask
    return attr_mask

def apply_input_mask(llm, rawn, cati, attr_mask):
    """
    remove original value + original embedding for masked attrs
    """
    llm = llm.clone()
    rawn = rawn.clone()
    cati = cati.clone()

    for attr_id, mod in enumerate(MODALITIES):
        masked_rows = attr_mask[:, attr_id] == 0
        if not masked_rows.any():
            continue

        llm[masked_rows, attr_id, :] = 0.0

        if mod == "cat":
            local_cat = attr_id - len(TEXTUAL_COLUMNS)
            cati[masked_rows, local_cat] = 0

        elif mod == "num":
            local_num = attr_id - len(TEXTUAL_COLUMNS) - len(CATEGORICAL_COLUMNS)
            rawn[masked_rows, local_num] = 0.0

    return llm, rawn, cati

def hidden_regularization(H, attr_mask=None):
    reg = (H ** 2).mean()
    if attr_mask is not None:
        masked_h = H * (1.0 - attr_mask.unsqueeze(-1))
        reg = reg + 0.1 * (masked_h ** 2).mean()
    return reg

def categorical_class_consistency_loss(H, cat_targets, attr_mask=None, visible_only=True):
    total_loss = 0.0
    total_terms = 0

    for j in range(len(CATEGORICAL_COLUMNS)):
        attr_id = len(TEXTUAL_COLUMNS) + j
        h = H[:, attr_id, :]
        y = cat_targets[:, j]

        if attr_mask is not None and visible_only:
            valid_rows = (attr_mask[:, attr_id] == 1)
        else:
            valid_rows = torch.ones(h.size(0), dtype=torch.bool, device=h.device)

        if valid_rows.sum() < 2:
            continue

        y_valid = y[valid_rows]
        h_valid = h[valid_rows]

        unique_classes = torch.unique(y_valid)
        for cls in unique_classes:
            cls_rows = (y_valid == cls)
            if cls_rows.sum() < 2:
                continue

            h_cls = h_valid[cls_rows]
            proto = h_cls.mean(dim=0, keepdim=True)
            total_loss = total_loss + ((h_cls - proto) ** 2).mean()
            total_terms += 1

    if total_terms == 0:
        return torch.tensor(0.0, device=H.device)

    return total_loss / total_terms

def weighted_average_by_count(items, value_key, count_key):
    valid_items = [
        x for x in items
        if x.get(value_key) is not None and x.get(count_key) is not None
    ]
    total_weight = sum(x[count_key] for x in valid_items)
    if total_weight <= 0:
        return None, 0
    score = sum(x[value_key] * x[count_key] for x in valid_items) / total_weight
    return float(score), int(total_weight)

def build_balanced_memory_row_ids(
    train_idx,
    df,
    categorical_columns,
    numerical_columns,
    target_column,
    memory_limit=300,
    bins_per_num=5,
    seed=42,
):
    """
    Build selective memory rows with broad coverage.
    """
    rng = np.random.RandomState(seed)
    train_idx = np.array(train_idx, dtype=int)
    train_df = df.iloc[train_idx].copy()

    selected = []
    selected_set = set()

    def add_row(rid):
        rid = int(rid)
        if rid not in selected_set and len(selected) < memory_limit:
            selected.append(rid)
            selected_set.add(rid)

    # 1) target coverage
    if target_column in train_df.columns:
        for _, sub in train_df.groupby(target_column):
            candidate_rows = sub.index.to_numpy()
            rng.shuffle(candidate_rows)
            if len(candidate_rows) > 0:
                add_row(candidate_rows[0])

    # 2) categorical coverage
    for col in categorical_columns:
        for _, sub in train_df.groupby(col):
            candidate_rows = sub.index.to_numpy()
            rng.shuffle(candidate_rows)
            if len(candidate_rows) > 0:
                add_row(candidate_rows[0])

    # 3) numerical bins
    for col in numerical_columns:
        valid_mask = train_df[col].notna()
        valid_df = train_df.loc[valid_mask, [col]]
        if len(valid_df) == 0:
            continue

        try:
            n_bins = min(bins_per_num, max(1, valid_df[col].nunique()))
            bin_ids = pd.qcut(valid_df[col], q=n_bins, duplicates="drop")
        except:
            continue

        tmp = valid_df.copy()
        tmp["_bin"] = bin_ids

        for _, sub in tmp.groupby("_bin"):
            candidate_rows = sub.index.to_numpy()
            rng.shuffle(candidate_rows)
            if len(candidate_rows) > 0:
                add_row(candidate_rows[0])

    # 4) strengthen rare categorical classes
    if len(selected) < memory_limit:
        for col in categorical_columns:
            value_counts = train_df[col].value_counts()
            rare_first = value_counts.sort_values().index.tolist()

            for val in rare_first:
                sub = train_df[train_df[col] == val]
                candidate_rows = sub.index.to_numpy()
                rng.shuffle(candidate_rows)
                for rid in candidate_rows[:2]:
                    add_row(rid)
                    if len(selected) >= memory_limit:
                        break
                if len(selected) >= memory_limit:
                    break
            if len(selected) >= memory_limit:
                break

    # 5) fill remaining randomly
    if len(selected) < memory_limit:
        remaining = [int(r) for r in train_idx if int(r) not in selected_set]
        rng.shuffle(remaining)
        for rid in remaining:
            add_row(rid)
            if len(selected) >= memory_limit:
                break

    return selected

# -----------------------------
# 14) EVENT-BASED MEMORY BANK
# -----------------------------
class EventMemoryBank:
    """
    Memory indexed by shared historical event ids.
    Stores only row ids; embeddings/values are retrieved lazily.
    """
    def __init__(self, normalized_embeddings, df, categorical_indices, numerical_values,
                 textual_columns, categorical_columns, numerical_columns, modalities):
        self.row_ids = []
        self.normalized_embeddings = normalized_embeddings
        self.df = df
        self.categorical_indices = categorical_indices
        self.numerical_values = numerical_values
        self.textual_columns = textual_columns
        self.categorical_columns = categorical_columns
        self.numerical_columns = numerical_columns
        self.modalities = modalities
        self.emb_cache = {}
        self.values_cache = {}
        self.valid_mask_cache = {}

    def set_row_ids(self, row_ids):
        self.row_ids = [int(x) for x in row_ids]
        self.emb_cache = {}
        self.values_cache = {}
        self.valid_mask_cache = {}

    def size(self):
        return len(self.row_ids)

    def get_row_ids(self):
        return self.row_ids

    def get_embeddings_for_attr(self, attr_id):
        if attr_id in self.emb_cache:
            return self.emb_cache[attr_id]
        if len(self.row_ids) == 0:
            return None
        emb = self.normalized_embeddings[self.row_ids, attr_id, :].detach().cpu()
        self.emb_cache[attr_id] = emb
        return emb

    def get_values_for_attr(self, attr_id):
        if attr_id in self.values_cache:
            return self.values_cache[attr_id]

        vals = []
        if self.modalities[attr_id] == "txt":
            local_txt = attr_id
            col = self.textual_columns[local_txt]
            for rid in self.row_ids:
                vals.append(str(self.df.iloc[rid][col]).strip())

        elif self.modalities[attr_id] == "cat":
            local_cat = attr_id - len(self.textual_columns)
            for rid in self.row_ids:
                vals.append(int(self.categorical_indices[rid, local_cat]))

        else:
            local_num = attr_id - len(self.textual_columns) - len(self.categorical_columns)
            for rid in self.row_ids:
                vals.append(float(self.numerical_values[rid, local_num]))

        self.values_cache[attr_id] = vals
        return vals

    def get_valid_mask_for_attr(self, attr_id):
        if attr_id in self.valid_mask_cache:
            return self.valid_mask_cache[attr_id]

        if len(self.row_ids) == 0:
            out = torch.zeros(0, dtype=torch.bool)
            self.valid_mask_cache[attr_id] = out
            return out

        mod = self.modalities[attr_id]

        if mod == "txt":
            local_txt = attr_id
            col = self.textual_columns[local_txt]
            mask = []
            for rid in self.row_ids:
                v = str(self.df.iloc[rid][col]).strip()
                mask.append(v != "")
            out = torch.tensor(mask, dtype=torch.bool)

        elif mod == "cat":
            out = torch.ones(len(self.row_ids), dtype=torch.bool)

        else:
            local_num = attr_id - len(self.textual_columns) - len(self.categorical_columns)
            mask = []
            for rid in self.row_ids:
                v = float(self.numerical_values[rid, local_num])
                mask.append(v != -1)
            out = torch.tensor(mask, dtype=torch.bool)

        self.valid_mask_cache[attr_id] = out
        return out

memory_bank = EventMemoryBank(
    normalized_embeddings=normalized_embeddings,
    df=df,
    categorical_indices=categorical_indices,
    numerical_values=numerical_values,
    textual_columns=TEXTUAL_COLUMNS,
    categorical_columns=CATEGORICAL_COLUMNS,
    numerical_columns=NUMERICAL_COLUMNS,
    modalities=MODALITIES
)

memory_row_ids = build_balanced_memory_row_ids(
    train_idx=train_idx,
    df=df,
    categorical_columns=CATEGORICAL_COLUMNS,
    numerical_columns=NUMERICAL_COLUMNS,
    target_column=TARGET_COLUMN,
    memory_limit=MEMORY_LIMIT,
    bins_per_num=MEMORY_NUM_BINS,
    seed=SEED,
)
memory_bank.set_row_ids(memory_row_ids)
print("Event-based memory bank built. size =", memory_bank.size())
print("First 10 memory row ids:", memory_row_ids[:10])

# -----------------------------
# 15) EDGE INDEX
# -----------------------------
def build_modality_based_edge_index(norm_embs, modality_types, top_k=2):
    N0, M0, D0 = norm_embs.shape
    attr_means = []
    for m in range(M0):
        e = norm_embs[:, m, :].detach().cpu()
        nz = (e.abs().sum(dim=1) != 0)
        if nz.any():
            mean_e = e[nz].mean(dim=0)
        else:
            mean_e = torch.zeros(D0)
        attr_means.append(mean_e)

    attr_means = torch.stack(attr_means, dim=0)
    attr_means = F.normalize(attr_means, dim=1)

    groups = defaultdict(list)
    for i, mod in enumerate(modality_types):
        groups[mod].append(i)

    edges = []
    for _, idxs in groups.items():
        if len(idxs) < 2:
            continue
        em = attr_means[idxs]
        sim = torch.matmul(em, em.T)
        for i, src in enumerate(idxs):
            top = torch.topk(sim[i], k=min(top_k + 1, len(idxs))).indices.tolist()
            for t in top:
                tgt = idxs[t]
                if src != tgt:
                    edges.append((src, tgt))
    edges += [(j, i) for (i, j) in edges]
    edges = list(set(edges))
    if len(edges) == 0:
        return torch.empty((2, 0), dtype=torch.long)
    return torch.tensor(edges, dtype=torch.long).T

edge_index = build_modality_based_edge_index(normalized_embeddings, MODALITIES, top_k=TOP_K_EDGE).to(DEVICE)
print("edge_index:", edge_index.shape)

# -----------------------------
# 16) THEORY-BASED MISSING INIT
# -----------------------------
def initialize_missing_embeddings_theory(
    missing_attr_idx,
    known_attr,
    known_embedding,
    memory_bank,
    modality_types,
    k=5,
    r=2,
    d_h=384,
):
    """
    keep memory event positions that appear in Top-K lists of at least r observed attrs
    then intersect with valid target-attribute memory
    initialize as mean of retrieved missing-attribute embeddings
    """
    device = known_embedding.device
    missing_modality = modality_types[missing_attr_idx]

    if known_embedding.dim() != 2:
        raise ValueError("known_embedding must be [num_attributes, d_h] for one sample")

    M_mem = memory_bank.size()
    if M_mem == 0:
        return torch.zeros(d_h, device=device), None, [], []

    topk_lists = []
    for i_obs in known_attr:
        mem_obs = memory_bank.get_embeddings_for_attr(i_obs)
        if mem_obs is None or mem_obs.size(0) == 0:
            continue

        mem_obs = mem_obs.to(device)
        non_zero_mask = mem_obs.abs().sum(dim=1) > 1e-6
        if not non_zero_mask.any():
            continue

        query = F.normalize(known_embedding[i_obs], dim=0)
        mem_norm = F.normalize(mem_obs[non_zero_mask], dim=1)
        sim = torch.matmul(mem_norm, query)

        topk_local = torch.topk(sim, k=min(k, sim.numel())).indices.tolist()
        valid_positions = torch.where(non_zero_mask)[0].tolist()
        topk_positions = [valid_positions[j] for j in topk_local]
        topk_lists.append(topk_positions)

    candidate_positions = []
    if len(topk_lists) > 0:
        counter = Counter()
        for lst in topk_lists:
            counter.update(lst)

        threshold = min(r, len(topk_lists))
        candidate_positions = [p for p, cnt in counter.items() if cnt >= threshold]

        if len(candidate_positions) == 0 and len(counter) > 0:
            max_count = max(counter.values())
            candidate_positions = [p for p, cnt in counter.items() if cnt == max_count]

    mem_missing = memory_bank.get_embeddings_for_attr(missing_attr_idx)
    if mem_missing is None or mem_missing.size(0) == 0:
        return torch.zeros(d_h, device=device), None, [], []

    mem_missing = mem_missing.to(device)
    valid_mask_missing = memory_bank.get_valid_mask_for_attr(missing_attr_idx).to(device)
    non_zero_missing = mem_missing.abs().sum(dim=1) > 1e-6
    valid_mask_missing = valid_mask_missing & non_zero_missing

    candidate_positions = [p for p in candidate_positions if p < mem_missing.size(0)]
    reliable_positions = [p for p in candidate_positions if valid_mask_missing[p].item()]

    if len(reliable_positions) == 0:
        reliable_positions = torch.where(valid_mask_missing)[0].cpu().tolist()

    if len(reliable_positions) == 0:
        return torch.zeros(d_h, device=device), None, [], []

    selected_embeddings = mem_missing[reliable_positions]
    init_embedding = selected_embeddings.mean(dim=0)

    init_value = None
    values_missing = memory_bank.get_values_for_attr(missing_attr_idx)
    selected_values = [values_missing[p] for p in reliable_positions]

    if missing_modality == "num":
        valid_values = []
        for v in selected_values:
            try:
                fv = float(v)
                if fv != -1:
                    valid_values.append(fv)
            except:
                pass
        if len(valid_values) > 0:
            init_value = sum(valid_values) / len(valid_values)

    elif missing_modality == "cat":
        valid_values = [int(v) for v in selected_values if v is not None]
        if len(valid_values) > 0:
            init_value = Counter(valid_values).most_common(1)[0][0]

    elif missing_modality == "txt":
        init_value = None

    retrieved_event_row_ids = [memory_bank.get_row_ids()[p] for p in reliable_positions]
    return init_embedding, init_value, reliable_positions, retrieved_event_row_ids

def apply_memory_initialization(
    llm,
    rawn,
    cati,
    attr_mask,
    memory_bank,
    modality_types,
    k=5,
    r=2,
):
    """
    after masking original info, fill masked attrs from memory
    """
    llm = llm.clone()
    rawn = rawn.clone()
    cati = cati.clone()

    B, M0, D0 = llm.shape
    device = llm.device

    for b in range(B):
        known_attr = torch.where(attr_mask[b] == 1)[0].tolist()
        missing_attr = torch.where(attr_mask[b] == 0)[0].tolist()

        for miss_id in missing_attr:
            init_emb, init_val, _, _ = initialize_missing_embeddings_theory(
                missing_attr_idx=miss_id,
                known_attr=known_attr,
                known_embedding=llm[b],
                memory_bank=memory_bank,
                modality_types=modality_types,
                k=k,
                r=r,
                d_h=D0,
            )

            llm[b, miss_id, :] = init_emb.to(device)

            if modality_types[miss_id] == "cat" and init_val is not None:
                local_cat = miss_id - len(TEXTUAL_COLUMNS)
                cati[b, local_cat] = int(init_val)

            elif modality_types[miss_id] == "num" and init_val is not None:
                local_num = miss_id - len(TEXTUAL_COLUMNS) - len(CATEGORICAL_COLUMNS)
                rawn[b, local_num] = float(init_val)

    return llm, rawn, cati

# -----------------------------
# 17) MARIP LOSS
# -----------------------------
class MARIPLoss(nn.Module):
    def __init__(self, alpha=0.1, epsilon=1e-6):
        super().__init__()
        self.alpha = alpha
        self.epsilon = epsilon
        self.class_counts = {}
        self.observed_freq = {}
        self.num_scale = 0.5
        self.cat_scale = 0.5

    def register_class_counts(self, attr_idx, class_labels):
        class_labels = np.asarray(class_labels)
        unique, counts = np.unique(class_labels, return_counts=True)
        self.class_counts[attr_idx] = dict(zip(unique.tolist(), counts.tolist()))

    def register_observed_frequency(self, attr_idx, observed_count, total):
        self.observed_freq[attr_idx] = observed_count / (total + self.epsilon)

    def compute_lambda(self, idx, modality, y_true_value=None):
        obs = self.observed_freq.get(idx, 0.1)
        lam_miss = (1.0 / (obs + self.epsilon)) ** 0.5
        lam_miss = float(np.clip(lam_miss, 0.75, 2.5))

        lam_cls = 1.0
        lam_num = 1.0

        if modality == "cat":
            if idx in self.class_counts and y_true_value is not None:
                counts_dict = self.class_counts[idx]
                count = counts_dict.get(int(y_true_value), 1)
                total_count = sum(counts_dict.values())
                num_classes = max(len(counts_dict), 1)
                avg_count = total_count / num_classes
                lam_cls = 1.0 + (avg_count / (count + self.epsilon)) ** 0.5
                lam_cls = float(np.clip(lam_cls, 1.0, 3.0))
            lam = lam_miss * lam_cls
        elif modality == "num":
            lam = lam_miss * lam_num
        else:
            lam = 1.0

        lam = float(np.clip(lam, 1.0, 3.0))
        return lam

marip_loss_fn = MARIPLoss(alpha=0.1)

for j, col in enumerate(CATEGORICAL_COLUMNS):
    global_attr_idx = len(TEXTUAL_COLUMNS) + j
    marip_loss_fn.register_class_counts(global_attr_idx, ground_truth_cat[:, j])

total_n = len(df)
for j, col in enumerate(CATEGORICAL_COLUMNS):
    global_attr_idx = len(TEXTUAL_COLUMNS) + j
    observed_count = int(df[col].notna().sum())
    marip_loss_fn.register_observed_frequency(global_attr_idx, observed_count, total_n)

for j, col in enumerate(NUMERICAL_COLUMNS):
    global_attr_idx = len(TEXTUAL_COLUMNS) + len(CATEGORICAL_COLUMNS) + j
    observed_count = int(df[col].notna().sum())
    marip_loss_fn.register_observed_frequency(global_attr_idx, observed_count, total_n)

def compute_masked_marip_loss(cat_outs, num_outs, cat_targets, num_targets, attr_mask, marip_loss_fn, loss_on_masked_only=False):
    total_loss = 0.0
    valid_count = 0

    for j, logits in enumerate(cat_outs):
        global_attr_idx = len(TEXTUAL_COLUMNS) + j
        y_true = cat_targets[:, j].long().to(logits.device)

        masked_rows = (attr_mask[:, global_attr_idx] == 0)
        visible_rows = (attr_mask[:, global_attr_idx] == 1)
        use_rows = masked_rows if loss_on_masked_only else visible_rows

        if not use_rows.any():
            continue

        logits_use = logits[use_rows]
        y_true_use = y_true[use_rows]
        rec_loss = F.cross_entropy(logits_use, y_true_use, reduction="none")

        lam_list = []
        for b in range(y_true_use.size(0)):
            lam = marip_loss_fn.compute_lambda(global_attr_idx, "cat", y_true_use[b].item())
            lam_list.append(lam)

        lam_tensor = torch.tensor(lam_list, dtype=rec_loss.dtype, device=rec_loss.device)
        total_loss = total_loss + marip_loss_fn.cat_scale * (lam_tensor * rec_loss).mean()
        valid_count += 1

    for j, pred in enumerate(num_outs):
        global_attr_idx = len(TEXTUAL_COLUMNS) + len(CATEGORICAL_COLUMNS) + j
        y_true = num_targets[:, j].to(pred.device)
        valid_num = y_true != -1

        masked_rows = (attr_mask[:, global_attr_idx] == 0)
        visible_rows = (attr_mask[:, global_attr_idx] == 1)
        use_rows = masked_rows if loss_on_masked_only else visible_rows
        use_valid = use_rows & valid_num

        if not use_valid.any():
            continue

        pred_use = pred[use_valid]
        y_true_use = y_true[use_valid]
        rec_loss = F.mse_loss(pred_use, y_true_use, reduction="none")

        lam_list = []
        for b in range(y_true_use.size(0)):
            lam = marip_loss_fn.compute_lambda(global_attr_idx, "num", float(y_true_use[b].item()))
            lam_list.append(lam)

        lam_tensor = torch.tensor(lam_list, dtype=rec_loss.dtype, device=rec_loss.device)
        total_loss = total_loss + marip_loss_fn.num_scale * (lam_tensor * rec_loss).mean()
        valid_count += 1

    if valid_count == 0:
        return torch.tensor(0.0, device=attr_mask.device, requires_grad=True)

    return total_loss

# -----------------------------
# 18) MODEL
# -----------------------------
class VectorizedCrossModalGNN(nn.Module):
    def __init__(self, d):
        super().__init__()
        self.Wg = nn.Linear(d, d, bias=False)
        self.Wq = nn.Linear(d, d, bias=False)
        self.Wk = nn.Linear(d, d, bias=False)
        self.Wc = nn.Linear(d, d, bias=False)
        self.a = nn.Parameter(torch.randn(2 * d))

    def forward(self, Z, edge_index):
        B, M0, D0 = Z.shape
        if edge_index.numel() == 0:
            Q = self.Wq(Z)
            K = self.Wk(Z)
            beta = F.softmax(torch.matmul(Q, K.transpose(1, 2)) / math.sqrt(D0), dim=-1)
            global_msg = torch.matmul(beta, self.Wc(Z))
            return F.leaky_relu(global_msg + Z)

        WgZ = self.Wg(Z)

        src = edge_index[0]
        tgt = edge_index[1]

        src_feat = WgZ[:, src, :]
        tgt_feat = WgZ[:, tgt, :]
        pair = torch.cat([src_feat, tgt_feat], dim=-1)
        e = F.leaky_relu(pair @ self.a)

        alpha = torch.zeros(B, M0, M0, device=Z.device)
        alpha[:, src, tgt] = e

        for i in range(M0):
            nbrs = tgt[src == i]
            if len(nbrs) > 0:
                alpha[:, i, nbrs] = F.softmax(alpha[:, i, nbrs], dim=-1)

        local_msg = torch.matmul(alpha, WgZ)

        Q = self.Wq(Z)
        K = self.Wk(Z)
        beta = F.softmax(torch.matmul(Q, K.transpose(1, 2)) / math.sqrt(D0), dim=-1)
        global_msg = torch.matmul(beta, self.Wc(Z))

        return F.leaky_relu(local_msg + global_msg + Z)

class FastRAGNetAirbnb(nn.Module):
    def __init__(self, embed_dim=384, d_shared=384, d_num=32, d_cat=16,
                 num_numerical_attrs=4, num_categorical_attrs=2,
                 num_classes_per_cat_attr=None, memory_bank=None):
        super().__init__()
        if num_classes_per_cat_attr is None:
            num_classes_per_cat_attr = [10] * num_categorical_attrs

        self.text_proj = nn.Linear(embed_dim, d_shared)

        self.num_proj = nn.ModuleList([
            nn.Sequential(nn.Linear(1, d_num), nn.ReLU(), nn.LayerNorm(d_num))
            for _ in range(num_numerical_attrs)
        ])
        self.cat_emb = nn.ModuleList([
            nn.Embedding(n_cls, d_cat) for n_cls in num_classes_per_cat_attr
        ])

        self.num_shared = nn.ModuleList([
            nn.Linear(embed_dim + d_num, d_shared) for _ in range(num_numerical_attrs)
        ])
        self.cat_shared = nn.ModuleList([
            nn.Linear(embed_dim + d_cat, d_shared) for _ in range(num_categorical_attrs)
        ])

        self.gnn = VectorizedCrossModalGNN(d_shared)
        self.memory_bank = memory_bank

        self.gate_h = nn.Linear(d_shared, d_shared)
        self.gate_m = nn.Linear(d_shared, d_shared)

        self.num_heads = nn.ModuleList([nn.Linear(d_shared, 1) for _ in range(num_numerical_attrs)])
        self.cat_heads = nn.ModuleList([nn.Linear(d_shared, n_cls) for n_cls in num_classes_per_cat_attr])

    def project_batch(self, llm_embeddings, raw_numericals, cat_indices):
        """
        pure projection only
        do NOT zero masked attrs here
        """
        Z = []
        num_idx = 0
        cat_idx = 0

        for i, modality in enumerate(MODALITIES):
            x = llm_embeddings[:, i, :]

            if modality == "txt":
                z = self.text_proj(x)

            elif modality == "cat":
                c = self.cat_emb[cat_idx](cat_indices[:, cat_idx])
                z = self.cat_shared[cat_idx](torch.cat([x, c], dim=-1))
                cat_idx += 1

            else:
                n = raw_numericals[:, num_idx].unsqueeze(-1)
                nvec = self.num_proj[num_idx](n)
                z = self.num_shared[num_idx](torch.cat([x, nvec], dim=-1))
                num_idx += 1

            Z.append(z)

        return torch.stack(Z, dim=1)

    def memory_fusion(self, H, attr_mask=None):
        if self.memory_bank is None:
            return H

        out = H.clone()
        B, M0, D0 = H.shape

        for i in range(M0):
            mem = self.memory_bank.get_embeddings_for_attr(i)
            if mem is None:
                continue

            mem = mem.to(H.device)
            valid_mask = self.memory_bank.get_valid_mask_for_attr(i).to(H.device)
            non_zero_mask = mem.abs().sum(dim=1) > 1e-8
            use_mask = valid_mask & non_zero_mask

            if use_mask.sum() == 0:
                continue

            mem = mem[use_mask]
            mem = mem / (mem.norm(dim=1, keepdim=True) + 1e-8)

            h = H[:, i, :]
            h_norm = h / (h.norm(dim=1, keepdim=True) + 1e-8)

            sim = torch.matmul(h_norm, mem.T)
            attn = F.softmax(sim, dim=-1)
            r = torch.matmul(attn, mem)

            g = torch.sigmoid(self.gate_h(h) + self.gate_m(r))
            out[:, i, :] = g * r + (1 - g) * h

        return out

    def forward(self, llm_embeddings, raw_numericals, cat_indices, edge_index, attr_mask=None):
        Z = self.project_batch(llm_embeddings, raw_numericals, cat_indices)
        H = self.gnn(Z, edge_index)
        H = self.memory_fusion(H, attr_mask=attr_mask)

        cat_outs = []
        num_outs = []

        cat_idx = 0
        num_idx = 0
        for i, modality in enumerate(MODALITIES):
            if modality == "cat":
                cat_outs.append(self.cat_heads[cat_idx](H[:, i, :]))
                cat_idx += 1
            elif modality == "num":
                num_outs.append(self.num_heads[num_idx](H[:, i, :]).squeeze(-1))
                num_idx += 1

        return cat_outs, num_outs, H

model = FastRAGNetAirbnb(
    embed_dim=384,
    d_shared=384,
    d_num=32,
    d_cat=16,
    num_numerical_attrs=len(NUMERICAL_COLUMNS),
    num_categorical_attrs=len(CATEGORICAL_COLUMNS),
    num_classes_per_cat_attr=num_classes_per_cat_attr,
    memory_bank=memory_bank
).to(DEVICE)

# -----------------------------
# 19) TRAIN TENSORS
# -----------------------------
X_all = normalized_embeddings

cat_targets_t = torch.tensor(ground_truth_cat, dtype=torch.long)
num_targets_t = torch.tensor(ground_truth_num, dtype=torch.float32)

raw_num_train = numerical_values_tensor[train_idx]
raw_num_test = numerical_values_tensor[test_idx]

cat_val_train = categorical_indices_tensor[train_idx]
cat_val_test = categorical_indices_tensor[test_idx]

# -----------------------------
# 20) TRAIN LOSS
# -----------------------------
def compute_masked_training_loss(cat_outs, num_outs, H, caty, numy, attr_mask,
                                 marip_loss_fn=None,
                                 loss_on_masked_only=False,
                                 reg_weight=1e-4,
                                 marip_blend=0.5,
                                 use_cat_class_consistency_loss=True,
                                 cat_class_consistency_weight=1e-2):
    recon_loss = 0.0
    valid_terms = 0

    cat_ptr = 0
    num_ptr = 0

    for attr_id, mod in enumerate(MODALITIES):
        masked_rows = (attr_mask[:, attr_id] == 0)
        visible_rows = (attr_mask[:, attr_id] == 1)
        use_rows = masked_rows if loss_on_masked_only else visible_rows

        if mod == "cat":
            if use_rows.any():
                logits = cat_outs[cat_ptr][use_rows]
                target = caty[use_rows, cat_ptr]
                recon_loss = recon_loss + F.cross_entropy(logits, target)
                valid_terms += 1
            cat_ptr += 1

        elif mod == "num":
            valid_num = numy[:, num_ptr] != -1
            use_valid = use_rows & valid_num
            if use_valid.any():
                pred = num_outs[num_ptr][use_valid]
                target = numy[use_valid, num_ptr]
                recon_loss = recon_loss + F.mse_loss(pred, target)
                valid_terms += 1
            num_ptr += 1

    if valid_terms == 0:
        recon_loss = torch.tensor(0.0, device=H.device, requires_grad=True)

    marip_loss = torch.tensor(0.0, device=H.device)
    if marip_loss_fn is not None:
        marip_loss = compute_masked_marip_loss(
            cat_outs=cat_outs,
            num_outs=num_outs,
            cat_targets=caty,
            num_targets=numy,
            attr_mask=attr_mask,
            marip_loss_fn=marip_loss_fn,
            loss_on_masked_only=loss_on_masked_only
        )

    reg_loss = hidden_regularization(H, attr_mask=attr_mask)

    cat_consistency_loss = torch.tensor(0.0, device=H.device)
    if use_cat_class_consistency_loss:
        cat_consistency_loss = categorical_class_consistency_loss(
            H=H,
            cat_targets=caty,
            attr_mask=attr_mask,
            visible_only=CONSISTENCY_ON_VISIBLE_ONLY
        )

    if marip_loss_fn is not None:
        total_loss = (
            (1.0 - marip_blend) * recon_loss
            + marip_blend * marip_loss
            + reg_weight * reg_loss
            + cat_class_consistency_weight * cat_consistency_loss
        )
    else:
        total_loss = recon_loss + reg_weight * reg_loss + cat_class_consistency_weight * cat_consistency_loss

    return (
        total_loss,
        recon_loss.detach(),
        marip_loss.detach(),
        reg_loss.detach(),
        cat_consistency_loss.detach()
    )

# -----------------------------
# 21) EVALUATION HELPERS
# -----------------------------
def retrieve_text_from_memory(query_h, mem_emb, mem_vals):
    query_h = query_h / (query_h.norm(dim=1, keepdim=True) + 1e-8)
    mem_emb = mem_emb / (mem_emb.norm(dim=1, keepdim=True) + 1e-8)
    sim = torch.matmul(query_h, mem_emb.T)
    idx = sim.argmax(dim=1).cpu().tolist()
    return [mem_vals[i] for i in idx]

def evaluate_standard(model, X_eval, raw_num_eval, cat_val_eval, cat_true, num_true):
    model.eval()

    cat_preds = []
    num_preds = []

    with torch.no_grad():
        for s in range(0, len(X_eval), 256):
            llm = X_eval[s:s+256].to(DEVICE)
            rawn = raw_num_eval[s:s+256].to(DEVICE)
            cati = cat_val_eval[s:s+256].to(DEVICE)

            cat_outs, num_outs, _ = model(llm, rawn, cati, edge_index, attr_mask=None)

            cur_cat = [x.argmax(dim=-1).cpu().numpy() for x in cat_outs]
            cur_num = [x.cpu().numpy() for x in num_outs]

            if len(cur_cat) > 0:
                cat_preds.append(np.stack(cur_cat, axis=1))
            if len(cur_num) > 0:
                num_preds.append(np.stack(cur_num, axis=1))

    cat_preds = np.concatenate(cat_preds, axis=0) if len(cat_preds) else np.zeros((len(X_eval), 0), dtype=np.int64)
    num_preds = np.concatenate(num_preds, axis=0) if len(num_preds) else np.zeros((len(X_eval), 0), dtype=np.float32)

    cat_true_np = cat_true.cpu().numpy()
    num_true_np = num_true.cpu().numpy()

    overall_cat_f1 = None
    overall_cat_acc = None
    if cat_true_np.size > 0:
        overall_cat_f1 = float(f1_score(cat_true_np.reshape(-1), cat_preds.reshape(-1), average="weighted", zero_division=0))
        overall_cat_acc = float((cat_true_np.reshape(-1) == cat_preds.reshape(-1)).mean())

    overall_num_mae = None
    overall_num_mse = None
    if num_true_np.size > 0:
        valid = num_true_np != -1
        if valid.any():
            overall_num_mae = float(np.mean(np.abs(num_true_np[valid] - num_preds[valid])))
            overall_num_mse = float(np.mean((num_true_np[valid] - num_preds[valid]) ** 2))

    cat_attr_metrics = compute_weighted_cat_f1_per_attribute(cat_true_np, cat_preds, CATEGORICAL_COLUMNS)
    num_attr_metrics = compute_weighted_num_mae_per_attribute(num_true_np, num_preds, NUMERICAL_COLUMNS)

    return {
        "cat_f1_weighted_global": overall_cat_f1,
        "cat_acc_weighted_global": overall_cat_acc,
        "num_mae_weighted_global": overall_num_mae,
        "num_mse_weighted_global": overall_num_mse,
        "cat_attribute_f1": cat_attr_metrics,
        "num_attribute_mae": num_attr_metrics,
        "text_metrics": {
            "text_bleu_mean": None,
            "note": "Standard evaluation has no text generation head; BLEU is reported in masked text retrieval evaluation."
        }
    }

def evaluate_single_masked_attribute(model, X_eval, raw_num_eval, cat_val_eval, cat_true, num_true, eval_df, attr_id, batch_size=256):
    model.eval()

    all_cat_preds = []
    all_num_preds = []
    text_preds = []

    with torch.no_grad():
        for s in range(0, len(X_eval), batch_size):
            llm = X_eval[s:s+batch_size].to(DEVICE).clone()
            rawn = raw_num_eval[s:s+batch_size].to(DEVICE).clone()
            cati = cat_val_eval[s:s+batch_size].to(DEVICE).clone()

            B = llm.size(0)
            attr_mask = torch.ones(B, len(MODALITIES), device=DEVICE)
            attr_mask[:, attr_id] = 0.0

            llm, rawn, cati = apply_input_mask(llm, rawn, cati, attr_mask)

            if USE_MISSING_INIT:
                llm, rawn, cati = apply_memory_initialization(
                    llm=llm,
                    rawn=rawn,
                    cati=cati,
                    attr_mask=attr_mask,
                    memory_bank=model.memory_bank,
                    modality_types=MODALITIES,
                    k=MISSING_INIT_TOPK,
                    r=MISSING_INIT_R,
                )

            cat_outs, num_outs, H = model(llm, rawn, cati, edge_index, attr_mask=attr_mask)

            cur_cat = [x.argmax(dim=-1).cpu().numpy() for x in cat_outs]
            cur_num = [x.cpu().numpy() for x in num_outs]

            if len(cur_cat) > 0:
                all_cat_preds.append(np.stack(cur_cat, axis=1))
            if len(cur_num) > 0:
                all_num_preds.append(np.stack(cur_num, axis=1))

            if MODALITIES[attr_id] == "txt":
                mem_emb = model.memory_bank.get_embeddings_for_attr(attr_id)
                mem_vals = model.memory_bank.get_values_for_attr(attr_id)
                valid_mask = model.memory_bank.get_valid_mask_for_attr(attr_id)

                if mem_emb is not None and mem_vals is not None and len(mem_vals) > 0:
                    mem_emb = mem_emb[valid_mask]
                    mem_vals = [v for j, v in enumerate(mem_vals) if valid_mask[j].item()]
                    if len(mem_vals) > 0:
                        preds = retrieve_text_from_memory(H[:, attr_id, :], mem_emb.to(DEVICE), mem_vals)
                    else:
                        preds = [""] * B
                else:
                    preds = [""] * B

                text_preds.extend(preds)

    cat_preds = np.concatenate(all_cat_preds, axis=0) if len(all_cat_preds) else None
    num_preds = np.concatenate(all_num_preds, axis=0) if len(all_num_preds) else None

    if MODALITIES[attr_id] == "cat":
        local_cat = attr_id - len(TEXTUAL_COLUMNS)
        y_true = cat_true.cpu().numpy()[:, local_cat]
        y_pred = cat_preds[:, local_cat]

        evaluated_count = int(len(y_true))
        f1_val = float(f1_score(y_true, y_pred, average="weighted", zero_division=0))

        return {
            "attr_id": int(attr_id),
            "attr_name": CATEGORICAL_COLUMNS[local_cat],
            "attr_type": "cat",
            "weighted_f1": f1_val,
            "accuracy": float((y_true == y_pred).mean()),
            "evaluated_masked_count": evaluated_count
        }

    elif MODALITIES[attr_id] == "num":
        local_num = attr_id - len(TEXTUAL_COLUMNS) - len(CATEGORICAL_COLUMNS)
        y_true = num_true.cpu().numpy()[:, local_num]
        y_pred = num_preds[:, local_num]

        valid = y_true != -1
        valid_count = int(valid.sum())

        mae = np.mean(np.abs(y_true[valid] - y_pred[valid])) if valid.any() else None
        mse = np.mean((y_true[valid] - y_pred[valid]) ** 2) if valid.any() else None

        return {
            "attr_id": int(attr_id),
            "attr_name": NUMERICAL_COLUMNS[local_num],
            "attr_type": "num",
            "weighted_mae": float(mae) if mae is not None else None,
            "weighted_mse": float(mse) if mse is not None else None,
            "evaluated_masked_count": valid_count
        }

    else:
        local_txt = attr_id
        refs = eval_df[TEXTUAL_COLUMNS[local_txt]].fillna("").astype(str).tolist()
        bleu = compute_text_bleu(refs, text_preds)
        return {
            "attr_id": int(attr_id),
            "attr_name": TEXTUAL_COLUMNS[local_txt],
            "attr_type": "txt",
            "text_bleu_mean": float(bleu),
            "evaluated_masked_count": int(len(refs))
        }

def evaluate_full(model, eval_idx):
    X_eval = X_all[eval_idx]
    raw_num_eval = numerical_values_tensor[eval_idx]
    cat_val_eval = categorical_indices_tensor[eval_idx]
    cat_true = cat_targets_t[eval_idx]
    num_true = num_targets_t[eval_idx]
    eval_df = df.iloc[eval_idx].reset_index(drop=True)

    standard_results = evaluate_standard(
        model=model,
        X_eval=X_eval,
        raw_num_eval=raw_num_eval,
        cat_val_eval=cat_val_eval,
        cat_true=cat_true,
        num_true=num_true
    )

    masked_results = []
    for attr_id in range(len(MODALITIES)):
        res = evaluate_single_masked_attribute(
            model=model,
            X_eval=X_eval,
            raw_num_eval=raw_num_eval,
            cat_val_eval=cat_val_eval,
            cat_true=cat_true,
            num_true=num_true,
            eval_df=eval_df,
            attr_id=attr_id,
            batch_size=256
        )
        masked_results.append(res)

    cat_items = [
        x for x in masked_results
        if x["attr_type"] == "cat"
        and x.get("weighted_f1") is not None
        and x.get("evaluated_masked_count") is not None
    ]
    num_items = [
        x for x in masked_results
        if x["attr_type"] == "num"
        and x.get("weighted_mae") is not None
        and x.get("evaluated_masked_count") is not None
    ]
    txt_items = [
        x for x in masked_results
        if x["attr_type"] == "txt"
        and x.get("text_bleu_mean") is not None
        and x.get("evaluated_masked_count") is not None
    ]

    masked_weighted_attribute_f1, total_cat_weight = weighted_average_by_count(
        cat_items, "weighted_f1", "evaluated_masked_count"
    )
    masked_weighted_attribute_mae, total_num_weight = weighted_average_by_count(
        num_items, "weighted_mae", "evaluated_masked_count"
    )
    masked_weighted_attribute_mse, _ = weighted_average_by_count(
        num_items, "weighted_mse", "evaluated_masked_count"
    )
    masked_text_bleu_mean, total_txt_weight = weighted_average_by_count(
        txt_items, "text_bleu_mean", "evaluated_masked_count"
    )

    out = dict(standard_results)
    out["masked_attribute_results"] = masked_results
    out["masked_summary"] = {
        "masked_weighted_attribute_f1": masked_weighted_attribute_f1,
        "masked_weighted_attribute_f1_total_weight": total_cat_weight,
        "masked_weighted_attribute_mae": masked_weighted_attribute_mae,
        "masked_weighted_attribute_mae_total_weight": total_num_weight,
        "masked_weighted_attribute_mse": masked_weighted_attribute_mse,
        "masked_text_bleu_mean": masked_text_bleu_mean,
        "masked_text_bleu_total_weight": total_txt_weight
    }
    out["eval_loss"] = None
    return out

# -----------------------------
# 22) TRAINING
# -----------------------------
def train_ragnet(model, train_idx, test_idx, epochs=3, lr=1e-4, eval_every=1):
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    history = []
    best_score = -1e18
    best_path = os.path.join(MODEL_DIR, "airbnb_ragnet_best.pt")

    for epoch in range(epochs):
        model.train()
        total_train_loss = 0.0
        total_recon_loss = 0.0
        total_marip_loss = 0.0
        total_reg_loss = 0.0
        total_cat_consistency_loss = 0.0
        valid_train = 0

        shuffled = np.random.permutation(train_idx)

        for s in tqdm(range(0, len(shuffled), TRAIN_BATCH_SIZE), desc=f"Epoch {epoch+1}/{epochs}", leave=False):
            batch_ids = shuffled[s:s+TRAIN_BATCH_SIZE]

            llm = X_all[batch_ids].to(DEVICE).clone()
            rawn = numerical_values_tensor[batch_ids].to(DEVICE).clone()
            cati = categorical_indices_tensor[batch_ids].to(DEVICE).clone()
            caty = cat_targets_t[batch_ids].to(DEVICE)
            numy = num_targets_t[batch_ids].to(DEVICE)

            if USE_RANDOM_MASK_TRAINING:
                attr_mask = build_training_attr_mask(
                    batch_size=llm.size(0),
                    num_attrs=len(MODALITIES),
                    device=DEVICE,
                    mask_one_attr_per_sample=MASK_ONE_ATTR_PER_SAMPLE,
                    mask_prob=MASK_PROB
                )

                llm, rawn, cati = apply_input_mask(llm, rawn, cati, attr_mask)

                if USE_MISSING_INIT:
                    llm, rawn, cati = apply_memory_initialization(
                        llm=llm,
                        rawn=rawn,
                        cati=cati,
                        attr_mask=attr_mask,
                        memory_bank=memory_bank,
                        modality_types=MODALITIES,
                        k=MISSING_INIT_TOPK,
                        r=MISSING_INIT_R,
                    )
            else:
                attr_mask = torch.ones(llm.size(0), len(MODALITIES), device=DEVICE)

            cat_outs, num_outs, H = model(llm, rawn, cati, edge_index, attr_mask=attr_mask)

            loss, recon_loss, marip_loss, reg_loss, cat_consistency_loss = compute_masked_training_loss(
                cat_outs=cat_outs,
                num_outs=num_outs,
                H=H,
                caty=caty,
                numy=numy,
                attr_mask=attr_mask,
                marip_loss_fn=marip_loss_fn if USE_MARIP else None,
                loss_on_masked_only=LOSS_ON_MASKED_ONLY,
                reg_weight=REG_WEIGHT,
                marip_blend=MARIP_BLEND,
                use_cat_class_consistency_loss=USE_CAT_CLASS_CONSISTENCY_LOSS,
                cat_class_consistency_weight=CAT_CLASS_CONSISTENCY_WEIGHT
            )

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            total_train_loss += float(loss.item())
            total_recon_loss += float(recon_loss.item())
            total_marip_loss += float(marip_loss.item())
            total_reg_loss += float(reg_loss.item())
            total_cat_consistency_loss += float(cat_consistency_loss.item())
            valid_train += 1

        avg_train_loss = total_train_loss / max(valid_train, 1)
        avg_recon_loss = total_recon_loss / max(valid_train, 1)
        avg_marip_loss = total_marip_loss / max(valid_train, 1)
        avg_reg_loss = total_reg_loss / max(valid_train, 1)
        avg_cat_consistency_loss = total_cat_consistency_loss / max(valid_train, 1)

        if (epoch + 1) % eval_every == 0:
            eval_results = evaluate_full(model, test_idx)
        else:
            eval_results = {"eval_loss": None, "masked_summary": {}}

        row = {
            "epoch": epoch + 1,
            "train_loss": avg_train_loss,
            "train_recon_loss": avg_recon_loss,
            "train_marip_loss": avg_marip_loss,
            "train_reg_loss": avg_reg_loss,
            "train_cat_consistency_loss": avg_cat_consistency_loss,
            "cat_f1_weighted_global": eval_results.get("cat_f1_weighted_global"),
            "num_mae_weighted_global": eval_results.get("num_mae_weighted_global"),
            "masked_weighted_attribute_f1": eval_results.get("masked_summary", {}).get("masked_weighted_attribute_f1"),
            "masked_weighted_attribute_mae": eval_results.get("masked_summary", {}).get("masked_weighted_attribute_mae"),
            "masked_text_bleu_mean": eval_results.get("masked_summary", {}).get("masked_text_bleu_mean"),
        }
        history.append(row)

        print(f"\n[Epoch {epoch+1}/{epochs}]")
        print(f"Train Loss                     : {avg_train_loss:.4f}")
        print(f"Train Recon Loss               : {avg_recon_loss:.4f}")
        print(f"Train MARIP Loss               : {avg_marip_loss:.4f}")
        print(f"Train Reg Loss                 : {avg_reg_loss:.6f}")
        print(f"Train Cat Consistency Loss     : {avg_cat_consistency_loss:.6f}")
        print(f"Cat F1 Weighted Global         : {eval_results.get('cat_f1_weighted_global', 0.0) or 0.0:.4f}")
        print(f"Num MAE Weighted Global        : {eval_results.get('num_mae_weighted_global', 0.0) or 0.0:.4f}")
        print(f"Masked Weighted Attribute F1   : {eval_results.get('masked_summary', {}).get('masked_weighted_attribute_f1', 0.0) or 0.0:.4f}")
        print(f"Masked Weighted Attribute MAE  : {eval_results.get('masked_summary', {}).get('masked_weighted_attribute_mae', 0.0) or 0.0:.4f}")
        print(f"Masked Text BLEU Mean          : {eval_results.get('masked_summary', {}).get('masked_text_bleu_mean', 0.0) or 0.0:.4f}")

        cur_score = 0.0
        if eval_results.get("masked_summary", {}).get("masked_weighted_attribute_f1") is not None:
            cur_score += float(eval_results["masked_summary"]["masked_weighted_attribute_f1"])
        if eval_results.get("masked_summary", {}).get("masked_text_bleu_mean") is not None:
            cur_score += float(eval_results["masked_summary"]["masked_text_bleu_mean"])
        if eval_results.get("masked_summary", {}).get("masked_weighted_attribute_mae") is not None:
            cur_score -= float(eval_results["masked_summary"]["masked_weighted_attribute_mae"])

        if cur_score > best_score:
            best_score = cur_score
            torch.save({"model_state_dict": model.state_dict(), "history": history}, best_path)

    return history

history = train_ragnet(model, train_idx, test_idx, epochs=EPOCHS, lr=LR, eval_every=1)

# -----------------------------
# 23) FINAL EVALUATION
# -----------------------------
final_results = evaluate_full(model, test_idx)

print("\n===== FINAL TEST RESULTS =====")
for k, v in final_results.items():
    if isinstance(v, dict):
        print(f"{k}: {v}")
    elif isinstance(v, list):
        print(f"{k}: {v[:3]} ... total={len(v)}")
    else:
        print(f"{k}: {v:.4f}" if isinstance(v, (int, float, np.floating)) and v is not None else f"{k}: {v}")

# -----------------------------
# 24) SAVE OUTPUTS
# -----------------------------
np.save(os.path.join(EMB_DIR, "textual_embeddings.npy"), textual_embeddings)
np.save(os.path.join(EMB_DIR, "categorical_embeddings.npy"), categorical_embeddings)
np.save(os.path.join(EMB_DIR, "numerical_embeddings.npy"), numerical_embeddings)
np.save(os.path.join(EMB_DIR, "categorical_indices.npy"), categorical_indices)
np.save(os.path.join(EMB_DIR, "numerical_values.npy"), numerical_values)

np.save(os.path.join(OUT_DIR, "all_embeddings.npy"), all_embeddings_np)
np.save(os.path.join(OUT_DIR, "normalized_embeddings.npy"), normalized_embeddings.detach().cpu().numpy())
np.save(os.path.join(OUT_DIR, "ground_truth_cat.npy"), ground_truth_cat)
np.save(os.path.join(OUT_DIR, "ground_truth_num.npy"), ground_truth_num)
np.save(os.path.join(OUT_DIR, "target_labels.npy"), target_labels)

torch.save(edge_index.detach().cpu(), os.path.join(OUT_DIR, "edge_index.pt"))
torch.save({"model_state_dict": model.state_dict()}, os.path.join(MODEL_DIR, "airbnb_ragnet_last.pt"))

with open(os.path.join(OUT_DIR, "history.json"), "w") as f:
    json.dump(history, f, indent=2)

with open(os.path.join(OUT_DIR, "final_results.json"), "w") as f:
    json.dump(final_results, f, indent=2, default=float)

metadata = {
    "data_path": DATA_PATH,
    "save_dir": SAVE_DIR,
    "textual_columns": TEXTUAL_COLUMNS,
    "categorical_columns": CATEGORICAL_COLUMNS,
    "numerical_columns": NUMERICAL_COLUMNS,
    "target_column": TARGET_COLUMN,
    "modalities": MODALITIES,
    "all_attribute_names": ALL_ATTRIBUTE_NAMES,
    "num_classes_per_cat_attr": num_classes_per_cat_attr,
    "target_map": target_map,
    "train_subset": len(train_idx),
    "test_subset": len(test_idx),
    "epochs": EPOCHS,
    "variant": "airbnb-corrected-event-memory-balanced-same-class-embedding-missing-init-weighted-masked-metrics",
    "memory_size": memory_bank.size(),
    "memory_row_ids": memory_bank.get_row_ids(),
    "use_missing_init": USE_MISSING_INIT,
    "missing_init_topk": MISSING_INIT_TOPK,
    "missing_init_r": MISSING_INIT_R
}
with open(os.path.join(OUT_DIR, "metadata.json"), "w") as f:
    json.dump(metadata, f, indent=2)

print("\nSaved files to:", SAVE_DIR)
print("Done.")

DEVICE: cuda
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Raw shape: (119981, 8)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
  0%|          | 0/938 [00:00<?, ?it/s]/tmp/ipykernel_3132/3195382695.py:268: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=(DEVICE.type == "cuda")):
  0%|          | 0/6 [00:00<?, ?it/s]/tmp/ipykernel_3132/3195382695.py:268: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=(DEVICE.type == "cuda")):
  0%|          | 0/1 [00:00<?, ?it/s]/tmp/ipykernel_3132/3195382695.py:268: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...

Event-based memory bank built. size = 300
First 10 memory row ids: [101976, 98220, 54622, 70126, 21617, 1613, 2723, 1973, 751, 2007]
edge_index: torch.Size([2, 10])



[Epoch 1/20]
Train Loss                     : 4.4267
Train Recon Loss               : 4.4920
Train MARIP Loss               : 3.8077
Train Reg Loss                 : 0.888904
Train Cat Consistency Loss     : 0.297867
Cat F1 Weighted Global         : 0.8253
Num MAE Weighted Global        : 0.1184
Masked Weighted Attribute F1   : 0.3337
Masked Weighted Attribute MAE  : 0.5166
Masked Text BLEU Mean          : 0.0031



[Epoch 2/20]
Train Loss                     : 1.2141
Train Recon Loss               : 1.2010
Train MARIP Loss               : 1.2715
Train Reg Loss                 : 2.288161
Train Cat Consistency Loss     : 0.581260
Cat F1 Weighted Global         : 0.9420
Num MAE Weighted Global        : 0.0848
Masked Weighted Attribute F1   : 0.3336
Masked Weighted Attribute MAE  : 0.5182
Masked Text BLEU Mean          : 0.0031


...


[Epoch 16/20]
Train Loss                     : 0.0104
Train Recon Loss               : 0.0094
Train MARIP Loss               : 0.0068
Train Reg Loss                 : 3.077846
Train Cat Consistency Loss     : 0.092268
Cat F1 Weighted Global         : 0.9987
Num MAE Weighted Global        : 0.0263
Masked Weighted Attribute F1   : 0.3448
Masked Weighted Attribute MAE  : 0.5125
Masked Text BLEU Mean          : 0.0033



[Epoch 17/20]
Train Loss                     : 0.0088
Train Recon Loss               : 0.0079
Train MARIP Loss               : 0.0057
Train Reg Loss                 : 3.064244
Train Cat Consistency Loss     : 0.076784
Cat F1 Weighted Global         : 0.9987
Num MAE Weighted Global        : 0.0252
Masked Weighted Attribute F1   : 0.3454
Masked Weighted Attribute MAE  : 0.5117
Masked Text BLEU Mean          : 0.0034



[Epoch 18/20]
Train Loss                     : 0.0079
Train Recon Loss               : 0.0071
Train MARIP Loss               : 0.0050
Train Reg Loss                 : 3.060618
Train Cat Consistency Loss     : 0.066533
Cat F1 Weighted Global         : 0.9988
Num MAE Weighted Global        : 0.0247
Masked Weighted Attribute F1   : 0.3453
Masked Weighted Attribute MAE  : 0.5114
Masked Text BLEU Mean          : 0.0034



[Epoch 19/20]
Train Loss                     : 0.0075
Train Recon Loss               : 0.0068
Train MARIP Loss               : 0.0047
Train Reg Loss                 : 3.061411
Train Cat Consistency Loss     : 0.058286
Cat F1 Weighted Global         : 0.9988
Num MAE Weighted Global        : 0.0245
Masked Weighted Attribute F1   : 0.3453
Masked Weighted Attribute MAE  : 0.5093
Masked Text BLEU Mean          : 0.0034



[Epoch 20/20]
Train Loss                     : 0.0071
Train Recon Loss               : 0.0065
Train MARIP Loss               : 0.0043
Train Reg Loss                 : 3.056529
Train Cat Consistency Loss     : 0.051889
Cat F1 Weighted Global         : 0.9988
Num MAE Weighted Global        : 0.0248
Masked Weighted Attribute F1   : 0.3454
Masked Weighted Attribute MAE  : 0.5101
Masked Text BLEU Mean          : 0.0033

===== FINAL TEST RESULTS =====
cat_f1_weighted_global: 0.9988
cat_acc_weighted_global: 0.9990
num_mae_weighted_global: 0.0248
num_mse_weighted_global: 0.0022
cat_attribute_f1: {'neighbourhood': 0.9975450828375783, 'room_type': 1.0, 'weighted_attribute_f1': 0.9987725414187891}
num_attribute_mae: {'price': 0.03904469311237335, 'latitude': 0.023876473307609558, 'longitude': 0.02128678932785988, 'availability_365': 0.017541879788041115, 'weighted_attribute_mae': 0.025437457486987114}
text_metrics: {'text_bleu_mean': None, 'note': 'Standard evaluation has no text generation head

In [ ]:
# ============================================================
# SHARED CORE: CORRECTED RAGNET PIPELINE FOR UCI TABULAR DATA
# ============================================================

!pip -q install transformers==4.38.2 nltk scikit-learn pandas numpy tqdm ucimlrepo

import os, re, json, math, random
from collections import defaultdict, Counter

import numpy as np
import pandas as pd
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score

import nltk
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
nltk.download("punkt", quiet=True)
try:
    nltk.download("punkt_tab", quiet=True)
except:
    pass

from transformers import AutoTokenizer, AutoModel
from google.colab import drive
from ucimlrepo import fetch_ucirepo

# -----------------------------
# GLOBAL CONFIG
# -----------------------------
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("DEVICE:", DEVICE)

EMBED_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"

BATCH_SIZE_EMB = 128
TRAIN_BATCH_SIZE = 256
MAX_LEN_SUMMARY = 32
MAX_LEN_ATTR = 16

TRAIN_SUBSET = 1000000
TEST_SUBSET = 100000

EPOCHS = 100
LR = 1e-4

TOP_K_EDGE = 2
MEMORY_LIMIT = 300

USE_RANDOM_MASK_TRAINING = True
MASK_ONE_ATTR_PER_SAMPLE = True
MASK_PROB = 0.15

LOSS_ON_MASKED_ONLY = False

USE_MARIP = True
MARIP_BLEND = 0.1
REG_WEIGHT = 1e-4

USE_CLASS_LOOKUP_EMB = True

USE_CAT_CLASS_CONSISTENCY_LOSS = True
CAT_CLASS_CONSISTENCY_WEIGHT = 1e-2
CONSISTENCY_ON_VISIBLE_ONLY = True

USE_MISSING_INIT = True
MISSING_INIT_TOPK = 5
MISSING_INIT_R = 2

MEMORY_NUM_BINS = 5

drive.mount('/content/drive')

# -----------------------------
# SHARED UTILS
# -----------------------------
def canonicalize_columns(df):
    out = df.copy()
    out.columns = [
        re.sub(r"[^a-z0-9]+", "_", str(c).strip().lower()).strip("_")
        for c in out.columns
    ]
    return out

def parse_numeric_series(series):
    if pd.api.types.is_numeric_dtype(series):
        return pd.to_numeric(series, errors="coerce")
    cleaned = (
        series.astype(str)
        .str.replace(r"[^0-9\.\-]", "", regex=True)
        .replace({"": np.nan, "nan": np.nan, "None": np.nan, "?": np.nan})
    )
    return pd.to_numeric(cleaned, errors="coerce")

def safe_text(x):
    if pd.isna(x):
        return "missing"
    x = str(x).strip()
    if x == "" or x == "?":
        return "missing"
    return x

def build_summary_text(df, use_columns, out_col="summary_text", prefix=None):
    def row_to_text(row):
        parts = []
        for c in use_columns:
            parts.append(f"{c}: {safe_text(row[c])}")
        txt = ", ".join(parts)
        if prefix is not None:
            txt = f"{prefix}. {txt}"
        return txt
    df[out_col] = df.apply(row_to_text, axis=1)
    return df

def make_attr_text(name, value):
    if pd.isna(value):
        return f"{name}: missing"
    return f"{name}: {value}"

def safe_word_tokenize(x):
    try:
        return nltk.word_tokenize(str(x))
    except:
        return str(x).split()

def compute_text_bleu(text_refs, text_preds):
    smoothie = SmoothingFunction().method1
    scores = []
    for ref, pred in zip(text_refs, text_preds):
        ref_tokens = safe_word_tokenize(ref)
        pred_tokens = safe_word_tokenize(pred)
        if len(ref_tokens) == 0 or len(pred_tokens) == 0:
            scores.append(0.0)
            continue
        scores.append(sentence_bleu([ref_tokens], pred_tokens, smoothing_function=smoothie))
    return float(np.mean(scores)) if len(scores) else 0.0

def compute_weighted_cat_f1_per_attribute(cat_true, cat_pred, categorical_columns):
    results = {}
    scores = []
    if len(categorical_columns) == 0:
        results["weighted_attribute_f1"] = None
        return results
    for j, col in enumerate(categorical_columns):
        f1 = f1_score(cat_true[:, j], cat_pred[:, j], average="weighted", zero_division=0)
        results[col] = float(f1)
        scores.append(f1)
    results["weighted_attribute_f1"] = float(np.mean(scores)) if len(scores) else 0.0
    return results

def compute_weighted_num_mae_per_attribute(num_true, num_pred, numerical_columns):
    results = {}
    maes = []
    if len(numerical_columns) == 0:
        results["weighted_attribute_mae"] = None
        return results
    for j, col in enumerate(numerical_columns):
        valid = num_true[:, j] != -1
        if valid.any():
            mae = np.mean(np.abs(num_true[valid, j] - num_pred[valid, j]))
            results[col] = float(mae)
            maes.append(mae)
        else:
            results[col] = None
    results["weighted_attribute_mae"] = float(np.mean(maes)) if len(maes) else 0.0
    return results

def build_training_attr_mask(batch_size, num_attrs, device, mask_one_attr_per_sample=True, mask_prob=0.15):
    attr_mask = torch.ones(batch_size, num_attrs, device=device)
    if mask_one_attr_per_sample:
        masked_attr = torch.randint(0, num_attrs, (batch_size,), device=device)
        attr_mask[torch.arange(batch_size, device=device), masked_attr] = 0.0
    else:
        rand_mask = (torch.rand(batch_size, num_attrs, device=device) < mask_prob).float()
        empty_rows = rand_mask.sum(dim=1) == 0
        if empty_rows.any():
            row_ids = torch.where(empty_rows)[0]
            forced = torch.randint(0, num_attrs, (row_ids.numel(),), device=device)
            rand_mask[row_ids] = 0.0
            rand_mask[row_ids, forced] = 1.0
        attr_mask = 1.0 - rand_mask
    return attr_mask

def apply_input_mask(llm, rawn, cati, attr_mask, textual_columns, categorical_columns):
    llm = llm.clone()
    rawn = rawn.clone()
    cati = cati.clone()

    num_text = len(textual_columns)
    num_cat = len(categorical_columns)

    total_attrs = attr_mask.shape[1]
    modalities = ["txt"] * num_text + ["cat"] * num_cat + ["num"] * (total_attrs - num_text - num_cat)

    for attr_id, mod in enumerate(modalities):
        masked_rows = attr_mask[:, attr_id] == 0
        if not masked_rows.any():
            continue

        llm[masked_rows, attr_id, :] = 0.0

        if mod == "cat":
            local_cat = attr_id - num_text
            cati[masked_rows, local_cat] = 0

        elif mod == "num":
            local_num = attr_id - num_text - num_cat
            rawn[masked_rows, local_num] = 0.0

    return llm, rawn, cati

def hidden_regularization(H, attr_mask=None):
    reg = (H ** 2).mean()
    if attr_mask is not None:
        masked_h = H * (1.0 - attr_mask.unsqueeze(-1))
        reg = reg + 0.1 * (masked_h ** 2).mean()
    return reg

def weighted_average_by_count(items, value_key, count_key):
    valid_items = [
        x for x in items
        if x.get(value_key) is not None and x.get(count_key) is not None
    ]
    total_weight = sum(x[count_key] for x in valid_items)
    if total_weight <= 0:
        return None, 0
    score = sum(x[value_key] * x[count_key] for x in valid_items) / total_weight
    return float(score), int(total_weight)

def build_balanced_memory_row_ids(
    train_idx,
    df,
    categorical_columns,
    numerical_columns,
    target_column,
    memory_limit=300,
    bins_per_num=5,
    seed=42,
):
    rng = np.random.RandomState(seed)
    train_idx = np.array(train_idx, dtype=int)
    train_df = df.iloc[train_idx].copy()

    selected = []
    selected_set = set()

    def add_row(rid):
        rid = int(rid)
        if rid not in selected_set and len(selected) < memory_limit:
            selected.append(rid)
            selected_set.add(rid)

    if target_column in train_df.columns:
        for _, sub in train_df.groupby(target_column):
            candidate_rows = sub.index.to_numpy()
            rng.shuffle(candidate_rows)
            if len(candidate_rows) > 0:
                add_row(candidate_rows[0])

    for col in categorical_columns:
        for _, sub in train_df.groupby(col):
            candidate_rows = sub.index.to_numpy()
            rng.shuffle(candidate_rows)
            if len(candidate_rows) > 0:
                add_row(candidate_rows[0])

    for col in numerical_columns:
        valid_mask = train_df[col].notna()
        valid_df = train_df.loc[valid_mask, [col]]
        if len(valid_df) == 0:
            continue
        try:
            n_bins = min(bins_per_num, max(1, valid_df[col].nunique()))
            bin_ids = pd.qcut(valid_df[col], q=n_bins, duplicates="drop")
        except:
            continue
        tmp = valid_df.copy()
        tmp["_bin"] = bin_ids
        for _, sub in tmp.groupby("_bin"):
            candidate_rows = sub.index.to_numpy()
            rng.shuffle(candidate_rows)
            if len(candidate_rows) > 0:
                add_row(candidate_rows[0])

    if len(selected) < memory_limit:
        for col in categorical_columns:
            value_counts = train_df[col].value_counts()
            rare_first = value_counts.sort_values().index.tolist()
            for val in rare_first:
                sub = train_df[train_df[col] == val]
                candidate_rows = sub.index.to_numpy()
                rng.shuffle(candidate_rows)
                for rid in candidate_rows[:2]:
                    add_row(rid)
                    if len(selected) >= memory_limit:
                        break
                if len(selected) >= memory_limit:
                    break
            if len(selected) >= memory_limit:
                break

    if len(selected) < memory_limit:
        remaining = [int(r) for r in train_idx if int(r) not in selected_set]
        rng.shuffle(remaining)
        for rid in remaining:
            add_row(rid)
            if len(selected) >= memory_limit:
                break

    return selected

# -----------------------------
# EMBEDDING MODEL
# -----------------------------
emb_tokenizer = AutoTokenizer.from_pretrained(EMBED_MODEL_NAME)
emb_model = AutoModel.from_pretrained(EMBED_MODEL_NAME).to(DEVICE)
if DEVICE.type == "cuda":
    emb_model = emb_model.half()
emb_model.eval()

def mean_pooling(model_output, attention_mask):
    token_embeddings = model_output.last_hidden_state
    mask = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
    return torch.sum(token_embeddings * mask, dim=1) / torch.clamp(mask.sum(dim=1), min=1e-9)

def embed_text_list(text_list, batch_size=128, max_length=16):
    all_embs = []
    for i in tqdm(range(0, len(text_list), batch_size), leave=False):
        batch = text_list[i:i+batch_size]
        encoded = emb_tokenizer(
            batch,
            padding=True,
            truncation=True,
            max_length=max_length,
            return_tensors="pt"
        ).to(DEVICE)
        with torch.no_grad():
            with torch.cuda.amp.autocast(enabled=(DEVICE.type == "cuda")):
                out = emb_model(**encoded)
                emb = mean_pooling(out, encoded["attention_mask"]).float()
        all_embs.append(emb.cpu().numpy())
        del encoded, out, emb
        if DEVICE.type == "cuda":
            torch.cuda.empty_cache()
    return np.vstack(all_embs)

# -----------------------------
# MEMORY BANK
# -----------------------------
class EventMemoryBank:
    def __init__(self, normalized_embeddings, df, categorical_indices, numerical_values,
                 textual_columns, categorical_columns, numerical_columns, modalities):
        self.row_ids = []
        self.normalized_embeddings = normalized_embeddings
        self.df = df
        self.categorical_indices = categorical_indices
        self.numerical_values = numerical_values
        self.textual_columns = textual_columns
        self.categorical_columns = categorical_columns
        self.numerical_columns = numerical_columns
        self.modalities = modalities
        self.emb_cache = {}
        self.values_cache = {}
        self.valid_mask_cache = {}

    def set_row_ids(self, row_ids):
        self.row_ids = [int(x) for x in row_ids]
        self.emb_cache = {}
        self.values_cache = {}
        self.valid_mask_cache = {}

    def size(self):
        return len(self.row_ids)

    def get_row_ids(self):
        return self.row_ids

    def get_embeddings_for_attr(self, attr_id):
        if attr_id in self.emb_cache:
            return self.emb_cache[attr_id]
        if len(self.row_ids) == 0:
            return None
        emb = self.normalized_embeddings[self.row_ids, attr_id, :].detach().cpu()
        self.emb_cache[attr_id] = emb
        return emb

    def get_values_for_attr(self, attr_id):
        if attr_id in self.values_cache:
            return self.values_cache[attr_id]

        vals = []
        num_text = len(self.textual_columns)
        num_cat = len(self.categorical_columns)

        if self.modalities[attr_id] == "txt":
            local_txt = attr_id
            col = self.textual_columns[local_txt]
            for rid in self.row_ids:
                vals.append(str(self.df.iloc[rid][col]).strip())

        elif self.modalities[attr_id] == "cat":
            local_cat = attr_id - num_text
            for rid in self.row_ids:
                vals.append(int(self.categorical_indices[rid, local_cat]))

        else:
            local_num = attr_id - num_text - num_cat
            for rid in self.row_ids:
                vals.append(float(self.numerical_values[rid, local_num]))

        self.values_cache[attr_id] = vals
        return vals

    def get_valid_mask_for_attr(self, attr_id):
        if attr_id in self.valid_mask_cache:
            return self.valid_mask_cache[attr_id]

        if len(self.row_ids) == 0:
            out = torch.zeros(0, dtype=torch.bool)
            self.valid_mask_cache[attr_id] = out
            return out

        num_text = len(self.textual_columns)
        num_cat = len(self.categorical_columns)
        mod = self.modalities[attr_id]

        if mod == "txt":
            local_txt = attr_id
            col = self.textual_columns[local_txt]
            mask = []
            for rid in self.row_ids:
                v = str(self.df.iloc[rid][col]).strip()
                mask.append(v != "")
            out = torch.tensor(mask, dtype=torch.bool)

        elif mod == "cat":
            out = torch.ones(len(self.row_ids), dtype=torch.bool)

        else:
            local_num = attr_id - num_text - num_cat
            mask = []
            for rid in self.row_ids:
                v = float(self.numerical_values[rid, local_num])
                mask.append(v != -1)
            out = torch.tensor(mask, dtype=torch.bool)

        self.valid_mask_cache[attr_id] = out
        return out

# -----------------------------
# THEORY-BASED INIT
# -----------------------------
def initialize_missing_embeddings_theory(
    missing_attr_idx,
    known_attr,
    known_embedding,
    memory_bank,
    modality_types,
    k=5,
    r=2,
    d_h=384,
):
    device = known_embedding.device
    missing_modality = modality_types[missing_attr_idx]

    if known_embedding.dim() != 2:
        raise ValueError("known_embedding must be [num_attributes, d_h] for one sample")

    M_mem = memory_bank.size()
    if M_mem == 0:
        return torch.zeros(d_h, device=device), None, [], []

    topk_lists = []
    for i_obs in known_attr:
        mem_obs = memory_bank.get_embeddings_for_attr(i_obs)
        if mem_obs is None or mem_obs.size(0) == 0:
            continue

        mem_obs = mem_obs.to(device)
        non_zero_mask = mem_obs.abs().sum(dim=1) > 1e-6
        if not non_zero_mask.any():
            continue

        query = F.normalize(known_embedding[i_obs], dim=0)
        mem_norm = F.normalize(mem_obs[non_zero_mask], dim=1)
        sim = torch.matmul(mem_norm, query)

        topk_local = torch.topk(sim, k=min(k, sim.numel())).indices.tolist()
        valid_positions = torch.where(non_zero_mask)[0].tolist()
        topk_positions = [valid_positions[j] for j in topk_local]
        topk_lists.append(topk_positions)

    candidate_positions = []
    if len(topk_lists) > 0:
        counter = Counter()
        for lst in topk_lists:
            counter.update(lst)

        threshold = min(r, len(topk_lists))
        candidate_positions = [p for p, cnt in counter.items() if cnt >= threshold]

        if len(candidate_positions) == 0 and len(counter) > 0:
            max_count = max(counter.values())
            candidate_positions = [p for p, cnt in counter.items() if cnt == max_count]

    mem_missing = memory_bank.get_embeddings_for_attr(missing_attr_idx)
    if mem_missing is None or mem_missing.size(0) == 0:
        return torch.zeros(d_h, device=device), None, [], []

    mem_missing = mem_missing.to(device)
    valid_mask_missing = memory_bank.get_valid_mask_for_attr(missing_attr_idx).to(device)
    non_zero_missing = mem_missing.abs().sum(dim=1) > 1e-6
    valid_mask_missing = valid_mask_missing & non_zero_missing

    candidate_positions = [p for p in candidate_positions if p < mem_missing.size(0)]
    reliable_positions = [p for p in candidate_positions if valid_mask_missing[p].item()]

    if len(reliable_positions) == 0:
        reliable_positions = torch.where(valid_mask_missing)[0].cpu().tolist()

    if len(reliable_positions) == 0:
        return torch.zeros(d_h, device=device), None, [], []

    selected_embeddings = mem_missing[reliable_positions]
    init_embedding = selected_embeddings.mean(dim=0)

    init_value = None
    values_missing = memory_bank.get_values_for_attr(missing_attr_idx)
    selected_values = [values_missing[p] for p in reliable_positions]

    if missing_modality == "num":
        valid_values = []
        for v in selected_values:
            try:
                fv = float(v)
                if fv != -1:
                    valid_values.append(fv)
            except:
                pass
        if len(valid_values) > 0:
            init_value = sum(valid_values) / len(valid_values)

    elif missing_modality == "cat":
        valid_values = [int(v) for v in selected_values if v is not None]
        if len(valid_values) > 0:
            init_value = Counter(valid_values).most_common(1)[0][0]

    elif missing_modality == "txt":
        init_value = None

    retrieved_event_row_ids = [memory_bank.get_row_ids()[p] for p in reliable_positions]
    return init_embedding, init_value, reliable_positions, retrieved_event_row_ids

def apply_memory_initialization(
    llm,
    rawn,
    cati,
    attr_mask,
    memory_bank,
    modality_types,
    textual_columns,
    categorical_columns,
    k=5,
    r=2,
):
    llm = llm.clone()
    rawn = rawn.clone()
    cati = cati.clone()

    B, M0, D0 = llm.shape
    device = llm.device
    num_text = len(textual_columns)
    num_cat = len(categorical_columns)

    for b in range(B):
        known_attr = torch.where(attr_mask[b] == 1)[0].tolist()
        missing_attr = torch.where(attr_mask[b] == 0)[0].tolist()

        for miss_id in missing_attr:
            init_emb, init_val, _, _ = initialize_missing_embeddings_theory(
                missing_attr_idx=miss_id,
                known_attr=known_attr,
                known_embedding=llm[b],
                memory_bank=memory_bank,
                modality_types=modality_types,
                k=k,
                r=r,
                d_h=D0,
            )

            llm[b, miss_id, :] = init_emb.to(device)

            if modality_types[miss_id] == "cat" and init_val is not None:
                local_cat = miss_id - num_text
                cati[b, local_cat] = int(init_val)

            elif modality_types[miss_id] == "num" and init_val is not None:
                local_num = miss_id - num_text - num_cat
                rawn[b, local_num] = float(init_val)

    return llm, rawn, cati

# -----------------------------
# MARIP
# -----------------------------
class MARIPLoss(nn.Module):
    def __init__(self, alpha=0.1, epsilon=1e-6):
        super().__init__()
        self.alpha = alpha
        self.epsilon = epsilon
        self.class_counts = {}
        self.observed_freq = {}
        self.num_scale = 0.5
        self.cat_scale = 0.5

    def register_class_counts(self, attr_idx, class_labels):
        class_labels = np.asarray(class_labels)
        unique, counts = np.unique(class_labels, return_counts=True)
        self.class_counts[attr_idx] = dict(zip(unique.tolist(), counts.tolist()))

    def register_observed_frequency(self, attr_idx, observed_count, total):
        self.observed_freq[attr_idx] = observed_count / (total + self.epsilon)

    def compute_lambda(self, idx, modality, y_true_value=None):
        obs = self.observed_freq.get(idx, 0.1)
        lam_miss = (1.0 / (obs + self.epsilon)) ** 0.5
        lam_miss = float(np.clip(lam_miss, 0.75, 2.5))

        lam_cls = 1.0
        lam_num = 1.0

        if modality == "cat":
            if idx in self.class_counts and y_true_value is not None:
                counts_dict = self.class_counts[idx]
                count = counts_dict.get(int(y_true_value), 1)
                total_count = sum(counts_dict.values())
                num_classes = max(len(counts_dict), 1)
                avg_count = total_count / num_classes
                lam_cls = 1.0 + (avg_count / (count + self.epsilon)) ** 0.5
                lam_cls = float(np.clip(lam_cls, 1.0, 3.0))
            lam = lam_miss * lam_cls
        elif modality == "num":
            lam = lam_miss * lam_num
        else:
            lam = 1.0

        lam = float(np.clip(lam, 1.0, 3.0))
        return lam

# -----------------------------
# MODEL
# -----------------------------
class VectorizedCrossModalGNN(nn.Module):
    def __init__(self, d):
        super().__init__()
        self.Wg = nn.Linear(d, d, bias=False)
        self.Wq = nn.Linear(d, d, bias=False)
        self.Wk = nn.Linear(d, d, bias=False)
        self.Wc = nn.Linear(d, d, bias=False)
        self.a = nn.Parameter(torch.randn(2 * d))

    def forward(self, Z, edge_index):
        B, M0, D0 = Z.shape
        if edge_index.numel() == 0:
            Q = self.Wq(Z)
            K = self.Wk(Z)
            beta = F.softmax(torch.matmul(Q, K.transpose(1, 2)) / math.sqrt(D0), dim=-1)
            global_msg = torch.matmul(beta, self.Wc(Z))
            return F.leaky_relu(global_msg + Z)

        WgZ = self.Wg(Z)

        src = edge_index[0]
        tgt = edge_index[1]

        src_feat = WgZ[:, src, :]
        tgt_feat = WgZ[:, tgt, :]
        pair = torch.cat([src_feat, tgt_feat], dim=-1)
        e = F.leaky_relu(pair @ self.a)

        alpha = torch.zeros(B, M0, M0, device=Z.device)
        alpha[:, src, tgt] = e

        for i in range(M0):
            nbrs = tgt[src == i]
            if len(nbrs) > 0:
                alpha[:, i, nbrs] = F.softmax(alpha[:, i, nbrs], dim=-1)

        local_msg = torch.matmul(alpha, WgZ)

        Q = self.Wq(Z)
        K = self.Wk(Z)
        beta = F.softmax(torch.matmul(Q, K.transpose(1, 2)) / math.sqrt(D0), dim=-1)
        global_msg = torch.matmul(beta, self.Wc(Z))

        return F.leaky_relu(local_msg + global_msg + Z)

class FastRAGNetGeneric(nn.Module):
    def __init__(self, embed_dim=384, d_shared=384, d_num=32, d_cat=16,
                 num_numerical_attrs=0, num_categorical_attrs=0,
                 num_classes_per_cat_attr=None, memory_bank=None, modalities=None):
        super().__init__()
        if num_classes_per_cat_attr is None:
            num_classes_per_cat_attr = []

        self.modalities = modalities
        self.text_proj = nn.Linear(embed_dim, d_shared)

        self.num_proj = nn.ModuleList([
            nn.Sequential(nn.Linear(1, d_num), nn.ReLU(), nn.LayerNorm(d_num))
            for _ in range(num_numerical_attrs)
        ])
        self.cat_emb = nn.ModuleList([
            nn.Embedding(n_cls, d_cat) for n_cls in num_classes_per_cat_attr
        ])

        self.num_shared = nn.ModuleList([
            nn.Linear(embed_dim + d_num, d_shared) for _ in range(num_numerical_attrs)
        ])
        self.cat_shared = nn.ModuleList([
            nn.Linear(embed_dim + d_cat, d_shared) for _ in range(num_categorical_attrs)
        ])

        self.gnn = VectorizedCrossModalGNN(d_shared)
        self.memory_bank = memory_bank

        self.gate_h = nn.Linear(d_shared, d_shared)
        self.gate_m = nn.Linear(d_shared, d_shared)

        self.num_heads = nn.ModuleList([nn.Linear(d_shared, 1) for _ in range(num_numerical_attrs)])
        self.cat_heads = nn.ModuleList([nn.Linear(d_shared, n_cls) for n_cls in num_classes_per_cat_attr])

    def project_batch(self, llm_embeddings, raw_numericals, cat_indices):
        Z = []
        num_idx = 0
        cat_idx = 0

        for i, modality in enumerate(self.modalities):
            x = llm_embeddings[:, i, :]

            if modality == "txt":
                z = self.text_proj(x)

            elif modality == "cat":
                c = self.cat_emb[cat_idx](cat_indices[:, cat_idx])
                z = self.cat_shared[cat_idx](torch.cat([x, c], dim=-1))
                cat_idx += 1

            else:
                n = raw_numericals[:, num_idx].unsqueeze(-1)
                nvec = self.num_proj[num_idx](n)
                z = self.num_shared[num_idx](torch.cat([x, nvec], dim=-1))
                num_idx += 1

            Z.append(z)

        return torch.stack(Z, dim=1)

    def memory_fusion(self, H, attr_mask=None):
        if self.memory_bank is None:
            return H

        out = H.clone()
        B, M0, D0 = H.shape

        for i in range(M0):
            mem = self.memory_bank.get_embeddings_for_attr(i)
            if mem is None:
                continue

            mem = mem.to(H.device)
            valid_mask = self.memory_bank.get_valid_mask_for_attr(i).to(H.device)
            non_zero_mask = mem.abs().sum(dim=1) > 1e-8
            use_mask = valid_mask & non_zero_mask

            if use_mask.sum() == 0:
                continue

            mem = mem[use_mask]
            mem = mem / (mem.norm(dim=1, keepdim=True) + 1e-8)

            h = H[:, i, :]
            h_norm = h / (h.norm(dim=1, keepdim=True) + 1e-8)

            sim = torch.matmul(h_norm, mem.T)
            attn = F.softmax(sim, dim=-1)
            r = torch.matmul(attn, mem)

            g = torch.sigmoid(self.gate_h(h) + self.gate_m(r))
            out[:, i, :] = g * r + (1 - g) * h

        return out

    def forward(self, llm_embeddings, raw_numericals, cat_indices, edge_index, attr_mask=None):
        Z = self.project_batch(llm_embeddings, raw_numericals, cat_indices)
        H = self.gnn(Z, edge_index)
        H = self.memory_fusion(H, attr_mask=attr_mask)

        cat_outs = []
        num_outs = []

        cat_idx = 0
        num_idx = 0
        for i, modality in enumerate(self.modalities):
            if modality == "cat":
                cat_outs.append(self.cat_heads[cat_idx](H[:, i, :]))
                cat_idx += 1
            elif modality == "num":
                num_outs.append(self.num_heads[num_idx](H[:, i, :]).squeeze(-1))
                num_idx += 1

        return cat_outs, num_outs, H

# -----------------------------
# RUNNER
# -----------------------------
def run_ragnet_dataset(
    df,
    dataset_name,
    save_dir,
    textual_columns,
    categorical_columns,
    numerical_columns,
    target_column,
    epochs=EPOCHS,
    lr=LR
):
    os.makedirs(save_dir, exist_ok=True)
    out_dir = os.path.join(save_dir, "outputs")
    model_dir = os.path.join(save_dir, "models")
    emb_dir = os.path.join(save_dir, "embeddings")
    for p in [out_dir, model_dir, emb_dir]:
        os.makedirs(p, exist_ok=True)

    df = df.copy().reset_index(drop=True)

    # clean
    for col in textual_columns + categorical_columns:
        df[col] = df[col].fillna("Missing").astype(str).str.strip()
        df.loc[df[col] == "", col] = "Missing"
        df.loc[df[col] == "?", col] = "Missing"

    for col in numerical_columns:
        df[col] = parse_numeric_series(df[col])

    if target_column not in df.columns:
        raise ValueError(f"target_column '{target_column}' not found")

    # target labels for split
    target_series = df[target_column].copy()
    if pd.api.types.is_numeric_dtype(target_series):
        if target_series.nunique(dropna=True) > 20:
            # bin numeric targets for stratified split only
            tmp = target_series.fillna(target_series.median() if target_series.notna().any() else 0.0)
            try:
                stratify_labels = pd.qcut(tmp, q=min(10, max(2, tmp.nunique())), duplicates="drop").astype(str)
            except:
                stratify_labels = pd.cut(tmp, bins=min(10, max(2, tmp.nunique())), duplicates="drop").astype(str)
        else:
            stratify_labels = target_series.fillna(-999).astype(str)
    else:
        stratify_labels = target_series.fillna("Missing").astype(str)

    # encode categoricals
    cat_maps = {}
    cat_encoded_df = pd.DataFrame(index=df.index)
    num_classes_per_cat_attr = []

    for col in categorical_columns:
        le = LabelEncoder()
        enc = le.fit_transform(df[col].astype(str))
        cat_encoded_df[col] = enc
        cat_maps[col] = [str(x) for x in le.classes_]
        num_classes_per_cat_attr.append(len(le.classes_))

    target_le = LabelEncoder()
    target_labels = target_le.fit_transform(stratify_labels.astype(str))
    target_map = {int(i): str(v) for i, v in enumerate(target_le.classes_)}

    # numerical arrays
    if len(numerical_columns) > 0:
        num_values_df = df[numerical_columns].copy()
        num_values_raw = num_values_df.fillna(-1).to_numpy(dtype=np.float32)
    else:
        num_values_raw = np.zeros((len(df), 0), dtype=np.float32)

    scaled_num = np.zeros_like(num_values_raw, dtype=np.float32)
    num_scale_stats = {}
    for j in range(num_values_raw.shape[1]):
        col = num_values_raw[:, j]
        mask = col != -1
        if mask.sum() > 0:
            mean_j = col[mask].mean()
            std_j = col[mask].std()
            if std_j < 1e-8:
                std_j = 1.0
            scaled_num[mask, j] = (col[mask] - mean_j) / std_j
            scaled_num[~mask, j] = -1.0
            num_scale_stats[numerical_columns[j]] = {"mean": float(mean_j), "std": float(std_j)}
        else:
            scaled_num[:, j] = -1.0
            num_scale_stats[numerical_columns[j]] = {"mean": 0.0, "std": 1.0}

    # embeddings
    textual_embeddings = np.stack([
        embed_text_list(df[col].fillna("").astype(str).tolist(), BATCH_SIZE_EMB, MAX_LEN_SUMMARY)
        for col in textual_columns
    ], axis=1).astype(np.float32)

    def build_same_class_categorical_embeddings_local(df, categorical_columns, batch_size=128, max_length=12):
        per_attr_embeddings = []
        categorical_class_embedding_maps = {}

        for col in categorical_columns:
            values = df[col].fillna("Missing").astype(str).str.strip().tolist()
            unique_vals = sorted(list(set(values)))
            unique_texts = [make_attr_text(col, v) for v in unique_vals]
            unique_embs = embed_text_list(unique_texts, batch_size=batch_size, max_length=max_length).astype(np.float32)
            val_to_emb = {v: unique_embs[i] for i, v in enumerate(unique_vals)}
            categorical_class_embedding_maps[col] = val_to_emb
            row_embs = np.stack([val_to_emb[v] for v in values], axis=0).astype(np.float32)
            per_attr_embeddings.append(row_embs)

        if len(per_attr_embeddings) == 0:
            return np.zeros((len(df), 0, 384), dtype=np.float32), {}
        return np.stack(per_attr_embeddings, axis=1).astype(np.float32), categorical_class_embedding_maps

    if len(categorical_columns) > 0:
        if USE_CLASS_LOOKUP_EMB:
            categorical_embeddings, categorical_class_embedding_maps = build_same_class_categorical_embeddings_local(
                df=df,
                categorical_columns=categorical_columns,
                batch_size=BATCH_SIZE_EMB,
                max_length=MAX_LEN_ATTR
            )
        else:
            categorical_embeddings = np.stack([
                embed_text_list(df[f"{col}_attr_text"].fillna("").astype(str).tolist(), BATCH_SIZE_EMB, MAX_LEN_ATTR)
                for col in categorical_columns
            ], axis=1).astype(np.float32)
            categorical_class_embedding_maps = None
    else:
        categorical_embeddings = np.zeros((len(df), 0, 384), dtype=np.float32)
        categorical_class_embedding_maps = None

    if len(numerical_columns) > 0:
        for col in numerical_columns:
            df[f"{col}_attr_text"] = df[col].apply(lambda x: make_attr_text(col, x))
        numerical_embeddings = np.stack([
            embed_text_list(df[f"{col}_attr_text"].fillna("").astype(str).tolist(), BATCH_SIZE_EMB, MAX_LEN_ATTR)
            for col in numerical_columns
        ], axis=1).astype(np.float32)
    else:
        numerical_embeddings = np.zeros((len(df), 0, 384), dtype=np.float32)

    all_embeddings_np = np.concatenate([textual_embeddings, categorical_embeddings, numerical_embeddings], axis=1)
    all_embeddings = torch.tensor(all_embeddings_np, dtype=torch.float32)

    normalized_embeddings = all_embeddings.clone()
    N, M, D = normalized_embeddings.shape
    for m in range(M):
        attr = all_embeddings[:, m, :]
        nz = (attr.abs().sum(dim=1) != 0)
        if nz.sum() > 0:
            mean_m = attr[nz].mean(dim=0)
            std_m = attr[nz].std(dim=0)
            normalized_embeddings[nz, m, :] = (attr[nz] - mean_m) / (std_m + 1e-8)

    # ground truth
    ground_truth_text = df[textual_columns].fillna("").astype(str).to_numpy()
    ground_truth_cat = cat_encoded_df.to_numpy(dtype=np.int64) if len(categorical_columns) > 0 else np.zeros((len(df), 0), dtype=np.int64)
    ground_truth_num = scaled_num.astype(np.float32)

    modalities = ["txt"] * len(textual_columns) + ["cat"] * len(categorical_columns) + ["num"] * len(numerical_columns)
    all_attribute_names = textual_columns + categorical_columns + numerical_columns

    categorical_indices = cat_encoded_df.to_numpy(dtype=np.int64) if len(categorical_columns) > 0 else np.zeros((len(df), 0), dtype=np.int64)
    numerical_values = num_values_raw.astype(np.float32)

    categorical_indices_tensor = torch.tensor(categorical_indices, dtype=torch.long)
    numerical_values_tensor = torch.tensor(numerical_values, dtype=torch.float32)

    # split
    indices = np.arange(len(df))
    train_idx, test_idx = train_test_split(
        indices, test_size=0.2, random_state=SEED, stratify=target_labels
    )
    train_idx = train_idx[:min(TRAIN_SUBSET, len(train_idx))]
    test_idx = test_idx[:min(TEST_SUBSET, len(test_idx))]
    test_df = df.iloc[test_idx].reset_index(drop=True)

    # memory
    memory_bank = EventMemoryBank(
        normalized_embeddings=normalized_embeddings,
        df=df,
        categorical_indices=categorical_indices,
        numerical_values=numerical_values,
        textual_columns=textual_columns,
        categorical_columns=categorical_columns,
        numerical_columns=numerical_columns,
        modalities=modalities
    )

    memory_row_ids = build_balanced_memory_row_ids(
        train_idx=train_idx,
        df=df,
        categorical_columns=categorical_columns,
        numerical_columns=numerical_columns,
        target_column=target_column,
        memory_limit=MEMORY_LIMIT,
        bins_per_num=MEMORY_NUM_BINS,
        seed=SEED,
    )
    memory_bank.set_row_ids(memory_row_ids)

    # edge index
    def build_modality_based_edge_index(norm_embs, modality_types, top_k=2):
        N0, M0, D0 = norm_embs.shape
        attr_means = []
        for m in range(M0):
            e = norm_embs[:, m, :].detach().cpu()
            nz = (e.abs().sum(dim=1) != 0)
            if nz.any():
                mean_e = e[nz].mean(dim=0)
            else:
                mean_e = torch.zeros(D0)
            attr_means.append(mean_e)

        attr_means = torch.stack(attr_means, dim=0)
        attr_means = F.normalize(attr_means, dim=1)

        groups = defaultdict(list)
        for i, mod in enumerate(modality_types):
            groups[mod].append(i)

        edges = []
        for _, idxs in groups.items():
            if len(idxs) < 2:
                continue
            em = attr_means[idxs]
            sim = torch.matmul(em, em.T)
            for i, src in enumerate(idxs):
                top = torch.topk(sim[i], k=min(top_k + 1, len(idxs))).indices.tolist()
                for t in top:
                    tgt = idxs[t]
                    if src != tgt:
                        edges.append((src, tgt))
        edges += [(j, i) for (i, j) in edges]
        edges = list(set(edges))
        if len(edges) == 0:
            return torch.empty((2, 0), dtype=torch.long)
        return torch.tensor(edges, dtype=torch.long).T

    edge_index = build_modality_based_edge_index(normalized_embeddings, modalities, top_k=TOP_K_EDGE).to(DEVICE)

    # MARIP
    marip_loss_fn = MARIPLoss(alpha=0.1)

    for j, col in enumerate(categorical_columns):
        global_attr_idx = len(textual_columns) + j
        marip_loss_fn.register_class_counts(global_attr_idx, ground_truth_cat[:, j])

    total_n = len(df)
    for j, col in enumerate(categorical_columns):
        global_attr_idx = len(textual_columns) + j
        observed_count = int(df[col].notna().sum())
        marip_loss_fn.register_observed_frequency(global_attr_idx, observed_count, total_n)

    for j, col in enumerate(numerical_columns):
        global_attr_idx = len(textual_columns) + len(categorical_columns) + j
        observed_count = int(df[col].notna().sum())
        marip_loss_fn.register_observed_frequency(global_attr_idx, observed_count, total_n)

    def compute_masked_marip_loss(cat_outs, num_outs, cat_targets, num_targets, attr_mask, marip_loss_fn, loss_on_masked_only=False):
        total_loss = 0.0
        valid_count = 0

        for j, logits in enumerate(cat_outs):
            global_attr_idx = len(textual_columns) + j
            y_true = cat_targets[:, j].long().to(logits.device)

            masked_rows = (attr_mask[:, global_attr_idx] == 0)
            visible_rows = (attr_mask[:, global_attr_idx] == 1)
            use_rows = masked_rows if loss_on_masked_only else visible_rows

            if not use_rows.any():
                continue

            logits_use = logits[use_rows]
            y_true_use = y_true[use_rows]
            rec_loss = F.cross_entropy(logits_use, y_true_use, reduction="none")

            lam_list = []
            for b in range(y_true_use.size(0)):
                lam = marip_loss_fn.compute_lambda(global_attr_idx, "cat", y_true_use[b].item())
                lam_list.append(lam)

            lam_tensor = torch.tensor(lam_list, dtype=rec_loss.dtype, device=rec_loss.device)
            total_loss = total_loss + marip_loss_fn.cat_scale * (lam_tensor * rec_loss).mean()
            valid_count += 1

        for j, pred in enumerate(num_outs):
            global_attr_idx = len(textual_columns) + len(categorical_columns) + j
            y_true = num_targets[:, j].to(pred.device)
            valid_num = y_true != -1

            masked_rows = (attr_mask[:, global_attr_idx] == 0)
            visible_rows = (attr_mask[:, global_attr_idx] == 1)
            use_rows = masked_rows if loss_on_masked_only else visible_rows
            use_valid = use_rows & valid_num

            if not use_valid.any():
                continue

            pred_use = pred[use_valid]
            y_true_use = y_true[use_valid]
            rec_loss = F.mse_loss(pred_use, y_true_use, reduction="none")

            lam_list = []
            for b in range(y_true_use.size(0)):
                lam = marip_loss_fn.compute_lambda(global_attr_idx, "num", float(y_true_use[b].item()))
                lam_list.append(lam)

            lam_tensor = torch.tensor(lam_list, dtype=rec_loss.dtype, device=rec_loss.device)
            total_loss = total_loss + marip_loss_fn.num_scale * (lam_tensor * rec_loss).mean()
            valid_count += 1

        if valid_count == 0:
            return torch.tensor(0.0, device=attr_mask.device, requires_grad=True)

        return total_loss

    def categorical_class_consistency_loss_local(H, cat_targets, attr_mask=None, visible_only=True):
        total_loss = 0.0
        total_terms = 0

        for j in range(len(categorical_columns)):
            attr_id = len(textual_columns) + j
            h = H[:, attr_id, :]
            y = cat_targets[:, j]

            if attr_mask is not None and visible_only:
                valid_rows = (attr_mask[:, attr_id] == 1)
            else:
                valid_rows = torch.ones(h.size(0), dtype=torch.bool, device=h.device)

            if valid_rows.sum() < 2:
                continue

            y_valid = y[valid_rows]
            h_valid = h[valid_rows]

            unique_classes = torch.unique(y_valid)
            for cls in unique_classes:
                cls_rows = (y_valid == cls)
                if cls_rows.sum() < 2:
                    continue

                h_cls = h_valid[cls_rows]
                proto = h_cls.mean(dim=0, keepdim=True)
                total_loss = total_loss + ((h_cls - proto) ** 2).mean()
                total_terms += 1

        if total_terms == 0:
            return torch.tensor(0.0, device=H.device)

        return total_loss / total_terms

    def compute_masked_training_loss(cat_outs, num_outs, H, caty, numy, attr_mask,
                                     marip_loss_fn=None,
                                     loss_on_masked_only=False,
                                     reg_weight=1e-4,
                                     marip_blend=0.5,
                                     use_cat_class_consistency_loss=True,
                                     cat_class_consistency_weight=1e-2):
        recon_loss = 0.0
        valid_terms = 0

        cat_ptr = 0
        num_ptr = 0

        for attr_id, mod in enumerate(modalities):
            masked_rows = (attr_mask[:, attr_id] == 0)
            visible_rows = (attr_mask[:, attr_id] == 1)
            use_rows = masked_rows if loss_on_masked_only else visible_rows

            if mod == "cat":
                if use_rows.any():
                    logits = cat_outs[cat_ptr][use_rows]
                    target = caty[use_rows, cat_ptr]
                    recon_loss = recon_loss + F.cross_entropy(logits, target)
                    valid_terms += 1
                cat_ptr += 1

            elif mod == "num":
                valid_num = numy[:, num_ptr] != -1
                use_valid = use_rows & valid_num
                if use_valid.any():
                    pred = num_outs[num_ptr][use_valid]
                    target = numy[use_valid, num_ptr]
                    recon_loss = recon_loss + F.mse_loss(pred, target)
                    valid_terms += 1
                num_ptr += 1

        if valid_terms == 0:
            recon_loss = torch.tensor(0.0, device=H.device, requires_grad=True)

        marip_loss = torch.tensor(0.0, device=H.device)
        if marip_loss_fn is not None:
            marip_loss = compute_masked_marip_loss(
                cat_outs=cat_outs,
                num_outs=num_outs,
                cat_targets=caty,
                num_targets=numy,
                attr_mask=attr_mask,
                marip_loss_fn=marip_loss_fn,
                loss_on_masked_only=loss_on_masked_only
            )

        reg_loss = hidden_regularization(H, attr_mask=attr_mask)

        cat_consistency_loss = torch.tensor(0.0, device=H.device)
        if use_cat_class_consistency_loss and len(categorical_columns) > 0:
            cat_consistency_loss = categorical_class_consistency_loss_local(
                H=H,
                cat_targets=caty,
                attr_mask=attr_mask,
                visible_only=CONSISTENCY_ON_VISIBLE_ONLY
            )

        if marip_loss_fn is not None:
            total_loss = (
                (1.0 - marip_blend) * recon_loss
                + marip_blend * marip_loss
                + reg_weight * reg_loss
                + cat_class_consistency_weight * cat_consistency_loss
            )
        else:
            total_loss = recon_loss + reg_weight * reg_loss + cat_class_consistency_weight * cat_consistency_loss

        return (
            total_loss,
            recon_loss.detach(),
            marip_loss.detach(),
            reg_loss.detach(),
            cat_consistency_loss.detach()
        )

    # model
    model = FastRAGNetGeneric(
        embed_dim=384,
        d_shared=384,
        d_num=32,
        d_cat=16,
        num_numerical_attrs=len(numerical_columns),
        num_categorical_attrs=len(categorical_columns),
        num_classes_per_cat_attr=num_classes_per_cat_attr,
        memory_bank=memory_bank,
        modalities=modalities
    ).to(DEVICE)

    X_all = normalized_embeddings
    cat_targets_t = torch.tensor(ground_truth_cat, dtype=torch.long)
    num_targets_t = torch.tensor(ground_truth_num, dtype=torch.float32)

    # evaluation helpers
    def retrieve_text_from_memory(query_h, mem_emb, mem_vals):
        query_h = query_h / (query_h.norm(dim=1, keepdim=True) + 1e-8)
        mem_emb = mem_emb / (mem_emb.norm(dim=1, keepdim=True) + 1e-8)
        sim = torch.matmul(query_h, mem_emb.T)
        idx = sim.argmax(dim=1).cpu().tolist()
        return [mem_vals[i] for i in idx]

    def evaluate_standard(model, X_eval, raw_num_eval, cat_val_eval, cat_true, num_true):
        model.eval()

        cat_preds = []
        num_preds = []

        with torch.no_grad():
            for s in range(0, len(X_eval), 256):
                llm = X_eval[s:s+256].to(DEVICE)
                rawn = raw_num_eval[s:s+256].to(DEVICE)
                cati = cat_val_eval[s:s+256].to(DEVICE)

                cat_outs, num_outs, _ = model(llm, rawn, cati, edge_index, attr_mask=None)

                cur_cat = [x.argmax(dim=-1).cpu().numpy() for x in cat_outs]
                cur_num = [x.cpu().numpy() for x in num_outs]

                if len(cur_cat) > 0:
                    cat_preds.append(np.stack(cur_cat, axis=1))
                if len(cur_num) > 0:
                    num_preds.append(np.stack(cur_num, axis=1))

        cat_preds = np.concatenate(cat_preds, axis=0) if len(cat_preds) else np.zeros((len(X_eval), 0), dtype=np.int64)
        num_preds = np.concatenate(num_preds, axis=0) if len(num_preds) else np.zeros((len(X_eval), 0), dtype=np.float32)

        cat_true_np = cat_true.cpu().numpy()
        num_true_np = num_true.cpu().numpy()

        overall_cat_f1 = None
        overall_cat_acc = None
        if cat_true_np.size > 0:
            overall_cat_f1 = float(f1_score(cat_true_np.reshape(-1), cat_preds.reshape(-1), average="weighted", zero_division=0))
            overall_cat_acc = float((cat_true_np.reshape(-1) == cat_preds.reshape(-1)).mean())

        overall_num_mae = None
        overall_num_mse = None
        if num_true_np.size > 0:
            valid = num_true_np != -1
            if valid.any():
                overall_num_mae = float(np.mean(np.abs(num_true_np[valid] - num_preds[valid])))
                overall_num_mse = float(np.mean((num_true_np[valid] - num_preds[valid]) ** 2))

        cat_attr_metrics = compute_weighted_cat_f1_per_attribute(cat_true_np, cat_preds, categorical_columns)
        num_attr_metrics = compute_weighted_num_mae_per_attribute(num_true_np, num_preds, numerical_columns)

        return {
            "cat_f1_weighted_global": overall_cat_f1,
            "cat_acc_weighted_global": overall_cat_acc,
            "num_mae_weighted_global": overall_num_mae,
            "num_mse_weighted_global": overall_num_mse,
            "cat_attribute_f1": cat_attr_metrics,
            "num_attribute_mae": num_attr_metrics,
            "text_metrics": {
                "text_bleu_mean": None,
                "note": "Standard evaluation has no text generation head; BLEU is reported in masked text retrieval evaluation."
            }
        }

    def evaluate_single_masked_attribute(model, X_eval, raw_num_eval, cat_val_eval, cat_true, num_true, eval_df, attr_id, batch_size=256):
        model.eval()

        all_cat_preds = []
        all_num_preds = []
        text_preds = []

        with torch.no_grad():
            for s in range(0, len(X_eval), batch_size):
                llm = X_eval[s:s+batch_size].to(DEVICE).clone()
                rawn = raw_num_eval[s:s+batch_size].to(DEVICE).clone()
                cati = cat_val_eval[s:s+batch_size].to(DEVICE).clone()

                B = llm.size(0)
                attr_mask = torch.ones(B, len(modalities), device=DEVICE)
                attr_mask[:, attr_id] = 0.0

                llm, rawn, cati = apply_input_mask(llm, rawn, cati, attr_mask, textual_columns, categorical_columns)

                if USE_MISSING_INIT:
                    llm, rawn, cati = apply_memory_initialization(
                        llm=llm,
                        rawn=rawn,
                        cati=cati,
                        attr_mask=attr_mask,
                        memory_bank=model.memory_bank,
                        modality_types=modalities,
                        textual_columns=textual_columns,
                        categorical_columns=categorical_columns,
                        k=MISSING_INIT_TOPK,
                        r=MISSING_INIT_R,
                    )

                cat_outs, num_outs, H = model(llm, rawn, cati, edge_index, attr_mask=attr_mask)

                cur_cat = [x.argmax(dim=-1).cpu().numpy() for x in cat_outs]
                cur_num = [x.cpu().numpy() for x in num_outs]

                if len(cur_cat) > 0:
                    all_cat_preds.append(np.stack(cur_cat, axis=1))
                if len(cur_num) > 0:
                    all_num_preds.append(np.stack(cur_num, axis=1))

                if modalities[attr_id] == "txt":
                    mem_emb = model.memory_bank.get_embeddings_for_attr(attr_id)
                    mem_vals = model.memory_bank.get_values_for_attr(attr_id)
                    valid_mask = model.memory_bank.get_valid_mask_for_attr(attr_id)

                    if mem_emb is not None and mem_vals is not None and len(mem_vals) > 0:
                        mem_emb = mem_emb[valid_mask]
                        mem_vals = [v for j, v in enumerate(mem_vals) if valid_mask[j].item()]
                        if len(mem_vals) > 0:
                            preds = retrieve_text_from_memory(H[:, attr_id, :], mem_emb.to(DEVICE), mem_vals)
                        else:
                            preds = [""] * B
                    else:
                        preds = [""] * B

                    text_preds.extend(preds)

        cat_preds = np.concatenate(all_cat_preds, axis=0) if len(all_cat_preds) else None
        num_preds = np.concatenate(all_num_preds, axis=0) if len(all_num_preds) else None

        if modalities[attr_id] == "cat":
            local_cat = attr_id - len(textual_columns)
            y_true = cat_true.cpu().numpy()[:, local_cat]
            y_pred = cat_preds[:, local_cat]

            evaluated_count = int(len(y_true))
            f1_val = float(f1_score(y_true, y_pred, average="weighted", zero_division=0))

            return {
                "attr_id": int(attr_id),
                "attr_name": categorical_columns[local_cat],
                "attr_type": "cat",
                "weighted_f1": f1_val,
                "accuracy": float((y_true == y_pred).mean()),
                "evaluated_masked_count": evaluated_count
            }

        elif modalities[attr_id] == "num":
            local_num = attr_id - len(textual_columns) - len(categorical_columns)
            y_true = num_true.cpu().numpy()[:, local_num]
            y_pred = num_preds[:, local_num]

            valid = y_true != -1
            valid_count = int(valid.sum())

            mae = np.mean(np.abs(y_true[valid] - y_pred[valid])) if valid.any() else None
            mse = np.mean((y_true[valid] - y_pred[valid]) ** 2) if valid.any() else None

            return {
                "attr_id": int(attr_id),
                "attr_name": numerical_columns[local_num],
                "attr_type": "num",
                "weighted_mae": float(mae) if mae is not None else None,
                "weighted_mse": float(mse) if mse is not None else None,
                "evaluated_masked_count": valid_count
            }

        else:
            local_txt = attr_id
            refs = eval_df[textual_columns[local_txt]].fillna("").astype(str).tolist()
            bleu = compute_text_bleu(refs, text_preds)
            return {
                "attr_id": int(attr_id),
                "attr_name": textual_columns[local_txt],
                "attr_type": "txt",
                "text_bleu_mean": float(bleu),
                "evaluated_masked_count": int(len(refs))
            }

    def evaluate_full(model, eval_idx):
        X_eval = X_all[eval_idx]
        raw_num_eval = numerical_values_tensor[eval_idx]
        cat_val_eval = categorical_indices_tensor[eval_idx]
        cat_true = cat_targets_t[eval_idx]
        num_true = num_targets_t[eval_idx]
        eval_df = df.iloc[eval_idx].reset_index(drop=True)

        standard_results = evaluate_standard(
            model=model,
            X_eval=X_eval,
            raw_num_eval=raw_num_eval,
            cat_val_eval=cat_val_eval,
            cat_true=cat_true,
            num_true=num_true
        )

        masked_results = []
        for attr_id in range(len(modalities)):
            res = evaluate_single_masked_attribute(
                model=model,
                X_eval=X_eval,
                raw_num_eval=raw_num_eval,
                cat_val_eval=cat_val_eval,
                cat_true=cat_true,
                num_true=num_true,
                eval_df=eval_df,
                attr_id=attr_id,
                batch_size=256
            )
            masked_results.append(res)

        cat_items = [
            x for x in masked_results
            if x["attr_type"] == "cat"
            and x.get("weighted_f1") is not None
            and x.get("evaluated_masked_count") is not None
        ]
        num_items = [
            x for x in masked_results
            if x["attr_type"] == "num"
            and x.get("weighted_mae") is not None
            and x.get("evaluated_masked_count") is not None
        ]
        txt_items = [
            x for x in masked_results
            if x["attr_type"] == "txt"
            and x.get("text_bleu_mean") is not None
            and x.get("evaluated_masked_count") is not None
        ]

        masked_weighted_attribute_f1, total_cat_weight = weighted_average_by_count(
            cat_items, "weighted_f1", "evaluated_masked_count"
        )
        masked_weighted_attribute_mae, total_num_weight = weighted_average_by_count(
            num_items, "weighted_mae", "evaluated_masked_count"
        )
        masked_weighted_attribute_mse, _ = weighted_average_by_count(
            num_items, "weighted_mse", "evaluated_masked_count"
        )
        masked_text_bleu_mean, total_txt_weight = weighted_average_by_count(
            txt_items, "text_bleu_mean", "evaluated_masked_count"
        )

        out = dict(standard_results)
        out["masked_attribute_results"] = masked_results
        out["masked_summary"] = {
            "masked_weighted_attribute_f1": masked_weighted_attribute_f1,
            "masked_weighted_attribute_f1_total_weight": total_cat_weight,
            "masked_weighted_attribute_mae": masked_weighted_attribute_mae,
            "masked_weighted_attribute_mae_total_weight": total_num_weight,
            "masked_weighted_attribute_mse": masked_weighted_attribute_mse,
            "masked_text_bleu_mean": masked_text_bleu_mean,
            "masked_text_bleu_total_weight": total_txt_weight
        }
        out["eval_loss"] = None
        return out

    # training
    def train_ragnet(model, train_idx, test_idx, epochs=3, lr=1e-4, eval_every=1):
        optimizer = torch.optim.Adam(model.parameters(), lr=lr)
        history = []
        best_score = -1e18
        best_path = os.path.join(model_dir, f"{dataset_name}_best.pt")

        for epoch in range(epochs):
            model.train()
            total_train_loss = 0.0
            total_recon_loss = 0.0
            total_marip_loss = 0.0
            total_reg_loss = 0.0
            total_cat_consistency_loss = 0.0
            valid_train = 0

            shuffled = np.random.permutation(train_idx)

            for s in tqdm(range(0, len(shuffled), TRAIN_BATCH_SIZE), desc=f"{dataset_name} Epoch {epoch+1}/{epochs}", leave=False):
                batch_ids = shuffled[s:s+TRAIN_BATCH_SIZE]

                llm = X_all[batch_ids].to(DEVICE).clone()
                rawn = numerical_values_tensor[batch_ids].to(DEVICE).clone()
                cati = categorical_indices_tensor[batch_ids].to(DEVICE).clone()
                caty = cat_targets_t[batch_ids].to(DEVICE)
                numy = num_targets_t[batch_ids].to(DEVICE)

                if USE_RANDOM_MASK_TRAINING:
                    attr_mask = build_training_attr_mask(
                        batch_size=llm.size(0),
                        num_attrs=len(modalities),
                        device=DEVICE,
                        mask_one_attr_per_sample=MASK_ONE_ATTR_PER_SAMPLE,
                        mask_prob=MASK_PROB
                    )

                    llm, rawn, cati = apply_input_mask(llm, rawn, cati, attr_mask, textual_columns, categorical_columns)

                    if USE_MISSING_INIT:
                        llm, rawn, cati = apply_memory_initialization(
                            llm=llm,
                            rawn=rawn,
                            cati=cati,
                            attr_mask=attr_mask,
                            memory_bank=memory_bank,
                            modality_types=modalities,
                            textual_columns=textual_columns,
                            categorical_columns=categorical_columns,
                            k=MISSING_INIT_TOPK,
                            r=MISSING_INIT_R,
                        )
                else:
                    attr_mask = torch.ones(llm.size(0), len(modalities), device=DEVICE)

                cat_outs, num_outs, H = model(llm, rawn, cati, edge_index, attr_mask=attr_mask)

                loss, recon_loss, marip_loss, reg_loss, cat_consistency_loss = compute_masked_training_loss(
                    cat_outs=cat_outs,
                    num_outs=num_outs,
                    H=H,
                    caty=caty,
                    numy=numy,
                    attr_mask=attr_mask,
                    marip_loss_fn=marip_loss_fn if USE_MARIP else None,
                    loss_on_masked_only=LOSS_ON_MASKED_ONLY,
                    reg_weight=REG_WEIGHT,
                    marip_blend=MARIP_BLEND,
                    use_cat_class_consistency_loss=USE_CAT_CLASS_CONSISTENCY_LOSS,
                    cat_class_consistency_weight=CAT_CLASS_CONSISTENCY_WEIGHT
                )

                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

                total_train_loss += float(loss.item())
                total_recon_loss += float(recon_loss.item())
                total_marip_loss += float(marip_loss.item())
                total_reg_loss += float(reg_loss.item())
                total_cat_consistency_loss += float(cat_consistency_loss.item())
                valid_train += 1

            avg_train_loss = total_train_loss / max(valid_train, 1)
            avg_recon_loss = total_recon_loss / max(valid_train, 1)
            avg_marip_loss = total_marip_loss / max(valid_train, 1)
            avg_reg_loss = total_reg_loss / max(valid_train, 1)
            avg_cat_consistency_loss = total_cat_consistency_loss / max(valid_train, 1)

            if (epoch + 1) % eval_every == 0:
                eval_results = evaluate_full(model, test_idx)
            else:
                eval_results = {"eval_loss": None, "masked_summary": {}}

            row = {
                "epoch": epoch + 1,
                "train_loss": avg_train_loss,
                "train_recon_loss": avg_recon_loss,
                "train_marip_loss": avg_marip_loss,
                "train_reg_loss": avg_reg_loss,
                "train_cat_consistency_loss": avg_cat_consistency_loss,
                "cat_f1_weighted_global": eval_results.get("cat_f1_weighted_global"),
                "num_mae_weighted_global": eval_results.get("num_mae_weighted_global"),
                "masked_weighted_attribute_f1": eval_results.get("masked_summary", {}).get("masked_weighted_attribute_f1"),
                "masked_weighted_attribute_mae": eval_results.get("masked_summary", {}).get("masked_weighted_attribute_mae"),
                "masked_text_bleu_mean": eval_results.get("masked_summary", {}).get("masked_text_bleu_mean"),
            }
            history.append(row)

            print(f"\n[{dataset_name} Epoch {epoch+1}/{epochs}]")
            print(f"Train Loss                     : {avg_train_loss:.4f}")
            print(f"Train Recon Loss               : {avg_recon_loss:.4f}")
            print(f"Train MARIP Loss               : {avg_marip_loss:.4f}")
            print(f"Train Reg Loss                 : {avg_reg_loss:.6f}")
            print(f"Train Cat Consistency Loss     : {avg_cat_consistency_loss:.6f}")
            print(f"Cat F1 Weighted Global         : {eval_results.get('cat_f1_weighted_global', 0.0) or 0.0:.4f}")
            print(f"Num MAE Weighted Global        : {eval_results.get('num_mae_weighted_global', 0.0) or 0.0:.4f}")
            print(f"Masked Weighted Attribute F1   : {eval_results.get('masked_summary', {}).get('masked_weighted_attribute_f1', 0.0) or 0.0:.4f}")
            print(f"Masked Weighted Attribute MAE  : {eval_results.get('masked_summary', {}).get('masked_weighted_attribute_mae', 0.0) or 0.0:.4f}")
            print(f"Masked Text BLEU Mean          : {eval_results.get('masked_summary', {}).get('masked_text_bleu_mean', 0.0) or 0.0:.4f}")

            cur_score = 0.0
            if eval_results.get("masked_summary", {}).get("masked_weighted_attribute_f1") is not None:
                cur_score += float(eval_results["masked_summary"]["masked_weighted_attribute_f1"])
            if eval_results.get("masked_summary", {}).get("masked_text_bleu_mean") is not None:
                cur_score += float(eval_results["masked_summary"]["masked_text_bleu_mean"])
            if eval_results.get("masked_summary", {}).get("masked_weighted_attribute_mae") is not None:
                cur_score -= float(eval_results["masked_summary"]["masked_weighted_attribute_mae"])

            if cur_score > best_score:
                best_score = cur_score
                torch.save({"model_state_dict": model.state_dict(), "history": history}, best_path)

        return history

    history = train_ragnet(model, train_idx, test_idx, epochs=epochs, lr=lr, eval_every=1)
    final_results = evaluate_full(model, test_idx)

    print("\n===== FINAL TEST RESULTS =====")
    for k, v in final_results.items():
        if isinstance(v, dict):
            print(f"{k}: {v}")
        elif isinstance(v, list):
            print(f"{k}: {v[:3]} ... total={len(v)}")
        else:
            print(f"{k}: {v:.4f}" if isinstance(v, (int, float, np.floating)) and v is not None else f"{k}: {v}")

    # save outputs
    np.save(os.path.join(emb_dir, "textual_embeddings.npy"), textual_embeddings)
    np.save(os.path.join(emb_dir, "categorical_embeddings.npy"), categorical_embeddings)
    np.save(os.path.join(emb_dir, "numerical_embeddings.npy"), numerical_embeddings)
    np.save(os.path.join(emb_dir, "categorical_indices.npy"), categorical_indices)
    np.save(os.path.join(emb_dir, "numerical_values.npy"), numerical_values)

    np.save(os.path.join(out_dir, "all_embeddings.npy"), all_embeddings_np)
    np.save(os.path.join(out_dir, "normalized_embeddings.npy"), normalized_embeddings.detach().cpu().numpy())
    np.save(os.path.join(out_dir, "ground_truth_cat.npy"), ground_truth_cat)
    np.save(os.path.join(out_dir, "ground_truth_num.npy"), ground_truth_num)
    np.save(os.path.join(out_dir, "target_labels.npy"), target_labels)

    torch.save(edge_index.detach().cpu(), os.path.join(out_dir, "edge_index.pt"))
    torch.save({"model_state_dict": model.state_dict()}, os.path.join(model_dir, f"{dataset_name}_last.pt"))

    with open(os.path.join(out_dir, "history.json"), "w") as f:
        json.dump(history, f, indent=2)

    with open(os.path.join(out_dir, "final_results.json"), "w") as f:
        json.dump(final_results, f, indent=2, default=float)

    metadata = {
        "dataset_name": dataset_name,
        "textual_columns": textual_columns,
        "categorical_columns": categorical_columns,
        "numerical_columns": numerical_columns,
        "target_column": target_column,
        "modalities": modalities,
        "all_attribute_names": all_attribute_names,
        "num_classes_per_cat_attr": num_classes_per_cat_attr,
        "target_map": target_map,
        "train_subset": len(train_idx),
        "test_subset": len(test_idx),
        "epochs": epochs,
        "memory_size": memory_bank.size(),
        "memory_row_ids": memory_bank.get_row_ids(),
        "use_missing_init": USE_MISSING_INIT,
        "missing_init_topk": MISSING_INIT_TOPK,
        "missing_init_r": MISSING_INIT_R
    }
    with open(os.path.join(out_dir, "metadata.json"), "w") as f:
        json.dump(metadata, f, indent=2)

    print("\nSaved files to:", save_dir)
    print("Done.")
    return {
        "df": df,
        "history": history,
        "final_results": final_results,
        "save_dir": save_dir
    }

DEVICE: cuda
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [ ]:
# ============================================================
# DATASET 1: AUSTRALIAN CREDIT APPROVAL (UCI ID 143)
# ============================================================
australian = fetch_ucirepo(id=143)

X = australian.data.features.copy()
y = australian.data.targets.copy()

df = pd.concat([X, y], axis=1)
df = canonicalize_columns(df)

# target column from UCI
target_column = "a15"

# explicit schema from UCI
categorical_columns = ["a1", "a4", "a5", "a6", "a8", "a9", "a11", "a12"]
numerical_columns = ["a2", "a3", "a7", "a10", "a13", "a14"]

# robust cleaning
for c in categorical_columns + [target_column]:
    df[c] = df[c].astype(str).replace({"?": "Missing"}).fillna("Missing")

for c in numerical_columns:
    df[c] = parse_numeric_series(df[c])

# synthetic text
summary_cols = categorical_columns[:4] + numerical_columns[:2]
df = build_summary_text(df, summary_cols, out_col="summary_text", prefix="Australian credit application")

result_australian = run_ragnet_dataset(
    df=df,
    dataset_name="australian_credit",
    save_dir="/content/drive/MyDrive/ragnet_uci/australian_credit",
    textual_columns=["summary_text"],
    categorical_columns=categorical_columns,
    numerical_columns=numerical_columns,
    target_column=target_column,
    epochs=EPOCHS,
    lr=LR
)

  0%|          | 0/6 [00:00<?, ?it/s]/tmp/ipykernel_3132/2381620237.py:350: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=(DEVICE.type == "cuda")):
  0%|          | 0/1 [00:00<?, ?it/s]/tmp/ipykernel_3132/2381620237.py:350: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=(DEVICE.type == "cuda")):
  0%|          | 0/1 [00:00<?, ?it/s]/tmp/ipykernel_3132/2381620237.py:350: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=(DEVICE.type == "cuda")):
  0%|          | 0/1 [00:00<?, ?it/s]/tmp/ipykernel_3132/2381620237.py:350: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with to


[australian_credit Epoch 1/100]
Train Loss                     : 13.8780
Train Recon Loss               : 14.1807
Train MARIP Loss               : 11.1523
Train Reg Loss                 : 0.052423
Train Cat Consistency Loss     : 0.007014
Cat F1 Weighted Global         : 0.8634
Num MAE Weighted Global        : 0.6094
Masked Weighted Attribute F1   : 0.4838
Masked Weighted Attribute MAE  : 0.6607
Masked Text BLEU Mean          : 0.6267



[australian_credit Epoch 2/100]
Train Loss                     : 11.9328
Train Recon Loss               : 12.2009
Train MARIP Loss               : 9.5195
Train Reg Loss                 : 0.056124
Train Cat Consistency Loss     : 0.007751
Cat F1 Weighted Global         : 0.9019
Num MAE Weighted Global        : 0.5406
Masked Weighted Attribute F1   : 0.4933
Masked Weighted Attribute MAE  : 0.6533
Masked Text BLEU Mean          : 0.6259



[australian_credit Epoch 3/100]
Train Loss                     : 9.9496
Train Recon Loss               : 10.1775
Train MARIP Loss               : 7.8976
Train Reg Loss                 : 0.061260
Train Cat Consistency Loss     : 0.008565
Cat F1 Weighted Global         : 0.9492
Num MAE Weighted Global        : 0.4809
Masked Weighted Attribute F1   : 0.4898
Masked Weighted Attribute MAE  : 0.6469
Masked Text BLEU Mean          : 0.6275



[australian_credit Epoch 4/100]
Train Loss                     : 8.6489
Train Recon Loss               : 8.8473
Train MARIP Loss               : 6.8616
Train Reg Loss                 : 0.070666
Train Cat Consistency Loss     : 0.010692
Cat F1 Weighted Global         : 0.9608
Num MAE Weighted Global        : 0.4356
Masked Weighted Attribute F1   : 0.4999
Masked Weighted Attribute MAE  : 0.6410
Masked Text BLEU Mean          : 0.6310


...


[australian_credit Epoch 96/100]
Train Loss                     : 0.0464
Train Recon Loss               : 0.0447
Train MARIP Loss               : 0.0264
Train Reg Loss                 : 1.386511
Train Cat Consistency Loss     : 0.339902
Cat F1 Weighted Global         : 1.0000
Num MAE Weighted Global        : 0.1603
Masked Weighted Attribute F1   : 0.5474
Masked Weighted Attribute MAE  : 0.6281
Masked Text BLEU Mean          : 0.6439



[australian_credit Epoch 97/100]
Train Loss                     : 0.0460
Train Recon Loss               : 0.0444
Train MARIP Loss               : 0.0269
Train Reg Loss                 : 1.352828
Train Cat Consistency Loss     : 0.324775
Cat F1 Weighted Global         : 1.0000
Num MAE Weighted Global        : 0.1613
Masked Weighted Attribute F1   : 0.5467
Masked Weighted Attribute MAE  : 0.6284
Masked Text BLEU Mean          : 0.6439



[australian_credit Epoch 98/100]
Train Loss                     : 0.0332
Train Recon Loss               : 0.0305
Train MARIP Loss               : 0.0210
Train Reg Loss                 : 1.347184
Train Cat Consistency Loss     : 0.352807
Cat F1 Weighted Global         : 1.0000
Num MAE Weighted Global        : 0.1611
Masked Weighted Attribute F1   : 0.5491
Masked Weighted Attribute MAE  : 0.6285
Masked Text BLEU Mean          : 0.6439



[australian_credit Epoch 99/100]
Train Loss                     : 0.0325
Train Recon Loss               : 0.0300
Train MARIP Loss               : 0.0207
Train Reg Loss                 : 1.330268
Train Cat Consistency Loss     : 0.332239
Cat F1 Weighted Global         : 1.0000
Num MAE Weighted Global        : 0.1599
Masked Weighted Attribute F1   : 0.5491
Masked Weighted Attribute MAE  : 0.6285
Masked Text BLEU Mean          : 0.6456



[australian_credit Epoch 100/100]
Train Loss                     : 0.0433
Train Recon Loss               : 0.0414
Train MARIP Loss               : 0.0255
Train Reg Loss                 : 1.396905
Train Cat Consistency Loss     : 0.340661
Cat F1 Weighted Global         : 1.0000
Num MAE Weighted Global        : 0.1587
Masked Weighted Attribute F1   : 0.5471
Masked Weighted Attribute MAE  : 0.6284
Masked Text BLEU Mean          : 0.6473

===== FINAL TEST RESULTS =====
cat_f1_weighted_global: 1.0000
cat_acc_weighted_global: 1.0000
num_mae_weighted_global: 0.1587
num_mse_weighted_global: 0.4849
cat_attribute_f1: {'a1': 1.0, 'a4': 1.0, 'a5': 1.0, 'a6': 1.0, 'a8': 1.0, 'a9': 1.0, 'a11': 1.0, 'a12': 1.0, 'weighted_attribute_f1': 1.0}
num_attribute_mae: {'a2': 0.16023299098014832, 'a3': 0.18158559501171112, 'a7': 0.11850450932979584, 'a10': 0.06696850061416626, 'a13': 0.14187367260456085, 'a14': 0.28285908699035645, 'weighted_attribute_mae': 0.15867072343826294}
text_metrics: {'text_bleu_mean'